# KSA NFI GeoAI Comprehensive Expanded Pipeline V6

Version 6 keeps every Part A section from V5 and rebuilds Parts B and C in the same reporting style (banners, KPI cards, styled tables, findings panels, figure cards). It also fixes several method and robustness problems, listed in the change log below.

| Part | Sections | Scope | Primary outputs |
|:--|:--|:--|:--|
| **A** (preserved) | 0 to 19 | Asir GIS universe, anti-clip lattice, Gate 1 and 2, cleaning, EDA, cartography, GEDI, GeoAI protocol | `outputs/` |
| **B** (expansion) | E0 to E13 | National LULC Validation, calculation-document tree and shrub biomass, missingness, ESDA, height model | `outputs/expansion/` |
| **C** (national) | N0 to N5 | Entire Kingdom tabular universe (all regions in the SU master) | `outputs_national_v5/` |

**Run order.** (1) bootstrap cell, (2) presentation kit cell, (3) Part A top to bottom, (4) Part B, (5) Part C. Parts B and C reload their own inputs from `config.yaml`, so after a kernel restart you only need steps 1, 2 and the part you want.

**National user guide (recommended for country-wide runs):** `NFI_National_GeoPipeline_Guide_V5.ipynb`

**Shareable national HTML products** (after Part C or the guide notebook):
- Report: `outputs_national_v5/documents/NFI_national_geopipeline_report.html`
- Atlas: `outputs_national_v5/maps/html_atlas/index.html` (fixed class colours, Voronoi or points, admin filters)

Primary foundation path:
`F:\1.KSA\29.NFI\KSA_NFI_GeoAI_Data_Preparation_Cursor\nfi_geoai_data_preparation\NFI_GeoAI_Data_Preparation_V4_Refined.ipynb`

HTML complement:
`F:\1.KSA\1.Forest\10.NFI\10.LULC_Validation\LULC_VALIDATION_REPORT.html`


## What changed in V6

| Area | Change | Why it matters |
|:--|:--|:--|
| Name clashes | Parts B and C use `nat_trees`, `nat_su`, `nat_plot`, `nat_shrubs`, `SEED`, `NAT_CFG`, `NAT_SUMMARY`. V5 reused `trees`, `su`, `RS` and `CFG`. | In V5 the random seed `RS` was overwritten by the run summary in Part C, and Part A's `trees` was overwritten in Part B. Re-running any cell afterwards could fail or change results. |
| Tree SU status (E6) | Units are classed as *no tree records*, *records but no usable DBH and height*, or *usable*. V5 counted only measured trees. | A unit whose trees all lack a height is a measurement gap, not an empty stand. |
| Row ledgers (E6, E8) | Every removed record is counted by reason. DBH and height are coerced to numbers and non-numeric entries are counted. | Silent drops hide how much of the national total is a lower bound. |
| Duplicate plots (E7) | Duplicate SU and plot keys are counted before the first row is kept. | V5 dropped them without a trace. |
| Shrub columns and unit (E8) | The detected columns, any competing height columns and the unit rule are shown. | A wrong height column or unit changes biomass by orders of magnitude without an error. |
| Moran's I (E10) | Computed on one point per SU, with k = 4, 8 and 12. V5 used plot points that repeated each SU value. | Repeated values inflate spatial autocorrelation. |
| Height model (E11) | Adds a mean baseline, a linear model and a log-log allometry beside the random forest, scored with random and SU-blocked folds. Folds are capped at the number of SUs. | A forest is only worth its complexity if it beats simple curves under honest validation. |
| Correlation labels (E9) | The stray "VIF" comment is removed. A finding states that derived metrics are correlated by construction. | The old code computed an OLS R² and called it VIF. |
| Maps (E12) | Latitude-corrected aspect, two discrete classes for validation, high values drawn last. | Equal degrees distort shapes at these latitudes, and sequential colour on a binary variable misleads. |
| Presentation (B and C) | Every section has a banner, an *About the code below* table, tables, figure cards and findings, in the neutral Part A style. The Part C green HTML blocks are gone. | One reporting style from start to finish, with colour kept for charts and maps. |
| Progress stepper | The kit stepper now shows the sections of the current part only (A, B or C). | Part B and C ids such as E3 and N2 are recognised. |
| Browser opening (N4) | Off by default (`OPEN_IN_BROWSER`). | Opening tabs on every run is surprising on shared machines. |
| GEDI paths (section 15) | The folder is set once as `GEDI_DIR`. Use the `NFI_GEDI_DIR` environment variable, or `data/gedi` inside the package. | V5 hard-coded `H:/` in four places. |


## Colab / portable bootstrap

Run this cell first on Google Colab. It mounts Drive if present, otherwise clones the public GitHub package, then sets the working directory so Part A (V4) and Part B (expansion) resolve `config.yaml` and `data/`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, warnings
warnings.filterwarnings('ignore')

IN_COLAB = False
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped/failed:', e)
    IN_COLAB = True
except Exception:
    pass

REPO_URL = 'https://github.com/gabrielsanya-ops/NFI_Comprehensive_Expanded_V5.git'
CLONE_DIR = Path('/content/NFI_Comprehensive_Expanded_V5')
ALIASES = [
    'NFI_Comprehensive_Expanded_V5',
    '12.NFI_Comprehensive_Expanded_V5',
]

def looks_like_root(p: Path) -> bool:
    return (p / 'config.yaml').exists() and (p / 'src').exists() and (p / 'data' / 'cleansed').exists()

CANDIDATES = []
for name in ALIASES:
    CANDIDATES += [
        Path('/content/drive/MyDrive') / name,
        Path('/content/drive/MyDrive/33.Projects') / name,
        Path('/content') / name,
        Path(r'F:/1.KSA/29.NFI') / name,
        Path(r'F:/1.KSA/29.NFI/12.NFI_Comprehensive_Expanded_V5'),
    ]
CANDIDATES += [Path.cwd(), CLONE_DIR]

ROOT = next((c.resolve() for c in CANDIDATES if c.exists() and looks_like_root(c)), None)
if ROOT is None and IN_COLAB:
    print('Package not on Drive: cloning from GitHub...')
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)])
    ROOT = CLONE_DIR.resolve()

if ROOT is None or not looks_like_root(ROOT):
    raise FileNotFoundError(
        'Could not locate NFI_Comprehensive_Expanded_V5. '
        'Sync the folder to Drive MyDrive or ensure the GitHub repo is public, then re-run.'
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('IN_COLAB', IN_COLAB)
print('ROOT', ROOT)
print('notebook', (ROOT / 'NFI_GeoAI_Comprehensive_Expanded_V5.ipynb').exists())
print('cleansed', sorted(p.name for p in (ROOT / 'data' / 'cleansed').glob('*.xlsx'))[:8], '...')


<div style="background:#0a0a0a;color:#f5f5f5;padding:30px 34px;border-radius:14px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
<div style="font-size:12px;letter-spacing:.16em;text-transform:uppercase;color:#a3a3a3">Kingdom of Saudi Arabia &nbsp;·&nbsp; National Forest Inventory (NFI)</div>
<div style="font-size:31px;font-weight:800;line-height:1.2;margin:8px 0 10px">Asir NFI Data Preparation, Validation &amp; Map Making</div>
<div style="font-size:14.5px;color:#d4d4d4;max-width:800px;line-height:1.55">An audit-first pipeline that turns cleansed NFI tables and the Asir GIS grid into reviewed, reproducible products: <b>Gate 1 / Gate 2</b> evidence, area appropriation, exploratory analysis, GEDI LiDAR cross-checks and interactive map products. Version 4: refined layout, written findings after every analysis, and a code guide before every code cell.</div>
</div>

&nbsp;

## At a glance

<a id="targets"></a>

**Embedded Gate targets** (from the Validation Point 1 and 2 packages). Every run is reconciled against these numbers, and the notebook stops if they do not match.

| Gate | Quantity | Target |
|:-----|:---------|-------:|
| Universe | Sampling units (SU) | **430** |
| Gate 1 | SUs with tree records / without | **240 / 190** |
| Design | Expected plots (430 × 5) | **2,150** |
| Gate 2 | Plots lacking trees inside tree-SUs / SUs affected | **544 / 204** |
| Observed | Plots with at least one tree record | **656** |

## How this notebook communicates

Every section follows the same pattern, so you always know where you are and what to conclude.

| You see | What it is for |
|:--------|:---------------|
| **Guide block** (markdown above each code cell) | Says what the code reads, what it does, what it writes, and what to check before moving on. |
| **Header comment** (first lines of each code cell) | The same guide in short form, next to the code that it describes. |
| **Numbered comments** (`# ---- 1. ...`) | Mark each step inside the cell, so a reader can follow the logic without running it. |
| **Section banner and completion bar** | Banner shows the position in the run; the bar shows the time taken. |
| **Key findings panel** | Plain-language sentences written from the data of this run (never typed in by hand), placed right after the table or chart they explain. |
| **Figure card** | Chart, then a *Finding* (what this run shows) and *How to read* (how to read this kind of chart). |
| **Executive summary** (section 19) | Collects every finding, the open items and the next steps, and writes a stand-alone HTML report. |

**Colour policy.** Banners, tables, cards and callouts are neutral grey on purpose. Colour is kept for charts and maps, where it carries data.

| Symbol | Meaning |
|:------:|:--------|
| ✔ | Check passed or step complete |
| ⚠ | Needs attention, but the run continues |
| ✖ | Check failed (the run stops where a hard `assert` guards the result) |
| 💾 | File or StageStore product written |
| ⏱ | Section run time |

## Roadmap

Run the notebook **top to bottom**. Section numbers below match the banner shown at the top of each section's output.

| # | Section | What happens | You get |
|--:|:--------|:-------------|:--------|
| 0 | [Environment](#s0) | Imports, `config.yaml`, output root, input-file check | Verified setup and Gate targets |
| 1 | [Ingest](#s1) | Tabular keys and the Asir GIS master | 430-SU universe |
| 2 | [Area](#s2) | `Area_H` by SU; policy checks | Area audit and a check of how `Area_H` was computed |
| 3 | [Lattice](#s3) | Anti-clip long-form SU × plot presence | 2,150-row lattice |
| 4 | [Gates](#s4) | Gate 1 and Gate 2 versus baked targets | Annexes, sign-off templates, composition chart |
| 5 | [Regions](#s5) | Region roll-ups and SU graphics | Tables and non-map charts |
| 6 | [Cleaning](#s6) | Spelling, characters, field types | **Corrections report** |
| 7 | [Ready tables](#s7) | Joined, cleaned deliverable | Error-free tables for maps and graphs |
| 7b | [Coordinate QA](#s7b) | Field vs office GPS cross-check | Statistics and displacement chart |
| 8 | [Population QA](#s8) | Missing vs zero vs measured; duplicate flag | Eligibility flags |
| 9 | [Statistical QA](#s9) | Robust statistics and outlier flags | QA profile |
| 10 | [EDA](#s10) | Figure pack on cleaned frames | Descriptive and statistical graphics |
| 11 | [Scorecard](#s11) | Clip/pivot method versus anti-clip | Integrity scorecard |
| 12 | [Cartography](#s12) | Interactive Leaflet grid-review maps | Maps in the HTML atlas |
| 13 | [HTML atlas](#s13) | NFI poster-style interactive maps | Leaflet HTML maps and portal index |
| 14 | [Spatial QA](#s14) | Plot coordinates joined to the grid | Join-status audit |
| 15 | [LiDAR validation](#s15) | GEDI L2A versus field height and elevation | Agreement metrics, graphs, 3D scene |
| 16 | [GeoAI protocol](#s16) | Frozen validation design | Protocol table |
| 17 | [StageStore](#s17) | Inventory of intermediates | Stage manifest |
| 18 | [Ledger and exports](#s18) | Row conservation, readiness, exports | Validated files |
| 19 | [Executive summary](#s19) | All findings, open items, next steps | HTML report and findings CSV |

## Analytical design principles

1. **Long-form lattice.** `SU × plot`, left-joined to tree counts, so zero cells stay visible.
2. **Anti-clip joins.** `Asir_SU_Final` is always the LEFT master.
3. **Area appropriation.** Attach `Area_H` by `SU_UID`; block stems/ha until `plot_area_ha` exists.
4. **Lexical QA, then derived clean.** Every correction is reported; source files are never overwritten.
5. **Ready clean tables.** One joined deliverable feeds maps and presentation graphs.
6. **StageStore.** Reviewable intermediates are saved under the output root.
7. **Map-ready deliverables.** Static PNGs and the interactive HTML atlas consume ready-clean tables only.
8. **Evidence, not imputation.** Outliers are flagged, never deleted; LiDAR is evidence, never a silent substitute for field values.

## Glossary

| Term | Meaning |
|:-----|:--------|
| **SU** | Sampling unit: one cell of the Asir grid. Each SU has 5 design plots. |
| **Plot** | A field plot inside an SU (`PLOT_UID` = `SU_UID` + plot number). |
| **Gate 1** | SUs with **no** tree records at all (190 expected). They need a field classification. |
| **Gate 2** | Plots with no trees **inside** SUs that do have trees (544 plots in 204 SUs expected). |
| **Lattice** | The full SU × plot table, including empty cells. |
| **Anti-clip** | A join that keeps every SU, so empty units are not silently dropped. |
| **StageStore** | The folder where each intermediate table is saved for review. |
| **DBH** | Diameter at breast height (cm). |
| **FV metrics** | Volume, biomass and carbon computed with the NFI-FV formulas (provisional here). |
| **IQR outlier** | A value outside Q1 - 1.5 × IQR or Q3 + 1.5 × IQR. A review flag, not an error. |
| **GEDI L2A** | NASA spaceborne LiDAR footprints (about 25 m wide) with relative-height metrics such as RH98. |
| **RH98** | Height below which 98% of the returned waveform energy lies: an upper-canopy proxy. |
| **Bland-Altman** | A plot of the difference between two methods against their mean, used to see bias and spread. |

> **Prerequisites:** a `config.yaml` beside this notebook, the project's `src/` modules (`nfi_audit`, `nfi_area_maps`, `nfi_datascience`, `nfi_beautiful_maps`, `nfi_gedi_l2a`, `nfi_lidar_global`, `nfi_imputation_candidates`), and the source Excel and shapefile inputs that `config.yaml` points to.

## Setup · presentation kit

Run this cell first. It defines the display helpers used by every section (banners, KPI cards, styled tables, findings panels, figure cards) and the chart theme. It contains **no analysis logic**.

| Helper | Use |
|:--|:--|
| `banner()` / `done()` | Open and close a section, with run time. |
| `sub()` | Lettered sub-step heading. |
| `callout()` / `say()` | Message box and one-line status. |
| `kpis()` / `kpi()` | Headline number cards. |
| `show()` | Styled table (data bars, grey heat-maps). |
| `report()` / `findings()` | Numbered plain-language findings written from the run's data. |
| `show_fig()` | Figure card with a *Finding* and a *How to read* note. |

Colour policy: the page chrome is grey; `CHART_COLORS` (colour-blind safe) is used inside charts and maps only.

In [ ]:
# =============================================================================
#  PRESENTATION KIT  ·  display helpers only; no analysis logic lives here.
#
#  Design rule  : neutral grey-scale chrome (banners, tables, cards, callouts).
#                 Colour is reserved for charts and maps, where it carries data.
#  Run order    : run this cell first. Every later cell calls these helpers.
#  Main helpers : banner() / done()        section header and completion bar
#                 sub()                    lettered sub-step heading
#                 callout() / say()        message boxes and one-line status
#                 kpis() / kpi()           headline number cards
#                 show()                   styled table (data bars, grey heat-maps)
#                 findings() / report()    numbered plain-language findings
#                 show_fig() / show_figs() figure cards with a finding + reading guide
# =============================================================================
import time, base64, html as _html, uuid, io
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
from cycler import cycler
from IPython.display import HTML, display, Image

# ----------------------------------------------------------------------------- palette
INK, MUTED, LINE, PAPER = "#171717", "#525252", "#d4d4d4", "#f5f5f5"
BRAND, BRAND_DK, ACCENT = "#262626", "#0a0a0a", "#404040"
FONT = "-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif"

# Colour is allowed ONLY inside charts. Colour-blind-safe (Okabe-Ito based) cycle.
CHART_COLORS = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#7A5195", "#6B7280"]

_KIND = {   # kind: (accent, tint, icon).  Status is carried by icon and weight, not by hue.
    "info":    ("#404040", "#f5f5f5", "ℹ"),
    "success": ("#171717", "#f5f5f5", "✔"),
    "warning": ("#525252", "#fafafa", "⚠"),
    "danger":  ("#000000", "#f0f0f0", "✖"),
    "tip":     ("#404040", "#fafafa", "★"),
    "note":    ("#737373", "#fafafa", "✎"),
    "save":    ("#404040", "#f5f5f5", "💾"),
}
_KPI_ACCENT = {"brand": "#404040", "ok": "#171717", "warn": "#737373", "bad": "#000000", "info": "#525252"}

SECTION_NAMES = {0: "Environment", 1: "Ingest", 2: "Area", 3: "Lattice", 4: "Gates", 5: "Regions",
                 6: "Cleaning", 7: "Ready tables", "7b": "Coord QA", 8: "Population QA", 9: "Statistical QA",
                 10: "EDA", 11: "Scorecard", 12: "Maps", 13: "HTML atlas", 14: "Spatial QA",
                 15: "LiDAR validation", 16: "GeoAI protocol", 17: "StageStore", 18: "Ledger & exports",
                 19: "Executive summary"}

SECTION_NAMES.update({
    "E0": "Setup", "E2": "LULC ingest", "E3": "Accuracy", "E4": "LULC figures", "E5": "Missingness",
    "E6": "Tree biomass", "E7": "Land use", "E8": "Shrubs", "E9": "Correlations", "E10": "ESDA",
    "E11": "Height model", "E12": "Maps", "E13": "Summary",
    "N0": "Setup", "N1": "Pipeline", "N2": "Gates", "N3": "Atlas", "N4": "Deliverables",
})

_ACRONYMS = {"dbh", "lucc", "fv", "su", "chm", "ecdf", "mlp", "rmse", "mae", "gis", "nfi", "uid", "csv",
             "iqr", "cdf", "tvt", "geoai", "ha", "eda", "qa", "gpkg", "html", "osm", "wri", "kpi", "gedi",
             "agb", "bgb", "rh", "l2a", "3d", "lulc", "esda", "oa", "cv", "rf", "pca", "mice", "ml"}
_T0, _SEC = time.time(), None
RUN_LOG = []          # one row per finished section: used by the executive summary
FINDINGS_LOG = []     # every findings() panel is recorded here: used by the executive summary and report export


def _section_order(x):
    """Sort key. Part A ids are 0 to 19 and ids like '7b'. Part B ids are 'E0' to 'E13'. Part C ids are 'N0' to 'N5'."""
    if isinstance(x, bool):
        return (float(int(x)), "")
    if isinstance(x, (int, float)):
        return (float(x), "")
    s = str(x).strip()
    m = re.match(r"^([A-Za-z]?)(\d+)([a-zA-Z]*)$", s)
    if m:
        base = {"": 0, "E": 100, "N": 200}.get(m.group(1).upper(), 300)
        return (base + float(m.group(2)), m.group(3).lower())
    return (10_000.0, s.lower())


PART_LABEL = {0: "Part A", 1: "Part B", 2: "Part C"}


def _part(x):
    """0 for Part A ids, 1 for Part B ids (E0 to E13), 2 for Part C ids (N0 to N5)."""
    return int(_section_order(x)[0] // 100)


def _esc(x):
    return _html.escape(str(x))


def _pretty(s):
    words = str(s).replace("_", " ").replace("-", " ").split()
    words = [w.upper() if w.lower() in _ACRONYMS else w for w in words]
    out = " ".join(words)
    return out[:1].upper() + out[1:] if out else out


def n_(x, d=0):
    """Thousands-separated number for narrative text."""
    try:
        return f"{float(x):,.{d}f}"
    except Exception:
        return str(x)


def pct(part, whole, d=1):
    """Percentage string, safe against zero or missing denominators."""
    try:
        return f"{100.0 * float(part) / float(whole):.{d}f}%" if float(whole) else "n/a"
    except Exception:
        return "n/a"


def _fmt_value(v):
    if isinstance(v, (bool, np.bool_)):
        return "Yes" if v else "No"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if pd.isna(v):
            return "-"
        return f"{v:,.0f}" if float(v).is_integer() else f"{v:,.2f}"
    return _esc(v)


# ----------------------------------------------------------------------------- banners
def banner(num, title, subtitle=""):
    """Section header with a progress stepper for the current part. Also starts the section timer."""
    global _T0, _SEC
    _T0, _SEC = time.time(), num
    part = _part(num)
    keys = sorted((k for k in SECTION_NAMES if _part(k) == part), key=_section_order)
    last_sec = keys[-1]
    num_key = _section_order(num)
    chips = []
    for n in keys:
        name = SECTION_NAMES[n]
        nk = _section_order(n)
        if nk < num_key:
            st = "background:#404040;color:#f5f5f5"
        elif nk == num_key:
            st = "background:#fff;color:#0a0a0a;font-weight:800;box-shadow:0 0 0 2px rgba(255,255,255,.35)"
        else:
            st = "background:rgba(255,255,255,.12);color:#a3a3a3"
        chips.append(f'<span title="{n}. {_esc(name)}" style="{st};display:inline-block;min-width:22px;'
                     f'text-align:center;padding:1px 5px;margin:2px 3px 0 0;border-radius:11px;font-size:11px">{n}</span>')
    where = f"{PART_LABEL.get(part, 'Part')} · Section {num} of {last_sec}"
    display(HTML(
        f'<div style="background:{BRAND_DK};color:#fff;padding:18px 22px;border-radius:12px;margin:8px 0 12px;'
        f'font-family:{FONT}">'
        f'<div style="font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#a3a3a3">{where}</div>'
        f'<div style="font-size:23px;font-weight:700;margin:3px 0 4px;line-height:1.25">{_esc(title)}</div>'
        f'<div style="font-size:13px;color:#d4d4d4">{_esc(subtitle)}</div>'
        f'<div style="margin-top:10px">{"".join(chips)}</div></div>'))


def done(msg=""):
    """Completion bar with elapsed time since the last banner(); also logs the run time."""
    secs = time.time() - _T0
    RUN_LOG.append({"section": _SEC, "name": SECTION_NAMES.get(_SEC, ""), "seconds": round(secs, 1), "note": msg})
    label = f"Section {_SEC} complete" if _SEC is not None else "Complete"
    extra = f" &nbsp;·&nbsp; {_esc(msg)}" if msg else ""
    display(HTML(
        f'<div style="display:flex;align-items:center;gap:10px;border-left:5px solid #171717;background:{PAPER};'
        f'color:{INK};padding:9px 14px;border-radius:8px;margin:12px 0;font-family:{FONT};font-size:13px">'
        f'<span style="font-size:16px">✔</span><span><b>{label}</b>{extra}</span>'
        f'<span style="margin-left:auto;color:{MUTED}">⏱ {secs:,.1f} s</span></div>'))


def sub(label, text):
    """Lettered sub-step heading inside a section."""
    display(HTML(
        f'<div style="margin:20px 0 8px;font-family:{FONT};font-weight:600;font-size:15px;color:{INK};'
        f'border-bottom:2px solid {LINE};padding-bottom:5px">'
        f'<span style="background:{ACCENT};color:#fff;border-radius:6px;padding:1px 8px;margin-right:9px;'
        f'font-size:12px;font-weight:700">{_esc(label)}</span>{_esc(text)}</div>'))


def callout(kind, title, body=""):
    """Message box. kind: info | success | warning | danger | tip | note."""
    acc, tint, icon = _KIND.get(kind, _KIND["info"])
    body_html = f'<div style="margin-top:3px;font-size:13px;line-height:1.5">{body}</div>' if body else ""
    display(HTML(
        f'<div style="border-left:5px solid {acc};background:{tint};color:{INK};padding:11px 15px;'
        f'border-radius:8px;margin:10px 0;font-family:{FONT}">'
        f'<div style="font-weight:700;color:{acc};font-size:14px">{icon}&nbsp; {title}</div>{body_html}</div>'))


def say(msg, kind="info", code=None):
    """One-line status message. `code` (optional) is shown as a monospace path or value."""
    acc, _, icon = _KIND.get(kind, _KIND["info"])
    code_html = (f' <code style="background:{PAPER};color:{INK};padding:1px 6px;border-radius:5px;'
                 f'font-size:12px">{_esc(code)}</code>') if code is not None else ""
    display(HTML(f'<div style="font-family:{FONT};font-size:13px;color:{INK};margin:2px 0">'
                 f'<span style="color:{acc};font-weight:700;display:inline-block;width:22px">{icon}</span>'
                 f'{msg}{code_html}</div>'))


def progress_bar(frac, label="", kind="ok"):
    frac = 0.0 if frac is None or pd.isna(frac) else max(0.0, min(1.0, float(frac)))
    acc = _KPI_ACCENT.get(kind, ACCENT)
    display(HTML(
        f'<div style="font-family:{FONT};margin:8px 0"><div style="display:flex;justify-content:space-between;'
        f'font-size:12px;color:{MUTED};margin-bottom:3px"><span>{_esc(label)}</span><span><b>{frac:.0%}</b></span></div>'
        f'<div style="background:#e5e5e5;border-radius:8px;height:10px;overflow:hidden">'
        f'<div style="width:{frac*100:.1f}%;height:100%;background:{acc}"></div></div></div>'))


# ----------------------------------------------------------------------------- findings (analysis reporting)
def _plain(t):
    """Strip light HTML so a finding can be exported as plain text."""
    t = re.sub(r"<br\s*/?>", " ", str(t))
    return re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", "", t))).strip()


def findings(items, title="Key findings"):
    """Numbered, plain-language findings written from the data of THIS run.

    items : list of strings (light HTML such as <b> or <code> is allowed).
    Use it right after a table or figure to say what the numbers mean.
    """
    items = [str(i) for i in (items or []) if i]
    if not items:
        return
    for t in items:
        FINDINGS_LOG.append({"section": _SEC, "section_name": SECTION_NAMES.get(_SEC, ""),
                             "topic": title, "finding": _plain(t)})
    rows = "".join(
        f'<div style="display:flex;gap:10px;margin:6px 0;font-size:13px;line-height:1.5">'
        f'<span style="flex:0 0 22px;height:22px;border-radius:50%;background:{ACCENT};color:#fff;'
        f'font-size:11.5px;font-weight:700;text-align:center;line-height:22px">{i}</span>'
        f'<span style="color:{INK}">{t}</span></div>'
        for i, t in enumerate(items, 1))
    display(HTML(
        f'<div style="font-family:{FONT};margin:12px 0;padding:12px 16px;background:#fff;border:1px solid {LINE};'
        f'border-left:5px solid {ACCENT};border-radius:8px">'
        f'<div style="font-weight:700;font-size:11px;letter-spacing:.08em;text-transform:uppercase;color:{MUTED};'
        f'margin-bottom:4px">{_esc(title)}</div>{rows}</div>'))


def report(builder, title="Key findings"):
    """Run `builder()` (returns a list of finding strings) and render it.

    A failure in the narration never stops the pipeline: the tables and figures above stay valid.
    """
    try:
        items = builder()
    except Exception as e:
        say(f"Findings could not be generated ({type(e).__name__}: {_esc(e)}). "
            "Tables and figures above are not affected.", "note")
        return
    findings(items, title)


# ----------------------------------------------------------------------------- KPI cards
def kpi(label, value, target=None, sub="", kind=None):
    """Build one KPI card spec. With `target`, the card shows a match or mismatch mark."""
    if target is not None:
        ok = (value == target)
        kind = kind or ("ok" if ok else "bad")
        sub = sub or f"target {_fmt_value(target)}  {'✔ match' if ok else '✖ mismatch'}"
    return {"label": label, "value": value, "sub": sub, "kind": kind or "brand"}


def kpis(items):
    cards = []
    for it in items:
        acc = _KPI_ACCENT.get(it["kind"], ACCENT)
        val = it["value"] if isinstance(it["value"], str) else _fmt_value(it["value"])
        sub_ = f'<div style="font-size:11.5px;color:{MUTED};margin-top:2px">{_esc(it["sub"])}</div>' if it["sub"] else ""
        cards.append(
            f'<div style="flex:1 1 170px;min-width:150px;background:#fff;border:1px solid {LINE};'
            f'border-top:4px solid {acc};border-radius:10px;padding:11px 14px">'
            f'<div style="font-size:10.5px;letter-spacing:.09em;text-transform:uppercase;color:{MUTED};font-weight:600">'
            f'{_esc(it["label"])}</div>'
            f'<div style="font-size:26px;font-weight:800;color:{INK};line-height:1.2;margin-top:3px">{val}</div>{sub_}</div>')
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:12px;margin:10px 0;font-family:{FONT}">{"".join(cards)}</div>'))


# ----------------------------------------------------------------------------- tables
_STATUS_COLS = {"match", "pass", "passed", "ok", "status_ok"}


def _fmt_num(v, name="", force_float=False):
    if pd.isna(v):
        return "-"
    n = str(name).lower()
    is_code = any(k in n for k in ("year", "_id", "uid", "code", "plot_no", "su_no"))
    if isinstance(v, (int, np.integer)):
        return str(int(v)) if is_code else f"{int(v):,}"
    v = float(v)
    if any(k in n for k in ("lat", "lon")):
        return f"{v:.5f}"
    if v.is_integer() and abs(v) < 1e15 and not force_float:
        return str(int(v)) if is_code else f"{int(v):,}"
    a = abs(v)
    return f"{v:,.1f}" if a >= 1000 else (f"{v:,.2f}" if a >= 1 else f"{v:.4f}")


def _table_html(df, max_rows=25, bars=None, good_high=None, good_low=None, index=None):
    if isinstance(df, pd.Series):
        df = df.to_frame()
    df = pd.DataFrame(df)
    if "geometry" in df.columns:
        df = df.drop(columns="geometry")
    n_rows, n_cols = df.shape
    if n_rows == 0:
        return (f'<div style="font-family:{FONT};font-size:13px;color:{MUTED};padding:10px 14px;'
                f'border:1px dashed {LINE};border-radius:8px">∅ No rows to display '
                f'({n_cols} column{"s" if n_cols != 1 else ""})</div>')
    view = df.head(max_rows)
    if index is None:
        index = not (view.index.name is None and pd.api.types.is_integer_dtype(view.index)
                     and not isinstance(view.index, pd.MultiIndex))
    try:
        assert view.columns.is_unique and view.index.is_unique
        sty = view.style
        if not index:
            sty = sty.hide(axis="index")
        sty = sty.set_uuid("t" + uuid.uuid4().hex[:8])
        sty = sty.set_table_styles([
            {"selector": "", "props": [("border-collapse", "separate"), ("border-spacing", "0"),
                                       ("font-family", FONT), ("font-size", "12.5px"),
                                       ("border", f"1px solid {LINE}"), ("border-radius", "10px"),
                                       ("overflow", "hidden")]},
            {"selector": "th.col_heading", "props": [("background", BRAND_DK), ("color", "#fff"),
                                                     ("padding", "7px 12px"), ("text-align", "left"),
                                                     ("font-weight", "600"), ("white-space", "nowrap")]},
            {"selector": "th.row_heading", "props": [("background", PAPER), ("color", INK),
                                                     ("padding", "6px 12px"), ("text-align", "left"),
                                                     ("font-weight", "600")]},
            {"selector": "th.blank", "props": [("background", BRAND_DK)]},
            {"selector": "td", "props": [("padding", "6px 12px"), ("border-bottom", "1px solid #e5e5e5"),
                                         ("color", INK), ("background", "#fff"), ("max-width", "420px"),
                                         ("overflow-wrap", "anywhere")]},
            {"selector": "tbody tr:nth-child(even) td", "props": [("background", "#fafafa")]},
            {"selector": "tbody tr:hover td", "props": [("background", PAPER)]},
        ])
        num_cols = [c for c in view.columns
                    if pd.api.types.is_numeric_dtype(view[c]) and not pd.api.types.is_bool_dtype(view[c])]
        bool_cols = [c for c in view.columns if pd.api.types.is_bool_dtype(view[c])]
        sty = sty.format(na_rep="-")
        if num_cols:
            _ff = {c: bool(pd.api.types.is_float_dtype(view[c]) and
                           not view[c].dropna().map(lambda x: float(x).is_integer()).all()) for c in num_cols}
            sty = sty.format({c: (lambda v, _c=c: _fmt_num(v, _c, _ff[_c])) for c in num_cols}, na_rep="-")
            sty = sty.set_properties(subset=num_cols, **{"text-align": "right", "font-variant-numeric": "tabular-nums"})
            sty = sty.set_table_styles({c: [{"selector": "th.col_heading", "props": [("text-align", "right")]}]
                                        for c in num_cols}, overwrite=False)
        for c in bool_cols:
            strict = str(c).lower() in _STATUS_COLS
            sty = sty.format({c: (lambda v, _s=strict: ("✔ pass" if v else "✖ fail") if _s else ("✔" if v else "-"))},
                             subset=[c])

            def _weight(v, _s=strict):
                if v:
                    return "font-weight:700"
                return "font-weight:800;text-decoration:underline" if _s else f"color:{MUTED}"
            _map = getattr(sty, "map", None) or sty.applymap
            sty = _map(_weight, subset=[c])
        bars = [c for c in (bars or []) if c in num_cols]
        if bars:
            sty = sty.bar(subset=bars, color="#d4d4d4", vmin=0)
        gh = [c for c in (good_high or []) if c in num_cols]
        gl = [c for c in (good_low or []) if c in num_cols]
        if gh:
            sty = sty.background_gradient(cmap="Greys", subset=gh)
        if gl:
            sty = sty.background_gradient(cmap="Greys_r", subset=gl)
        body = sty.to_html()
    except Exception:
        body = view.to_html(border=0, na_rep="-", max_rows=max_rows)
    shown = f"Showing first {min(n_rows, max_rows):,} of {n_rows:,} rows" if n_rows > max_rows else f"{n_rows:,} rows"
    foot = (f'<div style="font-family:{FONT};font-size:11.5px;color:{MUTED};margin:4px 2px 0">'
            f'{shown} · {n_cols:,} column{"s" if n_cols != 1 else ""}</div>')
    return f'<div style="overflow-x:auto;max-width:100%">{body}</div>{foot}'


def _title_html(title, caption=None):
    if not title:
        return ""
    cap = f'<div style="font-size:12px;color:{MUTED};margin-top:1px">{caption}</div>' if caption else ""
    return (f'<div style="font-family:{FONT};margin:14px 0 6px"><div style="font-weight:700;font-size:14px;color:{INK}">'
            f'▦&nbsp; {_esc(title)}</div>{cap}</div>')


def show(df, title=None, caption=None, max_rows=25, bars=None, good_high=None, good_low=None, index=None):
    """Styled table. bars= data-bars; good_high / good_low= grey heat-maps (darker = higher / lower)."""
    display(HTML(_title_html(title, caption) + _table_html(df, max_rows, bars, good_high, good_low, index)))


def show_pair(left, right, left_title="Before", right_title="After", max_rows=12, index=None):
    """Two tables side by side (for example before and after cleaning)."""
    def cell(df, t):
        return (f'<div style="flex:1 1 320px;min-width:280px">{_title_html(t)}'
                f'{_table_html(df, max_rows, index=index)}</div>')
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:18px">{cell(left, left_title)}{cell(right, right_title)}</div>'))


def checklist(df, item_col="item", pass_col="pass", title="Readiness checklist"):
    """Pass / open checklist with a progress bar."""
    rows, n_ok = [], 0
    for _, r in df.iterrows():
        ok = bool(r[pass_col]); n_ok += ok
        acc, tint = ("#171717", PAPER) if ok else ("#737373", "#fafafa")
        rows.append(
            f'<div style="display:flex;align-items:center;gap:10px;padding:7px 12px;margin:4px 0;border-radius:8px;'
            f'background:{tint};border-left:4px solid {acc};font-size:13px;color:{INK}">'
            f'<span style="color:{acc};font-weight:800;width:18px">{"✔" if ok else "●"}</span>'
            f'<span style="flex:1">{_esc(_pretty(r[item_col]))}</span>'
            f'<span style="color:{acc};font-weight:700;font-size:12px">{"PASS" if ok else "OPEN"}</span></div>')
    display(HTML(_title_html(title) + f'<div style="font-family:{FONT}">{"".join(rows)}</div>'))
    progress_bar(n_ok / max(len(df), 1), f"{n_ok} of {len(df)} readiness items passed", "ok" if n_ok == len(df) else "warn")


# ----------------------------------------------------------------------------- figures
def _embed(p: Path, max_px: int = 1500):
    """Return (mime, base64) for a figure. Wide images are down-scaled for the screen only;
    the full-resolution file stays on disk. Keeps the notebook light."""
    mime = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".svg": "image/svg+xml"}.get(p.suffix.lower(), "image/png")
    raw = p.read_bytes()
    if p.suffix.lower() in (".png", ".jpg", ".jpeg"):
        try:
            from PIL import Image as _PIL
            im = _PIL.open(io.BytesIO(raw))
            if im.width > max_px:
                im = im.resize((max_px, int(im.height * max_px / im.width)), _PIL.LANCZOS)
                buf = io.BytesIO()
                if p.suffix.lower() == ".png":
                    im.save(buf, format="PNG", optimize=True)
                else:
                    im.convert("RGB").save(buf, format="JPEG", quality=88)
                raw = buf.getvalue()
        except Exception:
            pass
    return mime, base64.b64encode(raw).decode()


def _figure_analysis(path, title=None) -> str:
    """Generic reading guide chosen from the figure's name or title (used only when the caller gives none)."""
    stem = Path(str(path)).stem.lower() if path else ""
    label = (title or stem).lower()
    text = f"{stem} {label}"
    rules = [
        (("region_su", "su board", "counts by region", "regional share", "su_dashboard"),
         "Left to right: SU counts per region, each region's share of the grid, and the split between SUs with trees and Gate 1 zero-tree SUs."),
        (("missingness",),
         "Each bar is the share of missing values in one field. Tall bars are fields to treat with care before modelling or mapping."),
        (("corr", "spearman", "pearson", "bivariate"),
         "Cell colour shows the strength of the link between two metrics (Spearman = rank based, Pearson = linear). Correlation is not causation."),
        (("categorical", "frequencies", "tree attributes", "attribute_dashboard"),
         "Each panel ranks the categories of one attribute. Long bars are dominant classes; short bars are rare but may still matter."),
        (("field_vs_office", "coord_crosscheck", "coordinate", "field vs office"),
         "Field GPS and office/database coordinates are compared. Overlapping distributions and small displacements mean the two sources agree."),
        (("field_tree_height", "field_height"),
         "Distribution of field-measured tree height. The peak and the median describe the typical canopy height recorded."),
        (("field_vs_chm", "gedi", "lidar", "bland", "field_lidar"),
         "Left: paired field and remote height (points on the 1:1 line agree). Middle and right: residuals and Bland-Altman show bias and spread."),
        (("univariate", "distribution", "ecdf", "kde"),
         "Shape of one variable: centre, spread and tails. Outliers appear as long tails; none are deleted."),
        (("gate", "stump", "access", "ownership"),
         "Category mix of the plotted attribute. Dominant classes drive map symbology; rare classes stay visible for QA."),
        (("map_", "asir_grid", "basemap", "map board"),
         "Spatial layout of the analytical grid and related attributes on an imagery or topographic basemap."),
        (("species", "health", "dbh", "carbon", "density"),
         "Thematic summary of the mapped attribute. Read the legend together with the chart inset."),
    ]
    for keys, msg in rules:
        if any(k in text for k in keys):
            return msg
    pretty = title or Path(str(path)).stem.replace("_", " ")
    return f"'{pretty}' summarises one plotted result. Read the axis labels and legend first, then look for the overall pattern and its exceptions."


def _note_block(label, note):
    if not note:
        return ""
    if isinstance(note, (list, tuple)):
        body = "".join(f'<div style="margin:2px 0">• {t}</div>' for t in note)
    else:
        body = str(note)
    return (f'<div style="margin-top:10px;padding:10px 12px;background:{PAPER};border-left:3px solid {ACCENT};'
            f'border-radius:6px;font-size:12.5px;line-height:1.45;color:{INK}">'
            f'<div style="font-weight:700;font-size:11px;letter-spacing:.06em;text-transform:uppercase;'
            f'color:{MUTED};margin-bottom:4px">{label}</div>{body}</div>')


def show_fig(path, title=None, caption=None, analysis=None, read=None):
    """Figure card.

    analysis : finding specific to this run (string or list of strings), shown as 'Finding'.
    read     : how to read this kind of chart, shown as 'How to read'. When neither is given,
               a generic reading guide is chosen from the figure name.
    """
    if not path:
        return
    p = Path(str(path))
    if not p.exists():
        say("Figure not found:", "warning", code=p)
        return
    mime, b64 = _embed(p)
    if analysis is None and read is None:
        read = _figure_analysis(p, title)
    cap = f'<figcaption style="font-size:12px;color:{MUTED};margin-top:8px">{caption}</figcaption>' if caption else ""
    src = f'<div style="font-size:11px;color:{MUTED};margin-top:8px">Full-resolution file: <code>{_esc(p)}</code></div>'
    display(HTML(
        f'<figure style="margin:14px 0;padding:12px 14px;background:#fff;border:1px solid {LINE};border-radius:12px;'
        f'font-family:{FONT}">'
        f'<div style="margin-bottom:8px"><span style="font-weight:700;font-size:13.5px;color:{INK}">◆&nbsp; {_esc(title or _pretty(p.stem))}</span></div>'
        f'<img src="data:{mime};base64,{b64}" style="max-width:100%;height:auto;display:block;margin:0 auto;border-radius:6px"/>'
        f'{cap}{_note_block("Finding", analysis)}{_note_block("How to read", read)}{src}</figure>'))


def show_figs(items):
    """items: path | (path, title) | (path, title, read). Skips empty entries."""
    for it in items:
        if isinstance(it, (tuple, list)):
            if not it or it[0] is None:
                continue
            if len(it) >= 3:
                show_fig(it[0], it[1], read=it[2])
            elif len(it) == 2:
                show_fig(it[0], it[1])
            else:
                show_fig(it[0])
        elif it is not None:
            show_fig(it)


def show_fig_grid(items, *, ncols=2, title="Map board", caption=None, analysis=None, read=None):
    """Several PNG/JPG figures in one grid card, with an optional finding and reading guide."""
    tiles, stems = [], []
    for it in items:
        path, ttl = (it if isinstance(it, (tuple, list)) else (it, None))
        if not path:
            continue
        p = Path(str(path))
        if not p.exists():
            continue
        mime, b64 = _embed(p, max_px=1100)
        stems.append(str(p.stem))
        tiles.append(
            f'<div style="min-width:0;background:#fafafa;border:1px solid {LINE};border-radius:10px;padding:8px">'
            f'<div style="font-size:12px;font-weight:700;color:{INK};margin:0 0 6px">{_esc(ttl or _pretty(p.stem))}</div>'
            f'<img src="data:{mime};base64,{b64}" style="width:100%;height:auto;display:block;border-radius:6px"/></div>')
    if not tiles:
        say("No figures for grid", "warning")
        return
    if analysis is None and read is None:
        read = _figure_analysis(" ".join(stems), title)
    cap = f'<div style="font-size:12px;color:{MUTED};margin-top:8px">{caption}</div>' if caption else ""
    display(HTML(
        f'<figure style="margin:14px 0;padding:14px 16px;background:#fff;border:1px solid {LINE};border-radius:12px;font-family:{FONT}">'
        f'<div style="font-weight:700;font-size:14px;color:{INK};margin-bottom:10px">◆&nbsp; {_esc(title)}</div>'
        f'<div style="display:grid;grid-template-columns:repeat({int(ncols)},minmax(0,1fr));gap:12px">'
        + "".join(tiles) +
        f'</div>{cap}{_note_block("Finding", analysis)}{_note_block("How to read", read)}</figure>'))


# ----------------------------------------------------------------------------- StageStore + outputs
_SAVE_LOG = []


def show_code_details(title: str, code: str):
    """Collapsible block that documents implementation without cluttering the page."""
    display(HTML(
        f'<details style="margin:10px 0;padding:8px 12px;border:1px solid {LINE};border-radius:8px;'
        f'background:{PAPER};font-family:{FONT}">'
        f'<summary style="cursor:pointer;font-weight:700;color:{INK};font-size:13px">{_esc(title)}</summary>'
        f'<pre style="margin:10px 0 4px;padding:10px;background:#fff;border:1px solid {LINE};'
        f'border-radius:6px;overflow:auto;font-size:11px;line-height:1.4;color:{INK}">'
        f'{_esc(code)}</pre>'
        f'<div style="font-size:11px;color:{MUTED}">Implementation lives in <code>src/nfi_area_maps.py</code> '
        f'and <code>src/nfi_beautiful_maps.py</code>. Maps open in the HTML atlas for zoom and pan.</div>'
        f'</details>'))


def show_html_map(path, title=None, *, open_browser=False):
    """Link to a Folium / HTML map. Files live under OUT (outside the Jupyter root), so an IFrame would
    return 403; a file:// link and an optional system-browser open are used instead."""
    import webbrowser
    p = Path(path)
    if not p.exists():
        callout("warning", "HTML map missing", f"<code>{_esc(p)}</code>")
        return
    uri = p.resolve().as_uri()
    label = _esc(title or p.name)
    display(HTML(
        f'<div style="margin:8px 0 12px;padding:10px 12px;border:1px solid {LINE};border-radius:10px;background:#fff;'
        f'font-family:{FONT}">'
        f'<div style="font-weight:700;color:{INK};margin-bottom:4px">{label}</div>'
        f'<a href="{uri}" target="_blank" rel="noopener" '
        f'style="color:{INK};font-weight:600;text-decoration:none;border-bottom:1px solid {MUTED}">Open map in browser</a>'
        f'<div style="margin-top:4px;font-size:11px;color:{MUTED}"><code>{_esc(p)}</code></div></div>'))
    if open_browser:
        webbrowser.open(uri)


def show_html_atlas(paths: dict, *, preview_keys=None, open_index=True):
    """List atlas HTML products with working file:// links; open the portal index once."""
    import webbrowser
    index = paths.get("index")
    if index and Path(index).exists():
        uri = Path(index).resolve().as_uri()
        display(HTML(
            f'<div style="margin:10px 0;padding:12px 14px;border-left:5px solid {ACCENT};background:{PAPER};'
            f'border-radius:8px;font-family:{FONT}">'
            f'<div style="font-weight:700;color:{INK}">Atlas portal</div>'
            f'<a href="{uri}" target="_blank" rel="noopener" style="color:{INK};font-weight:600">Open index.html in browser</a>'
            f'<div style="font-size:11px;color:{MUTED};margin-top:4px"><code>{_esc(index)}</code></div>'
            f'<div style="font-size:12px;margin-top:6px;color:{MUTED}">'
            f'Maps sit outside the Jupyter root, so they open in your system browser (this avoids a 403 Forbidden error).'
            f'</div></div>'))
        if open_index:
            webbrowser.open(uri)
    keys = preview_keys or ["tree_density", "species_diversity", "tree_health", "dbh", "carbon",
                            "access", "ownership", "species_points"]
    rows = []
    for key in keys:
        p = paths.get(key)
        if not p or not Path(p).exists():
            continue
        uri = Path(p).resolve().as_uri()
        rows.append(
            f'<tr><td style="padding:4px 8px;font-weight:600">{_esc(_pretty(key))}</td>'
            f'<td style="padding:4px 8px"><a href="{uri}" target="_blank" rel="noopener">Open</a></td>'
            f'<td style="padding:4px 8px;font-size:11px;color:{MUTED}"><code>{_esc(Path(p).name)}</code></td></tr>')
    if rows:
        display(HTML(
            f'<div style="font-family:{FONT};margin:8px 0"><div style="font-weight:700;color:{INK};margin-bottom:6px">'
            f'Quick map links</div><table style="border-collapse:collapse;width:100%">{chr(10).join(rows)}</table></div>'))


def install_stage_logger(store):
    """Wrap store.save so each cell can report what it wrote (call flush_saved() at the end of a cell)."""
    try:
        orig = store.save

        def logged(stage, name, obj, *a, **kw):
            r = orig(stage, name, obj, *a, **kw)
            try:
                n = len(obj)
            except Exception:
                n = None
            _SAVE_LOG.append({"stage": stage, "product": name, "rows": n, "note": kw.get("note", "")})
            return r
        store.save = logged
    except Exception as e:
        say(f"StageStore logging unavailable ({e}); saves still work.", "warning")


def flush_saved(title="StageStore products written"):
    if _SAVE_LOG:
        show(pd.DataFrame(_SAVE_LOG), title=title, bars=["rows"])
        _SAVE_LOG.clear()


def file_inventory(folder, pattern="*", title=None, limit=30):
    folder = Path(folder)
    rows = []
    for f in sorted(folder.glob(pattern), key=lambda x: x.stat().st_mtime, reverse=True)[:limit]:
        if f.is_file():
            st = f.stat()
            rows.append({"file": f.name, "size_mb": st.st_size / 1e6,
                         "modified": pd.Timestamp(st.st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")})
    show(pd.DataFrame(rows), title=title or f"Files in {folder.name}/", bars=["size_mb"], max_rows=limit)


def output_overview(root):
    root = Path(root)
    items = []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        n = sum(1 for f in d.rglob("*") if f.is_file())
        items.append(kpi(d.name, n, sub="files"))
    if items:
        kpis(items)


def export_report_html(path, title, headline, open_items=None, next_steps=None):
    """Write one self-contained, neutral HTML report from the findings recorded during this run.

    headline   : list of (label, value) pairs shown as cards at the top.
    open_items : list of (item, action) pairs still awaiting sign-off.
    next_steps : list of strings.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    cards = "".join(
        f'<div class="card"><div class="k">{_esc(k)}</div><div class="v">{_esc(v)}</div></div>' for k, v in headline)
    by_sec = {}
    for r in FINDINGS_LOG:
        by_sec.setdefault((r["section"], r["section_name"]), []).append(r)
    body = ""
    for (sec, name), rows in sorted(by_sec.items(), key=lambda kv: _section_order(kv[0][0])):
        body += f'<h2>{_esc(sec)} · {_esc(name)}</h2>'
        for r in rows:
            body += f'<p><span class="tag">{_esc(r["topic"])}</span> {_esc(r["finding"])}</p>'
    oi = "".join(f"<li><b>{_esc(_pretty(a))}</b>: {_esc(b)}</li>" for a, b in (open_items or []))
    ns = "".join(f"<li>{_esc(x)}</li>" for x in (next_steps or []))
    doc = (
        f'<!doctype html><html lang="en"><head><meta charset="utf-8"><title>{_esc(title)}</title>'
        '<style>body{font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;color:#171717;'
        'max-width:920px;margin:32px auto;padding:0 20px;line-height:1.55}'
        'h1{font-size:26px;margin:0 0 4px}h2{font-size:17px;border-bottom:2px solid #d4d4d4;padding-bottom:4px;margin-top:28px}'
        '.sub{color:#525252;font-size:13px;margin-bottom:18px}.cards{display:flex;flex-wrap:wrap;gap:10px}'
        '.card{flex:1 1 160px;border:1px solid #d4d4d4;border-top:4px solid #404040;border-radius:8px;padding:10px 12px}'
        '.k{font-size:11px;text-transform:uppercase;letter-spacing:.08em;color:#525252}.v{font-size:24px;font-weight:800}'
        '.tag{font-size:11px;background:#f5f5f5;border:1px solid #d4d4d4;border-radius:5px;padding:1px 6px;margin-right:6px;color:#404040}'
        'li{margin:4px 0}</style></head><body>'
        f'<h1>{_esc(title)}</h1><div class="sub">Generated {pd.Timestamp.now():%Y-%m-%d %H:%M}. '
        'Every statement below was computed from the data of this run.</div>'
        f'<div class="cards">{cards}</div>'
        + (f'<h2>Open items awaiting sign-off</h2><ul>{oi}</ul>' if oi else '')
        + (f'<h2>Recommended next steps</h2><ol>{ns}</ol>' if ns else '')
        + body + '</body></html>')
    path.write_text(doc, encoding="utf-8")
    return path


def run_times_table():
    """Section run times collected by done(); used by the executive summary."""
    return pd.DataFrame(RUN_LOG)


# ----------------------------------------------------------------------------- matplotlib theme
def apply_chart_theme():
    """Chart theme: white canvas, light grid, colour-blind-safe cycle. Colour is free inside charts."""
    mpl.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
        "figure.facecolor": "white", "axes.facecolor": "white",
        "axes.prop_cycle": cycler(color=CHART_COLORS),
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": "#a3a3a3", "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": INK, "ytick.color": INK,
        "axes.grid": True, "grid.color": "#e5e5e5", "grid.linestyle": "-", "grid.linewidth": 0.6,
        "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 13, "axes.labelsize": 10.5,
        "legend.frameon": False, "lines.linewidth": 2.0,
    })


apply_fancy_theme = apply_chart_theme      # backwards-compatible name used by later cells
apply_chart_theme()
say("Presentation kit loaded: neutral banners, KPI cards, styled tables, findings panels, figure cards, colour-safe chart theme.", "success")


<a id="s0"></a>

## 0 · Environment, configuration and StageStore

Loads the project modules, reads `config.yaml`, creates the output folders, checks that every input file exists, and shows the baked Gate targets that every later check is compared against.

**About the code below**

| | |
|:--|:--|
| **Reads** | `config.yaml`; project modules under `src/`. |
| **Does** | (1) imports and clears stale `nfi_*` modules, (2) loads the config and creates the output root, (3) checks the input files, (4) shows the Gate targets. |
| **Writes** | The output folder tree (`audit/`, `gates/`, `figures/`, `maps/`, `validated/`, `stages/`, ...). Defines `CFG`, `OUT`, `STAGES`, `GATE`. |
| **Check before moving on** | Every row of the *Input files* table shows ✔. If one shows -, fix the path in `config.yaml` before running on. |

> **Why it matters.** A missing input otherwise fails several sections later with a less helpful message.

> **Tip:** modules named `nfi_*` are dropped from the import cache on each run, so edits in `src/` are picked up without restarting the kernel. Edit the *user switches* at the top of the code cell to change how maps open.

In [ ]:
# =============================================================================
#  SECTION 0 · ENVIRONMENT, CONFIGURATION AND STAGESTORE
#  Reads  : config.yaml and the project modules under src/
#  Does   : imports, loads config, creates output folders, checks that every input file exists
#  Writes : the output root (audit/, gates/, figures/, maps/, validated/, stages/, ...)
#  Result : CFG (config), OUT (output root), STAGES (intermediate store), GATE (targets)
# =============================================================================
from pathlib import Path
import re
import sys, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from IPython.display import Image, display as idisplay

# ---- USER SWITCHES · edit here ---------------------------------------------------------
OPEN_MAPS_IN_BROWSER = True   # True: HTML maps open in your browser as they are built; False: links only

banner(0, "Environment, configuration & StageStore",
       "Imports · config.yaml · output root · baked Gate targets")

# ---- 1. quiet noisy warnings and set display options ------------------------------------
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# ---- 2. make the project modules importable ------------------------------------------
SRC = Path("src").resolve()
sys.path.insert(0, str(SRC))
# Drop stale nfi_* modules so notebook re-runs pick up src edits without kernel restart
for _mod in [m for m in list(sys.modules) if m == "nfi_audit" or m.startswith("nfi_")]:
    del sys.modules[_mod]

# ---- 3. import project functions (grouped by module) ----------------------------------
from nfi_audit import (
    load_config, ensure_output_dirs, scan_files, read_first_sheet, attach_keys,
    validate_key, reconcile, plot_coordinate_qa, load_asir_su_universe,
    aseer_gate1_zero_tree_sus, aseer_gate2_plot_completeness, compare_to_doc_targets,
    biological_tree_flags, table_presence_matrix, row_conservation, make_su_uid,
    anti_clip_su_tree_frame, demonstrate_clip_loss, plots_to_geodataframe,
    left_sjoin_plots_to_su_grid, StageStore, lexical_quality_audit,
    apply_derived_cleaning,
    cleaning_guide_notes,
    field_type_audit,
    build_cleaning_corrections_report,
    build_ready_clean_deliverable,
    narrate_cleaning_story,
    narrate_ready_deliverable_story,
    narrate_region_su_story,
)
from nfi_area_maps import (
    area_appropriation_audit, longform_su_plot_tree_presence,
    attach_grid_area_by_key, region_governorate_summaries, map_asir_grid_regions,
    interactive_asir_maps,
)
from nfi_beautiful_maps import (
    build_beautiful_html_atlas,
    build_su_map_metrics,
)
from nfi_datascience import (
    missingness_report, univariate_numeric, outlier_iqr_flags, bivariate_numeric,
    plot_univariate_grid, plot_corr_heatmap, plot_corr_pair, plot_missingness_bars, plot_missingness_profile, plot_missingness_pair, plot_boxplots,
    plot_old_vs_new_bars,
    compare_old_vs_new_method, accuracy_scorecard, apply_plot_style,     eda_limitations_nfi,
    plot_categorical_bars,
    plot_categorical_grid,
    plot_grouped_boxplots,
    plot_scatter_matrix,
    plot_ecdf,
    eda_readout_bivariate,
    eda_readout_numeric,
    bivariate_cat_numeric,
    univariate_categorical,
    engineer_tree_volume_features_fv, fv_deliverable_frame, FV_DELIVERABLE_COLS,
    build_asir_statistical_figure_pack,
    plot_region_su_dashboard,
    field_vs_office_coordinate_crosscheck,
)

# ---- 4. load the configuration and create the output folders ---------------------------
CFG = load_config("config.yaml")

# --- Colab/portable: resolve relative config paths against cwd ---
def resolve_cfg_paths(cfg):
    root = Path.cwd()
    # Do not path-resolve GIS attribute keys (asir_su_key=SU caused KeyError in Colab).
    skip_keys = {'asir_su_key', 'asir_grid_key'}
    path_exts = ('.xlsx', '.xls', '.csv', '.shp', '.gpkg', '.docx', '.pdf', '.md', '.yaml', '.yml', '.js', '.json')
    for section in ('paths', 'tables', 'gis', 'documents_reference_only'):
        if section not in cfg:
            continue
        for k, v in list(cfg[section].items()):
            if k in skip_keys or not isinstance(v, str) or not v:
                continue
            if Path(v).is_absolute():
                continue
            looks_path = (
                '/' in v or '\\' in v or v.startswith('data') or v.startswith('outputs')
                or v.lower().endswith(path_exts)
            )
            if looks_path:
                cfg[section][k] = str((root / v).resolve())
    return cfg
CFG = resolve_cfg_paths(CFG)
OUT = ensure_output_dirs(CFG)
STAGES = StageStore(OUT)
install_stage_logger(STAGES)          # lets each section report what it saved
GATE = CFG["aseer_gate_expected"]

# ---- 5. check that every input file exists (fail early, with a clear message) ---------------
def _input_row(label, path):
    p = Path(str(path))
    return {"input": label, "file": p.name, "found": p.exists(), "folder": str(p.parent)}


input_check = pd.DataFrame(
    [_input_row(f"table · {k}", v) for k, v in CFG["tables"].items()]
    + [_input_row("gis · asir_grid", CFG["gis"]["asir_grid"])]
)
_missing = input_check.loc[~input_check["found"], "input"].tolist()

# ---- 6. summarise the environment and the Gate targets ------------------------------
env = pd.DataFrame([
    ("Output root", str(OUT)),
    ("Source modules", str(SRC)),
    ("nfi_audit", __import__("nfi_audit").__file__),
    ("anti_clip_su_tree_frame available", "yes" if callable(anti_clip_su_tree_frame) else "NO"),
    ("Python", platform.python_version()),
    ("pandas / geopandas / numpy", f"{pd.__version__} / {gpd.__version__} / {np.__version__}"),
], columns=["item", "value"])
show(env, title="Runtime environment")
show(input_check, title="Input files", caption="Every input named in config.yaml. A missing file (-) must be fixed before section 1.")

kpis([
    kpi("Sampling units (universe)", GATE.get("total_sus"), sub="Gate target"),
    kpi("Gate 1 · SUs without trees", GATE.get("sus_no_trees"), sub="Gate target", kind="warn"),
    kpi("Expected plots (430 × 5)", GATE.get("expected_plots"), sub="Gate target"),
    kpi("Gate 2 · plots lacking trees", GATE.get("gate2_plots"), sub="Gate target", kind="warn"),
])
show(pd.Series(GATE, name="baked_target").to_frame(),
     title="Baked Gate targets", caption="From the Validation Point 1 and 2 packages")
if not callable(anti_clip_su_tree_frame):
    callout("danger", "anti_clip_su_tree_frame is not callable", "Check the <code>src/nfi_audit</code> import.")
elif _missing:
    callout("warning", f"{len(_missing)} input file(s) not found",
            "Fix these paths in <code>config.yaml</code> before running section 1:<br>"
            + "<br>".join(f"• <code>{m}</code>" for m in _missing))
else:
    callout("success", "Environment ready",
            f"All inputs found. Output root: <code>{OUT}</code><br>Every stage below writes reviewable products there.")
done()



<a id="s1"></a>

## 1 · Tabular ingest and Asir sampling-unit universe

Reads every source table, attaches the join keys (`SU_UID`, `PLOT_UID`), and loads the Asir GIS master.

**About the code below**

| | |
|:--|:--|
| **Reads** | The Excel tables named in `config.yaml` and the Asir grid shapefile. |
| **Does** | (1) reads each table and attaches keys, (2) lists what was read, (3) loads the Asir SU master and grid, (4) asserts the universe has exactly the expected sampling units. |
| **Writes** | StageStore products under `01_load` (SU master, plots, trees, Asir SU, Asir grid). |
| **Check before moving on** | The *Sampling units in Asir master* card shows ✔ match (430), and the findings panel lists no empty tables. |

> **Why it matters.** Every later join keeps this universe as its LEFT master, so an error here would spread everywhere.

In [ ]:
# =============================================================================
#  SECTION 1 · TABULAR INGEST AND ASIR SAMPLING-UNIT UNIVERSE
#  Reads  : the Excel tables named in config.yaml and the Asir grid shapefile
#  Does   : reads each table, attaches join keys (SU_UID, PLOT_UID), loads the Asir SU master, asserts the universe
#  Writes : StageStore stage 01_load
#  Result : su, plot, trees, asir_su, asir_grid (all keyed by SU_UID)
# =============================================================================
banner(1, "Tabular ingest & Asir sampling-unit universe",
       "Read source tables · attach join keys · load the 430-SU GIS master")

# ---- 1. read every source table and attach its join keys ------------------------------
tables_raw, tables = {}, {}
ingest_log = []
for name, path in CFG["tables"].items():
    df = read_first_sheet(path)
    tables_raw[name] = df
    if name == "su":
        tables[name] = attach_keys(df, CFG, kind="su")
    elif name == "plot":
        tables[name] = attach_keys(df, CFG, kind="plot")
    elif name == "fauna":
        tables[name] = attach_keys(df, CFG, kind="fauna")
    elif name == "teams":
        t = df.copy(); t["SU_UID"] = make_su_uid(t[CFG["columns"]["su_id_in_child_tables"]]); tables[name] = t
    else:
        tables[name] = attach_keys(df, CFG, kind="child")
    ingest_log.append({"table": name, "rows": len(df), "columns": df.shape[1], "source_file": Path(str(path)).name})

# ---- 2. show what was read ----------------------------------------------------------
show(pd.DataFrame(ingest_log), title="Source tables ingested", bars=["rows"], max_rows=40)

# ---- 3. load the Asir GIS master and assert the universe size ---------------------------
su, plot, trees = tables["su"], tables["plot"], tables["trees"]
asir_su = load_asir_su_universe(CFG)
asir_grid = gpd.read_file(CFG["gis"]["asir_grid"])
asir_grid["SU_UID"] = make_su_uid(asir_grid[CFG["gis"]["asir_grid_key"]])
assert len(asir_su) == GATE["total_sus"]

kpis([
    kpi("Sampling units in Asir master", len(asir_su), target=GATE["total_sus"]),
    kpi("Grid cells", len(asir_grid)),
    kpi("Plots (national table)", len(plot)),
    kpi("Trees (national table)", len(trees)),
])
callout("success", "Universe verified",
        f"The Asir GIS master holds exactly <b>{len(asir_su):,}</b> sampling units, matching the Gate target. "
        "All later joins keep this table as the LEFT master.")

# ---- 4. how much of the national data belongs to the Asir universe? -----------------------
_asir_ids = set(asir_su["SU_UID"].astype(str))
_trees_in = int(trees["SU_UID"].astype(str).isin(_asir_ids).sum())
_plots_in = int(plot["SU_UID"].astype(str).isin(_asir_ids).sum())
_empty_tables = [r["table"] for r in ingest_log if r["rows"] == 0]
report(lambda: [
    f"<b>{n_(len(ingest_log))}</b> source tables were read; the largest is <b>{max(ingest_log, key=lambda r: r['rows'])['table']}</b> "
    f"with {n_(max(r['rows'] for r in ingest_log))} rows.",
    f"<b>{n_(_plots_in)}</b> of {n_(len(plot))} plot rows ({pct(_plots_in, len(plot))}) and <b>{n_(_trees_in)}</b> of {n_(len(trees))} tree rows "
    f"({pct(_trees_in, len(trees))}) belong to sampling units of the Asir grid. Only these enter the analysis.",
    ("Empty tables: <b>" + ", ".join(_empty_tables) + "</b>. Check the source files.") if _empty_tables else
    "No source table is empty.",
])

STAGES.save("01_load", "su_master", su, note="national SU table with SU_UID")
STAGES.save("01_load", "plot", plot, note="national plots with PLOT_UID")
STAGES.save("01_load", "trees", trees, note="national trees long")
STAGES.save("01_load", "asir_su", asir_su, note="430-SU GIS universe")
STAGES.save("01_load", "asir_grid", asir_grid, note="430 grid cells with Area_H")
flush_saved()
done()


<a id="s2"></a>

## 2 · Inventory-cell area appropriation and policy checks

Compares the attribute `Area_H` with an equal-area recomputation and with the Web-Mercator area that careless reprojection would give, then records the policy checks.

**About the code below**

| | |
|:--|:--|
| **Reads** | The Asir grid with its `Area_H` attribute. |
| **Does** | (1) runs the area audit, (2) shows the policy table and the summary, (3) saves the audit tables, (4) writes findings on how `Area_H` compares with the recomputed areas. |
| **Writes** | `audit/area_appropriation_*.csv`, `audit/grid_area_qa_detail.csv`, StageStore `02_area`. |
| **Check before moving on** | Read the findings panel. If it warns about Web-Mercator inflation, confirm with the data owner how `Area_H` was produced before it is used as a denominator. |

> **Why it matters.** Stems per hectare and any design-based expansion divide by area. A systematic area error of about 10% changes every per-hectare result by the same amount.

> **Policy:** stems per hectare stay blocked until `plot_area_ha` exists (design principle 3).

In [ ]:
# =============================================================================
#  SECTION 2 · INVENTORY-CELL AREA APPROPRIATION AND POLICY CHECKS
#  Reads  : asir_grid (with its Area_H attribute)
#  Does   : compares Area_H with an equal-area recomputation and with the Web-Mercator area; records the policy checks
#  Writes : audit/area_appropriation_*.csv, audit/grid_area_qa_detail.csv, StageStore 02_area
#  Result : area (dict of audit tables), used again in section 3
# =============================================================================
banner(2, "Area appropriation & policy checks",
       "Attach Area_H by SU key · compare with equal-area and Mercator-misuse areas")

# ---- 1. run the area audit ----------------------------------------------------------
area = area_appropriation_audit(asir_grid, CFG)
show(area["policy"], title="Area policy checks")
show(area["summary"], title="Area appropriation summary")
# ---- 2. save the audit tables ---------------------------------------------------------
area["summary"].to_csv(OUT / "audit" / "area_appropriation_summary.csv", index=False)
area["policy"].to_csv(OUT / "audit" / "area_appropriation_policy.csv", index=False)
area["grid_area"].drop(columns="geometry", errors="ignore").to_csv(
    OUT / "audit" / "grid_area_qa_detail.csv", index=False
)
STAGES.save("02_area", "grid_area_qa", area["grid_area"], note="Area_H attr vs equal-area vs mercator misuse")

# ---- 3. headline numbers -----------------------------------------------------------
_row = area["summary"].iloc[0]
kpis([
    kpi("Median |Mercator bias|", _row["median_abs_mercator_bias_ha"], sub="hectares per cell", kind="warn"),
    kpi("stems/ha status", _row["stems_per_ha_status"], kind="warn"),
])
callout("note", "Why this matters",
        "Web-Mercator inflates areas away from the equator, so <code>Area_H</code> is taken from the attribute "
        "table and cross-checked, never recomputed in Mercator. Stems per hectare stay blocked until "
        "<code>plot_area_ha</code> exists.")
# ---- 4. findings: how does the attribute area compare with the recomputed areas? ----------
def _area_findings():
    n = int(_row["n_cells"]); a_sum = float(_row["Area_H_attr_sum"]); e_sum = float(_row["equal_area_recalc_sum"])
    med = float(_row["Area_H_attr_median"]); merc = float(_row["median_abs_mercator_bias_ha"])
    gap = (a_sum / e_sum - 1) * 100
    ga = area["grid_area"]["Area_H_attr"].astype(float)
    off = int(((ga - med).abs() > 0.10 * med).sum())
    out = [
        f"<code>Area_H</code> totals <b>{n_(a_sum)} ha</b> across {n_(n)} cells (median <b>{n_(med)} ha</b> per cell). "
        f"An equal-area recomputation totals {n_(e_sum)} ha, so the attribute total is <b>{gap:+.1f}%</b> against the recomputed total.",
        f"<b>{n_(off)}</b> of {n_(len(ga))} cells ({pct(off, len(ga))}) differ from the median cell by more than 10%. "
        f"The smallest cell is {n_(ga.min(), 1)} ha and the largest {n_(ga.max())} ha. Cells far from the nominal size deserve a look before they are used as weights.",
    ]
    if abs(gap) > 5 and merc < 1:
        lat = float(asir_su["Y"].mean()) if "Y" in asir_su.columns else None
        exp = f" At about {lat:.0f}°N that inflation is roughly {(1 / np.cos(np.radians(lat)) ** 2 - 1) * 100:.0f}%." if lat else ""
        out.append(
            "<b>Please verify how <code>Area_H</code> was produced.</b> Assuming <code>median_abs_mercator_bias_ha</code> is the median "
            f"|Web-Mercator area - Area_H|, the attribute matches the Web-Mercator area (difference {merc:.2f} ha) while the equal-area "
            "recomputation is lower. That pattern is consistent with Web-Mercator inflation." + exp +
            " Confirm with the data owner before <code>Area_H</code> is used as a denominator for per-hectare or design-based estimates.")
    out.append("Stems per hectare and design expansion stay <b>BLOCKED</b> until <code>plot_area_ha</code> is supplied.")
    return out


report(_area_findings)
flush_saved()
done()


<a id="s3"></a>

## 3 · Long-form SU × plot presence lattice (anti-clip)

Contrasts an inner join (which silently clips zero-tree units) with a LEFT join that keeps the full 430-SU universe, then builds the 2,150-row plot lattice used for Gate review.

**About the code below**

| | |
|:--|:--|
| **Reads** | `asir_su`, `trees`, the area audit. |
| **Does** | (A) shows what an inner join would lose, (B) builds the SU × plot lattice with `Area_H` attached by key, (C) builds the anti-clip SU frame. |
| **Writes** | StageStore `03_longform` (lattice and anti-clip SU frame). |
| **Check before moving on** | The lattice has exactly 2,150 rows and the anti-clip frame keeps 430 SUs. |

> **Why it matters.** Zero-tree plots are information. Dropping them changes every downstream count.

In [ ]:
# =============================================================================
#  SECTION 3 · LONG-FORM SU × PLOT PRESENCE LATTICE (ANTI-CLIP)
#  Reads  : asir_su, trees, the area audit from section 2
#  Does   : shows what an inner join loses, builds the 2,150-row lattice, builds the anti-clip SU frame
#  Writes : StageStore stage 03_longform
#  Result : longform (SU × plot lattice) and su_tree (430-SU frame with tree flags and Area_H)
# =============================================================================
banner(3, "Long-form SU × plot presence lattice",
       "Inner join versus LEFT join · 2,150-row plot lattice with zero cells retained")

# ---- 1. what would an inner join lose? ---------------------------------------------
sub("A", "Wrong versus right clip behaviour")
show(demonstrate_clip_loss(asir_su, trees), title="Clip-loss demonstration",
     caption="An inner join silently drops SUs with no tree records; the LEFT join keeps all of them.")

sub("B", "Long-form SU × plot lattice (zeros retained)")
# ---- 2. build the lattice with a LEFT join so empty cells survive -----------------------
longform = longform_su_plot_tree_presence(
    asir_su, trees,
    plots_per_su=CFG["project"]["plots_per_su_design"],
    sep=CFG["project"]["plot_uid_separator"],
)
# Attach SU-cell Area_H by key (not spatial)
longform = attach_grid_area_by_key(longform, area["grid_area"])
assert len(longform) == GATE["expected_plots"]

_zero = int((~longform["has_tree_record"]).sum())
kpis([
    kpi("Long-form rows (expected plots)", len(longform), target=GATE["expected_plots"]),
    kpi("Plots with a tree record", len(longform) - _zero, kind="ok"),
    kpi("Zero-tree plot cells retained", _zero, kind="warn"),
])
progress_bar((len(longform) - _zero) / len(longform), "Share of lattice cells with at least one tree record")

def _lattice_findings():
    n = len(longform); n_tree = n - _zero
    per = longform.loc[longform["has_tree_record"], "n_tree_records"]
    return [
        f"Of <b>{n_(n)}</b> design plots, <b>{n_(n_tree)}</b> ({pct(n_tree, n)}) hold at least one tree record. "
        f"<b>{n_(_zero)}</b> ({pct(_zero, n)}) are empty and stay visible as zero cells.",
        f"Where trees exist, a plot holds a median of <b>{per.median():.0f}</b> tree records (maximum {per.max():.0f}).",
        "An inner join or a pivot would have removed the empty cells. Section 4 splits them into Gate 1 (the whole SU is empty) "
        "and Gate 2 (empty plots inside SUs that do have trees).",
    ]


report(_lattice_findings)
show(longform.head(10), title="Lattice preview", caption="First 10 rows of the long-form lattice")
STAGES.save("03_longform", "su_plot_tree_lattice", longform, note="anti-pivot full lattice with Area_H_attr")

sub("C", "Anti-clip SU frame")
# ---- 3. build the SU frame with tree flags and attach Area_H by key -----------------------
su_tree = anti_clip_su_tree_frame(asir_su, trees)
su_tree = su_tree.merge(
    area["grid_area"][["SU_UID", "Area_H_attr"]].drop_duplicates("SU_UID"),
    on="SU_UID", how="left",
)
STAGES.save("03_longform", "su_tree_anti_clip", su_tree, note="430 SUs + tree flags + Area_H")
kpis([kpi("SUs retained by anti-clip frame", len(su_tree), target=GATE["total_sus"])])
show(su_tree.drop(columns="geometry", errors="ignore").head(8), title="Anti-clip SU frame preview")
flush_saved()
done()


<a id="s4"></a>

## 4 · Gate 1 and Gate 2 evaluation against baked targets

**Gate 1** lists sampling units with no tree records. **Gate 2** lists plots without trees inside SUs that do have trees. Both are reconciled with the validation packages, and field-classification templates are written for sign-off.

**About the code below**

| | |
|:--|:--|
| **Reads** | The SU universe, the trees table and the lattice. |
| **Does** | (A) computes both gates and reconciles them with the targets, (B) draws where the empty cells come from, (C) writes the annexes and the sign-off templates. |
| **Writes** | `gates/*.csv` (annexes, reconciliation, two `*_TEMPLATE.csv` files); StageStore `04_gates`. |
| **Check before moving on** | Every reconciliation row shows ✔ pass. The notebook stops here if any count differs. |

> **Why it matters.** Empty cells are not zero density. They need a field classification (no forest, inaccessible, not yet surveyed) before any density is computed.

In [ ]:
# =============================================================================
#  SECTION 4 · GATE 1 AND GATE 2 EVALUATION
#  Reads  : asir_su, trees, longform
#  Does   : computes both gates, reconciles them with the baked targets, charts the composition, writes sign-off templates
#  Writes : gates/*.csv (annexes, reconciliation, two *_TEMPLATE.csv), figures/gates/, StageStore 04_gates
#  Result : gate1, gate2, check (reconciliation table)
# =============================================================================
banner(4, "Gate 1 & Gate 2 evaluation",
       "Reconcile observed counts with the baked validation-package targets")

# ---- 1. compute Gate 1 (empty SUs) and Gate 2 (empty plots inside tree-SUs) -----------------
sub("A", "Reconcile with the baked targets")
gate1 = aseer_gate1_zero_tree_sus(asir_su, trees)
gate2 = aseer_gate2_plot_completeness(
    asir_su, trees,
    plots_per_su=CFG["project"]["plots_per_su_design"],
    sep=CFG["project"]["plot_uid_separator"],
    gate1_zero_su_ids=gate1["zero_su_ids"],
)
obs = {**gate1["summary"].iloc[0].to_dict(), **gate2["summary"].iloc[0].to_dict()}
check = compare_to_doc_targets(obs, GATE)

kpis([
    kpi("Sampling units", len(asir_su), target=GATE["total_sus"]),
    kpi("Gate 1 · SUs without trees", len(gate1["zero_su_ids"]), target=GATE["sus_no_trees"]),
    kpi("Expected plots", len(longform), target=GATE["expected_plots"]),
    kpi("Gate 2 · plots lacking trees", len(gate2["gate2_plots"]), target=GATE["gate2_plots"]),
])
show(gate1["summary"], title="Gate 1 · sampling units with versus without tree records")
show(gate2["summary"], title="Gate 2 · plot completeness inside tree-SUs")
show(check, title="Reconciliation against baked targets", caption="The <code>match</code> column must be true on every row")

# ---- 2. stop if any count differs from its target -------------------------------------
_ok = bool(check["match"].all())
if _ok:
    callout("success", "All Gate targets reconciled", "Observed counts equal the Validation Point 1 and 2 packages.")
else:
    callout("danger", "Gate mismatch", "At least one observed count differs from its baked target. See the table above.")
assert _ok

# ---- 3. chart: where do the empty cells come from? -------------------------------------
sub("B", "Where the empty cells come from")
g1s, g2s = gate1["summary"].iloc[0], gate2["summary"].iloc[0]
n_su, n_su_t, n_su_0 = int(g1s["total_sus"]), int(g1s["sus_with_trees"]), int(g1s["sus_no_trees"])
n_pl, n_pl_t = int(g2s["expected_plots"]), int(g2s["plots_with_trees"])
n_pl_g2, n_pl_g1 = int(g2s["gate2_plots"]), int(g2s["plots_in_gate1_zero_sus"])
_C_TREE, _C_G2, _C_G1 = CHART_COLORS[2], CHART_COLORS[1], CHART_COLORS[3]
_bars = [
    ("Sampling units\n(n = %s)" % n_(n_su), [(n_su_t, "With tree records", _C_TREE), (n_su_0, "Gate 1 · no tree records", _C_G1)]),
    ("Design plots\n(n = %s)" % n_(n_pl), [(n_pl_t, "With tree records", _C_TREE), (n_pl_g2, "Gate 2 · empty plots in tree-SUs", _C_G2),
                                          (n_pl_g1, "Gate 1 · no tree records", _C_G1)]),
]
fig, ax = plt.subplots(figsize=(10.5, 3.6))
_seen = {}
for _y, (_lab, _parts) in enumerate(_bars):
    _tot = sum(p[0] for p in _parts); _left = 0.0
    for _v, _name, _col in _parts:
        _w = 100.0 * _v / _tot
        _b = ax.barh(_y, _w, left=_left, color=_col, edgecolor="white", height=0.55)
        _seen.setdefault(_name, _b)
        ax.text(_left + _w / 2, _y, f"{_v:,}\n{_w:.1f}%", ha="center", va="center", color="white", fontsize=9.5, fontweight="bold")
        _left += _w
ax.set_yticks(range(len(_bars))); ax.set_yticklabels([b[0] for b in _bars]); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("Share of the group (%)"); ax.grid(axis="y", visible=False)
ax.set_title("Which sampling units and plots carry tree records?")
ax.legend(_seen.values(), _seen.keys(), loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=4, fontsize=9)
_gdir = OUT / "figures" / "gates"; _gdir.mkdir(parents=True, exist_ok=True)
_gpath = _gdir / "gate_composition.png"
fig.savefig(_gpath, dpi=170); plt.close(fig)
show_fig(_gpath, "Gate composition · sampling units and design plots",
         read="Each bar adds up to 100% of its group. Top bar: sampling units. Bottom bar: the design plots. "
              "Green cells hold tree records; the other two colours are empty cells that need a field classification.")


def _gate_findings():
    tree_su_plots = n_pl - n_pl_g1
    return [
        f"<b>Gate 1.</b> {n_(n_su_0)} of {n_(n_su)} sampling units ({pct(n_su_0, n_su)}) have no tree record at all. "
        f"They hold {n_(n_pl_g1)} of the {n_(n_pl)} design plots ({pct(n_pl_g1, n_pl)}).",
        f"<b>Gate 2.</b> Inside the {n_(n_su_t)} SUs that do have trees, {n_(n_pl_g2)} of {n_(tree_su_plots)} plots ({pct(n_pl_g2, tree_su_plots)}) are empty. "
        f"They are spread over {n_(int(g2s['gate2_sus_affected']))} SUs ({pct(int(g2s['gate2_sus_affected']), n_su_t)} of the tree-SUs), so empty plots are the rule, not the exception.",
        f"Only <b>{n_(n_pl_t)}</b> of {n_(n_pl)} design plots ({pct(n_pl_t, n_pl)}) yield tree records. An empty plot must not be read as zero density until it is classified "
        "(for example no forest, inaccessible, or not yet surveyed).",
        "Every target reconciles, so the counts are consistent with the Validation Point 1 and 2 packages. The two annexes below are the work list for field sign-off.",
    ]


report(_gate_findings)

sub("C", "Annexes and sign-off templates")
gate1["annex"].drop(columns="geometry", errors="ignore").to_csv(OUT/"gates"/"gate1_zero_tree_su_annex.csv", index=False)
gate2["annex"].to_csv(OUT/"gates"/"gate2_missing_plot_annex.csv", index=False)
check.to_csv(OUT/"gates"/"document_target_reconciliation.csv", index=False)

# ---- 4. sign-off templates: reviewers fill in survey_status, validated_by, date, comments ------
t1 = gate1["annex"][["SU_UID"]].copy()
if "gover_e" in gate1["annex"].columns: t1["governorate"] = gate1["annex"]["gover_e"].values
t1["survey_status"] = ""; t1["validated_by"] = ""; t1["date"] = ""; t1["comments"] = ""
t1.to_csv(OUT/"gates"/"gate1_field_classification_TEMPLATE.csv", index=False)
t2 = gate2["annex"].copy()
t2["survey_status"] = ""; t2["validated_by"] = ""; t2["date"] = ""; t2["comments"] = ""
t2.to_csv(OUT/"gates"/"gate2_field_classification_TEMPLATE.csv", index=False)

STAGES.save("04_gates", "gate1_annex", t1, note="190 SUs needing survey-status class")
STAGES.save("04_gates", "gate2_annex", gate2["annex"], note="544 plots needing class")
flush_saved()
callout("tip", "Next action for reviewers",
        "Complete <code>survey_status</code>, <code>validated_by</code>, <code>date</code> and <code>comments</code> in the two "
        "<code>*_field_classification_TEMPLATE.csv</code> files under <code>gates/</code>. Both stay <b>OPEN</b> in the "
        "readiness checklist (section 18) until they are signed off.")
done()


<a id="s5"></a>

## 5 · Regional roll-ups and SU-number graphics

Builds region and governorate tables from the anti-clip lattice, then draws **non-map** SU-count graphics for the six administrative regions on the Asir grid (Al Bahah, Ar Riyad / Ar Riyadh, Aseer / Asir, Jazan, Makkah Al Mukarramah, Najran). Chart colours match the cartography legend.

**About the code below**

| | |
|:--|:--|
| **Reads** | The lattice and the anti-clip SU frame. |
| **Does** | (1) builds region and governorate roll-ups, (A) prints the region story, (B) draws the compact SU board. |
| **Writes** | `audit/aseer_by_region.csv`, `audit/aseer_by_governorate.csv`, `audit/aseer_plots_by_governorate.csv`, region figures, StageStore `05_regions`. |
| **Check before moving on** | The findings name the dominant region, the regions too small for stand-alone estimates and the governorate with most Gate 1 units. |

> **Why it matters.** Sampling units are unevenly spread. Small regions cannot support regional estimates on their own, and the Gate 1 workload is concentrated in a few governorates.

In [ ]:
# =============================================================================
#  SECTION 5 · REGIONAL ROLL-UPS AND SU-NUMBER GRAPHICS
#  Reads  : longform and su_tree from section 3
#  Does   : builds region and governorate tables, prints the region story, draws the SU board
#  Writes : audit/aseer_by_*.csv, figures/region_su/, StageStore 05_regions
#  Result : rolls (dict with by_region, by_governorate, plots_by_governorate)
# =============================================================================
banner(5, "Regional roll-ups & SU-number graphics",
       "Region and governorate tables · non-map SU charts in legend colours")

# ---- 1. roll the lattice up to region and governorate --------------------------------
rolls = region_governorate_summaries(longform, su_tree)
show(rolls["by_region"], title="Sampling units by region",
     bars=["n_su", "n_with_trees", "n_gate1"])
show(rolls["by_governorate"].sort_values("n_gate1", ascending=False).head(20),
     title="Top 20 governorates by Gate 1 sampling units", bars=["n_su", "n_gate1"], max_rows=20)
rolls["by_region"].to_csv(OUT / "audit" / "aseer_by_region.csv", index=False)
rolls["by_governorate"].to_csv(OUT / "audit" / "aseer_by_governorate.csv", index=False)
rolls["plots_by_governorate"].to_csv(OUT / "audit" / "aseer_plots_by_governorate.csv", index=False)
STAGES.save("05_regions", "by_region", rolls["by_region"])
STAGES.save("05_regions", "by_governorate", rolls["by_governorate"])

# ---- 2. the narrative written by the project module ------------------------------------
sub("A", "Region SU story")
narrate_region_su_story(rolls["by_region"].rename(columns={"reg_e": "region"}))

# ---- 3. compact SU board: counts, share and composition in one row ---------------------
sub("B", "Compact SU board (counts · share · composition)")
su_fig_dir = OUT / "figures" / "region_su"
su_paths = plot_region_su_dashboard(
    rolls["by_region"].rename(columns={"reg_e": "region"}),
    su_fig_dir,
    region_col="region",
    su_col="n_su",
    title="Sampling units (SU) by administrative region",
)
say("Region SU graphics written to", "save", code=su_fig_dir)
show_fig(su_paths["su_dashboard_row"], title="SU board · counts · share · composition",
         read="Left to right: SU count per region, each region's share of the grid, and the split between SUs with trees and Gate 1 zero-tree SUs.")

# ---- 4. findings written from the tables above ---------------------------------------------
_reg = rolls["by_region"].rename(columns={"reg_e": "region"}).copy()
_gov = rolls["by_governorate"].copy()


def _region_findings():
    tot = int(_reg["n_su"].sum()); top = _reg.loc[_reg["n_su"].idxmax()]
    out = [f"<b>{top['region']}</b> holds <b>{n_(top['n_su'])}</b> of {n_(tot)} sampling units ({pct(top['n_su'], tot)}). "
           f"The other {len(_reg) - 1} regions share the remaining {n_(tot - top['n_su'])}."]
    small = _reg.loc[_reg["n_su"] < 10]
    if len(small):
        out.append("Regions with fewer than 10 sampling units (" + ", ".join(f"{r.region} {int(r.n_su)}" for r in small.itertuples()) +
                   ") are too small for stand-alone regional estimates. Report them as counts, or group them with a neighbour.")
    z = _reg.assign(share=_reg["n_gate1"] / _reg["n_su"]).sort_values(["share", "n_su"], ascending=False).iloc[0]
    out.append(f"The highest share of zero-tree (Gate 1) units is in <b>{z['region']}</b>: {n_(z['n_gate1'])} of {n_(z['n_su'])} ({pct(z['n_gate1'], z['n_su'])}).")
    g = _gov.sort_values("n_gate1", ascending=False).iloc[0]
    out.append(f"<b>{g['gover_e']}</b> ({g['reg_e']}) alone accounts for <b>{n_(g['n_gate1'])}</b> of the {n_(_reg['n_gate1'].sum())} Gate 1 units "
               f"({pct(g['n_gate1'], _reg['n_gate1'].sum())}) and {pct(g['n_gate1'], g['n_su'], 0)} of its own units are empty. Put it first in the field-verification queue.")
    return out


report(_region_findings)
STAGES.save("05_regions", "su_graphic_paths", pd.Series(su_paths, name="path").reset_index().rename(columns={"index": "figure"}))
flush_saved()
done()


<a id="s6"></a>

## 6 · Data cleaning and value-quality control

Inventories **observed** problems, builds **derived** cleaned tables, and prints a visible corrections story. Source Excel files and shapefiles are never overwritten.

**About the code below**

| | |
|:--|:--|
| **Reads** | The Asir slice of the plots and trees tables. |
| **Does** | Steps A to I in the table below. |
| **Writes** | Seven files under `validated/` and StageStore `06_cleaning`. Aliases `plot_aseer` and `trees_aseer` now point to the derived-clean frames. |
| **Check before moving on** | Read the findings: they say how many defects were found, which single defect is largest, and which ones still need a human decision. |

> **Why it matters.** Graphs and maps built on uncorrected labels split one class into several (for example *Innaccessible* and *Inaccessible*).

| Step | What it does |
|:----:|:-------------|
| A | Guiding-notes checklist |
| B | Asir universe slice |
| C | Lexical audit: spelling, illegal characters, contamination |
| D | Field-type audit: text-as-numeric, float-coded categories |
| E | Derived cleaning (`apply_derived_cleaning`) |
| F | Before / after demo on access labels |
| G | **Corrections story** and full report table |
| H | Persist findings under `validated/` and StageStore `06_cleaning` |
| I | Cleaned frames assigned to `plot_aseer` / `trees_aseer` |

In [ ]:
# =============================================================================
#  SECTION 6 · DATA CLEANING AND VALUE-QUALITY CONTROL
#  Reads  : the Asir slice of the plots and trees tables
#  Does   : inventories defects (spelling, characters, field types), builds derived-clean tables, prints the corrections story
#  Writes : validated/cleaning_*.csv and aseer_*_derived_clean.csv, StageStore 06_cleaning
#  Result : plot_aseer and trees_aseer now point to the derived-clean frames; source files are untouched
# =============================================================================
banner(6, "Data cleaning & value-quality control",
       "Spelling · illegal characters · field types · derived clean tables · corrections story")

sub("A", "Guiding notes")
guide = cleaning_guide_notes()
show(guide, title="Cleaning guide notes", max_rows=40)

sub("B", "Asir universe slice")
universe = set(asir_su["SU_UID"].astype(str))
trees_aseer_raw = trees.loc[trees["SU_UID"].isin(universe)].copy()
plot_aseer_raw = plot.loc[plot["SU_UID"].isin(universe)].copy()
su_aseer_raw = (
    tables["su"].loc[tables["su"]["SU_UID"].isin(universe)].copy()
    if "SU_UID" in tables["su"].columns
    else tables["su"].copy()
)
kpis([kpi("Sampling units", len(universe)), kpi("Plots in slice", len(plot_aseer_raw)),
      kpi("Trees in slice", len(trees_aseer_raw))])

sub("C", "Lexical, character and spelling audit")
lex = lexical_quality_audit(
    {
        "su": su_aseer_raw,
        "plot": plot_aseer_raw,
        "trees": trees_aseer_raw,
        "asir_su_attrs": asir_su.drop(columns="geometry", errors="ignore"),
    },
    path_hints=CFG.get("paths", {}),
)
say(f"Lexical findings: <b>{len(lex['findings']):,}</b>", "warning" if len(lex["findings"]) else "success")
show(lex["summary"], title="Lexical audit summary")
show(lex["findings"].sort_values(["severity", "n_rows"], ascending=[True, False]).head(40),
     title="Lexical findings", caption="Top 40, sorted by severity then affected rows", max_rows=40)
show(lex["correction_measures"], title="Correction measures", max_rows=40)

sub("D", "Field-type audit")
types = field_type_audit({"plot": plot_aseer_raw, "trees": trees_aseer_raw}, CFG)
if len(types):
    say(f"Type findings: <b>{len(types):,}</b>", "warning")
    show(types.head(30), title="Field-type findings", caption="Text stored as numeric, float-coded categories, and similar", max_rows=30)
else:
    callout("success", "No field-type mismatches found")

sub("E", "Derived cleaning (sources untouched)")
plot_aseer_clean = apply_derived_cleaning(plot_aseer_raw, kind="plot")
trees_aseer_clean = apply_derived_cleaning(trees_aseer_raw, kind="trees")
say("Derived clean tables built; source Excel files and shapefiles were not modified.", "success")

sub("F", "Before / after: access labels")
if "plot_info1_plot_access1_label" in plot_aseer_raw.columns:
    before = plot_aseer_raw["plot_info1_plot_access1_label"].astype(str).value_counts().head(10)
    after = plot_aseer_clean["plot_info1_plot_access1_label"].astype(str).value_counts().head(10)
    show_pair(before.to_frame("n"), after.to_frame("n"), "Before cleaning", "After cleaning", index=True)
else:
    say("Access-label column not present in this slice; demo skipped.", "note")

sub("G", "Corrections report")
corrections = build_cleaning_corrections_report(
    {"plot": plot_aseer_raw, "trees": trees_aseer_raw},
    {"plot": plot_aseer_clean, "trees": trees_aseer_clean},
)
kpis([kpi("Lexical findings", len(lex["findings"]), kind="warn"), kpi("Type findings", len(types), kind="warn"),
      kpi("Correction rows logged", len(corrections), kind="ok")])
# Full narrative + tables shown to the user in this cell output
narrate_cleaning_story(corrections, lex_findings=lex["findings"], type_findings=types)


def _cleaning_findings():
    _f, _s = lex["findings"], lex["summary"]
    sev = _s.groupby("severity")["n_findings"].sum().reindex(["high", "medium", "low"]).dropna()
    out = [f"The lexical audit logged <b>{n_(len(_f))}</b> findings ({', '.join(f'{n_(v)} {k}' for k, v in sev.items())}) "
           f"in {n_(_s['issue'].nunique())} defect types."]
    if len(_f):
        top = _f.sort_values("n_rows", ascending=False).iloc[0]
        out.append(f"The largest single defect is <code>{top['table']}.{top['column']}</code>: “{top['observed']}” should read “{top['proposed']}” "
                   f"({n_(top['n_rows'])} rows). One typo split one class in two, so every access chart would have shown it twice.")
        acts = _f["action"].value_counts()
        out.append("Proposed actions: " + ", ".join(f"{n_(v)} × <code>{k}</code>" for k, v in acts.items()) +
                   ". Anything other than an automatic derived-layer correction needs a human decision.")
    if len(types):
        tc = types["issue"].value_counts()
        out.append(f"<b>{n_(len(types))}</b> field-type findings ({', '.join(f'{n_(v)} {k}' for k, v in tc.items())}). "
                   "Float-coded categories are plotted through their <code>_label</code> companion, never as continuous numbers.")
    out.append(f"<b>{n_(len(corrections))}</b> correction rules were applied, in derived layers only. Source Excel files and shapefiles were not changed, "
               "so every correction can be reversed by re-running from source.")
    return out


report(_cleaning_findings)

sub("H", "Persist validated outputs")
val = OUT / "validated"
val.mkdir(parents=True, exist_ok=True)
lex["findings"].to_csv(val / "cleaning_findings.csv", index=False)
lex["correction_measures"].to_csv(val / "cleaning_correction_measures.csv", index=False)
types.to_csv(val / "cleaning_field_type_findings.csv", index=False)
guide.to_csv(val / "cleaning_guide_notes.csv", index=False)
corrections.to_csv(val / "cleaning_corrections_report.csv", index=False)
plot_aseer_clean.to_csv(val / "aseer_plots_derived_clean.csv", index=False)
trees_aseer_clean.to_csv(val / "aseer_trees_derived_clean.csv", index=False)
say("Seven validated cleaning tables written to", "save", code=val)

STAGES.save("06_cleaning", "findings", lex["findings"], note="Spelling, characters, contamination")
STAGES.save("06_cleaning", "field_types", types if len(types) else pd.DataFrame({"msg": ["none"]}), note="Type mismatches")
STAGES.save("06_cleaning", "guide", guide, note="Operator guiding notes")
STAGES.save("06_cleaning", "corrections_report", corrections if len(corrections) else pd.DataFrame({"msg": ["none"]}), note="Post-clean corrections log")
STAGES.save("06_cleaning", "plots_derived_clean", plot_aseer_clean.head(5000), note="Derived cleaned plots")
STAGES.save("06_cleaning", "trees_derived_clean", trees_aseer_clean.head(5000), note="Derived cleaned trees")
flush_saved()

sub("I", "Aliases for downstream sections")
plot_aseer = plot_aseer_clean
trees_aseer = trees_aseer_clean
say("<code>plot_aseer</code> and <code>trees_aseer</code> now point to the derived-clean frames.", "success")
done()


<a id="s7"></a>

## 7 · Ready clean deliverable for maps and graphs

Joins derived-clean plots and trees to the Asir SU master and applies reporting aliases on region names (`Aseer` → `Asir`, `Ar Riyad` → `Ar Riyadh`). The result is a set of **complete clean tables** plus refreshed SU-number graphics, which feed the EDA plots and cartography.

**About the code below**

| | |
|:--|:--|
| **Reads** | The derived-clean plots and trees, the Asir SU master. |
| **Does** | (1) attaches coordinate QA columns when needed, (2) builds the ready tables, (A) tells the deliverable story, (B) redraws the SU board with the reporting labels. |
| **Writes** | `validated/ready_*.csv` (four tables), StageStore `07_ready`. `plot_aseer` and `trees_aseer` now hold the ready tables. |
| **Check before moving on** | Row counts equal the universe (430 SUs, 2,150 plots) and every region has `_ready_for_graphs` = ✔. |

In [ ]:
# =============================================================================
#  SECTION 7 · READY CLEAN DELIVERABLE FOR MAPS AND GRAPHS
#  Reads  : derived-clean plots and trees, the Asir SU master
#  Does   : attaches coordinate QA, joins everything to the SU master, applies reporting aliases, redraws the SU board
#  Writes : validated/ready_*.csv (four tables), StageStore 07_ready
#  Result : su_ready, plots_ready, trees_ready; plot_aseer and trees_aseer are re-pointed to them
# =============================================================================
banner(7, "Ready clean deliverable for maps & graphs",
       "Join cleaned plots, trees and SU master · reporting aliases · refreshed SU graphics")

# ---- 1. attach coordinate QA columns to the cleaned plots when they are missing -----------------
if "coord_source" not in plot_aseer.columns or "coord_displacement_m" not in plot_aseer.columns:
    _qa = plot_coordinate_qa(plot, CFG)
    _qa = _qa.loc[_qa["SU_UID"].isin(set(asir_su["SU_UID"].astype(str)))].copy()
    _cols = [c for c in [
        "PLOT_UID", "coord_source", "coord_confidence", "coord_displacement_m",
        "lat_validated", "lon_validated",
    ] if c in _qa.columns]
    plot_aseer = plot_aseer.merge(_qa[_cols].drop_duplicates("PLOT_UID"), on="PLOT_UID", how="left", suffixes=("", "_qa"))
    say("Coordinate QA columns attached to the cleaned plots.", "info")

# ---- 2. build the ready tables (LEFT join to the SU master, region aliases applied) ------------
ready = build_ready_clean_deliverable(trees_aseer, plot_aseer, asir_su)
su_ready = ready["su_ready"]
plots_ready = ready["plots_ready"]
trees_ready = ready["trees_ready"]
by_region_ready = ready["by_region"]

kpis([
    kpi("SU rows", len(su_ready)), kpi("Plot rows", len(plots_ready)), kpi("Tree rows", len(trees_ready)),
    kpi("Regions", len(by_region_ready)),
])

# On-screen story + complete clean tables for the analyst
sub("A", "Deliverable story")
narrate_ready_deliverable_story(ready)
narrate_region_su_story(by_region_ready)
show(by_region_ready, title="Ready clean · SU counts by reporting region", bars=["n_su"])


def _ready_findings():
    r = by_region_ready
    tot_t, tot_p = r["n_tree_records"].sum(), r["n_plots"].sum()
    top = r.sort_values("n_tree_records", ascending=False).iloc[0]
    out = [f"The ready deliverable holds <b>{n_(len(su_ready))}</b> SUs, <b>{n_(len(plots_ready))}</b> plots and <b>{n_(len(trees_ready))}</b> tree rows "
           f"in {len(r)} reporting regions.",
           f"<b>{top['region']}</b> supplies {n_(top['n_tree_records'])} of {n_(tot_t)} tree records ({pct(top['n_tree_records'], tot_t)}) "
           f"and {n_(top['n_plots'])} of {n_(tot_p)} plots ({pct(top['n_plots'], tot_p)})."]
    none = r.loc[r["n_tree_records"] == 0, "region"].tolist()
    if none:
        out.append("No tree records at all in <b>" + ", ".join(none) + "</b>. These regions can be mapped as zero-tree or unclassified, "
                   "but no forest statistics can be derived for them.")
    thin = r.loc[(r["n_tree_records"] > 0) & (r["n_tree_records"] < 250), "region"].tolist()
    if thin:
        out.append("Fewer than 250 tree records in <b>" + ", ".join(thin) + "</b>. Regional means for these regions are unstable.")
    out.append("Region aliases (Aseer → Asir, Ar Riyad → Ar Riyadh) are applied, and every row keeps its source label in <code>region_source</code> for traceability.")
    return out


report(_ready_findings)

# Persist ready tables
val = OUT / "validated"
val.mkdir(parents=True, exist_ok=True)
by_region_ready.to_csv(val / "ready_by_region_su_counts.csv", index=False)
su_ready.to_csv(val / "ready_su_clean_for_maps_graphs.csv", index=False)
plots_ready.to_csv(val / "ready_plots_clean_for_maps_graphs.csv", index=False)
# Full trees can be large; always write, StageStore keeps a sample
trees_ready.to_csv(val / "ready_trees_clean_for_maps_graphs.csv", index=False)
say("Ready tables written to", "save", code=val)

STAGES.save("07_ready", "by_region", by_region_ready, note="Clean region SU counts")
STAGES.save("07_ready", "su_ready", su_ready.head(2000) if len(su_ready) > 2000 else su_ready, note="SU clean for maps/graphs")
STAGES.save("07_ready", "plots_ready", plots_ready.head(5000), note="Plots clean for maps/graphs")
STAGES.save("07_ready", "trees_ready", trees_ready.head(5000), note="Trees clean for maps/graphs")
flush_saved()

# SU graphics from CLEANED reporting region labels
sub("B", "Compact SU board from ready clean region labels")
su_fig_dir = OUT / "figures" / "region_su_ready"
su_paths_ready = plot_region_su_dashboard(
    by_region_ready,
    su_fig_dir,
    region_col="region",
    su_col="n_su",
    title="Ready clean: SU counts by reporting region",
)
show_fig(su_paths_ready["su_dashboard_row"], title="Ready clean · SU board (one row)",
         read="After cleaning, regional SU counts and shares must equal the analytical universe. Composition bars split SUs that carry trees from Gate 1 zero-tree SUs.")

# Prefer ready frames for downstream EDA / maps
plot_aseer = plots_ready
trees_aseer = trees_ready
callout("success", "Ready clean frames assigned",
        "<code>plot_aseer</code> and <code>trees_aseer</code> now hold the ready clean tables used by EDA, maps and models.")
done()


<a id="s7b"></a>

## 7b · Field vs office coordinate cross-check

Aligned with **Mapping Approach** (FAO/NCVC, June 2025), *Field data verification and cross checking*: paired Field GPS versus Office/DB coordinates, overlapping histograms, paired t-tests, MANOVA, and optional Moran's I. Map symbology for health follows **page 8** (proportional symbols, health colour legend).

**About the code below**

| | |
|:--|:--|
| **Reads** | The ready plots table with field and office coordinates. |
| **Does** | (1) runs `field_vs_office_coordinate_crosscheck`, (2) shows the tests and the scatter and histogram board, (3) draws how far apart the two sources are, (4) writes findings. |
| **Writes** | `figures/coord_crosscheck/` (board, displacement chart, paired CSV), StageStore `07b_coord_crosscheck`. |
| **Check before moving on** | The findings state the median and the tail of the displacement. A tail above 1 km means some plots have gross coordinate errors that need review. |

> **Why it matters.** A good median displacement can hide a few plots that are kilometres off. Those plots would fall in the wrong grid cell and the wrong region.

In [ ]:
# =============================================================================
#  SECTION 7b · FIELD VS OFFICE COORDINATE CROSS-CHECK
#  Reads  : the ready plots table (field and office/DB coordinates)
#  Does   : paired tests, scatter and histograms, displacement distribution, written findings
#  Writes : figures/coord_crosscheck/ (board, displacement chart, paired CSV), StageStore 07b_coord_crosscheck
#  Result : coord_qa (dict with stats, pairs, figure, narrative)
# =============================================================================
banner("7b", "Field vs office coordinate cross-check",
       "Mapping Approach PDF · paired Field GPS vs Office/DB")

# ---- 1. state the method before showing results ------------------------------------------
callout("info", "Method (Mapping Approach p.5)",
        "Compare <b>Field</b> (<code>plot_info1_plot_coor1_*</code>, blue) with <b>Office/DB</b> "
        "(<code>plot_info1_coor_db_*</code>, red). Report paired t-tests on X/Y, MANOVA, "
        "optional Moran’s I on displacement, and median/p95 displacement (m).")

# ---- 2. use the ready plot frame with QA columns; otherwise rebuild it from raw plots + CFG ----
_plots_cc = globals().get("plot_aseer", None)
if _plots_cc is None or "lon_field_orig" not in getattr(_plots_cc, "columns", []):
    _plots_cc = plot_coordinate_qa(globals().get("plot_aseer", plot), CFG)

coord_dir = OUT / "figures" / "coord_crosscheck"
coord_qa = field_vs_office_coordinate_crosscheck(_plots_cc, coord_dir)
if not coord_qa.get("ok"):
    callout("warning", "Coordinate cross-check incomplete", coord_qa.get("error", "unknown"))
else:
    callout("info", "Automatic test summary", coord_qa["narrative"] +
            "<br><i>Read this together with the displacement chart and the findings below: a test can look reassuring while a few plots are far off.</i>")
    show(coord_qa["stats"], title="Field vs office statistical tests", index=False)
    show_fig(coord_qa["figure"], "Field vs office · scatter & histograms (Mapping Approach p.5)",
         read="Tight scatter around the 1:1 line and overlapping histograms mean the two sources agree. Points far from the line are candidates for manual review.")
    say("Paired plot table", "save", code=coord_qa["pairs_csv"])

    # ---- 3. how far apart are the two GPS sources? (distribution of displacement) -----------------
    _pairs = coord_qa.get("pairs")
    _d = pd.Series(dtype=float)
    if _pairs is not None and "displacement_m" in getattr(_pairs, "columns", []):
        _d = pd.to_numeric(_pairs["displacement_m"], errors="coerce").dropna()
        _d = _d[_d >= 0]
    _pos = _d[_d > 0]
    if len(_pos) >= 5:
        fig, ax = plt.subplots(figsize=(9, 4.2))
        _bins = np.logspace(np.log10(max(_pos.min(), 0.1)), np.log10(max(_pos.max(), 10) * 1.05), 40)
        ax.hist(_pos, bins=_bins, color=CHART_COLORS[0], edgecolor="white")
        ax.set_xscale("log")
        for _v, _lab, _col in [(30, "30 m", CHART_COLORS[2]), (100, "100 m", CHART_COLORS[1]), (1000, "1 km", CHART_COLORS[3])]:
            ax.axvline(_v, color=_col, ls="--", lw=1.8, label=f"{pct((_d <= _v).sum(), len(_d), 0)} of pairs within {_lab}")
        ax.set(xlabel="Field ↔ office displacement (m, log scale)", ylabel="Number of plots",
               title="How far apart are the field and office coordinates?")
        ax.legend(loc="upper right", fontsize=9)
        _dp = coord_dir / "displacement_distribution.png"
        fig.savefig(_dp, dpi=170); plt.close(fig)
        show_fig(_dp, "Field ↔ office displacement",
                 read="Each bar counts plots by the distance between the field and the office position (log scale). "
                      "The dashed lines mark 30 m, 100 m and 1 km. A long tail to the right means a few plots have gross coordinate errors.")

    # ---- 4. findings written from the statistics and the distribution -------------------------------
    def _coord_findings():
        _st = coord_qa["stats"].set_index("test")["p_value"]

        def _p(prefix):
            m = _st[_st.index.astype(str).str.startswith(prefix)]
            return float(m.iloc[0]) if len(m) and pd.notna(m.iloc[0]) else np.nan
        px, py, pm = _p("Paired t-test X"), _p("Paired t-test Y"), _p("MANOVA")
        out = []
        if len(_d):
            gross = int((_d > 1000).sum())
            out.append(f"<b>{n_(len(_d))}</b> plots have both a field and an office position. The median displacement is <b>{_d.median():.1f} m</b>, "
                       f"and {pct((_d <= 30).sum(), len(_d), 0)} of the pairs lie within 30 m, {pct((_d <= 100).sum(), len(_d), 0)} within 100 m.")
            out.append(f"The 95th percentile is <b>{n_(_d.quantile(.95))} m</b> and <b>{n_(gross)}</b> plots ({pct(gross, len(_d))}) differ by more than 1 km. "
                       "Typical causes are swapped axes, a wrong UTM zone or a typing error. Neither source is automatically the correct one, "
                       "so check these plots against the field forms first.")
        tests = [v for v in (px, py, pm) if pd.notna(v)]
        if tests:
            sig = [n for n, v in (("X", px), ("Y", py), ("MANOVA", pm)) if pd.notna(v) and v < 0.05]
            out.append(f"Paired tests: X p = {px:.4f}, Y p = {py:.4f}, MANOVA p = {pm:.4f}. "
                       + (f"The test(s) on <b>{', '.join(sig)}</b> are significant at 5%, so the sources are not statistically interchangeable. "
                          if sig else "No test is significant at 5%. ")
                       + "With a heavy-tailed displacement distribution these tests are sensitive to a handful of extreme pairs, "
                         "so the percentile bands above are the more informative summary.")
        return out
    report(_coord_findings)
    STAGES.save("07b_coord_crosscheck", "stats", coord_qa["stats"], note="Field vs office tests")
    STAGES.save("07b_coord_crosscheck", "pairs_head", coord_qa["pairs"].head(500),
                note="Paired Field/Office coordinates + displacement_m")
    flush_saved()

callout("tip", "Atlas symbology (Mapping Approach p.8)",
        "Re-run the Beautiful HTML atlas cell to rebuild maps with page-8 health colours "
        "and the new <b>Coordinate QA</b> branch (field↔office pairs, displacement, proportional health).")
done()


<a id="s8"></a>

## 8 · Analytical-population gate

This gate prevents the most consequential NFI statistical error: treating **missing**, **not applicable**, and **true zero** as the same thing. It does not alter source measurements. It creates explicit eligibility flags for downstream analysis, and flags possible duplicate records without removing them.

**About the code below**

| | |
|:--|:--|
| **Reads** | `trees_aseer` (ready clean trees). |
| **Does** | (1) finds the height, DBH, species and plot columns, (2) classifies each record, (3) flags possible duplicates, (4) asserts that no row was dropped. |
| **Writes** | `audit/tree_analytical_population.csv`, StageStore `08_population`. |
| **Check before moving on** | The classification table adds up to all tree rows, and the duplicate finding is read by someone who can check the field forms. |

> **Why it matters.** Silent conversion of missing to zero (or zero to missing) biases every mean, density and volume that follows.

In [ ]:
# =============================================================================
#  SECTION 8 · ANALYTICAL POPULATION AND ZERO/NULL SEMANTICS
#  Reads  : trees_aseer (ready clean trees)
#  Does   : finds the key columns, classifies missing vs zero vs measured, flags possible duplicates, asserts no row is dropped
#  Writes : audit/tree_analytical_population.csv, StageStore 08_population
#  Result : tree_population (all rows kept, plus eligibility and flag columns), HEIGHT_COL, DBH_COL, SPECIES_COL, PLOT_KEY
# =============================================================================
banner(8, "Analytical population & zero/null semantics",
       "Classify measured trees · preserve missingness · define eligibility without imputation")

# ---- 1. helper: find a column by several possible names -----------------------------------
def _first_col(df, candidates):
    lut = {str(c).strip().lower(): c for c in df.columns}
    for x in candidates:
        if x.lower() in lut:
            return lut[x.lower()]
    # relaxed contains fallback
    for c in df.columns:
        n = str(c).strip().lower().replace("_"," ")
        if any(x.lower() in n for x in candidates):
            return c
    return None

HEIGHT_COL = _first_col(trees_aseer, ["hight", "height", "tree total height", "total height"])
DBH_COL = _first_col(trees_aseer, ["dbh(cm)", "dbh", "diameter at breast height"])
SPECIES_COL = _first_col(trees_aseer, ["species scientific name", "species"])
PLOT_KEY = "PLOT_UID" if "PLOT_UID" in trees_aseer.columns else _first_col(trees_aseer, ["plot id","plot_id"])

# ---- 2. classify every tree record (nothing is dropped, nothing is imputed) -------------------
tree_population = trees_aseer.copy()
tree_population["_height_obs"] = pd.to_numeric(tree_population[HEIGHT_COL], errors="coerce") if HEIGHT_COL else np.nan
tree_population["_dbh_obs"] = pd.to_numeric(tree_population[DBH_COL], errors="coerce") if DBH_COL else np.nan
tree_population["height_observed"] = tree_population["_height_obs"].notna()
tree_population["dbh_observed"] = tree_population["_dbh_obs"].notna()
tree_population["eligible_height_analysis"] = tree_population["height_observed"] & (tree_population["_height_obs"] > 0)
tree_population["eligible_dbh_analysis"] = tree_population["dbh_observed"] & (tree_population["_dbh_obs"] > 0)
tree_population["eligible_structure_analysis"] = tree_population["eligible_height_analysis"] & tree_population["eligible_dbh_analysis"]
tree_population["zero_semantics"] = np.select(
    [
        tree_population["_height_obs"].isna() & tree_population["_dbh_obs"].isna(),
        (tree_population["_height_obs"] == 0) | (tree_population["_dbh_obs"] == 0),
        tree_population["eligible_structure_analysis"],
    ],
    ["MISSING_MEASUREMENTS", "RECORDED_ZERO_REVIEW", "MEASURED_POSITIVE"],
    default="PARTIAL_MEASUREMENT"
)

# ---- 3. flag possible duplicate records (flag only; rows are never removed) --------------------
_dup_cols = [c for c in [PLOT_KEY, SPECIES_COL, DBH_COL, HEIGHT_COL, "crown_diameterl1", "crown_diameterl2"]
             if c and c in tree_population.columns]
if len(_dup_cols) >= 3:
    tree_population["possible_duplicate_record"] = tree_population.duplicated(subset=_dup_cols, keep=False)
    _n_dup_extra = int(tree_population.duplicated(subset=_dup_cols, keep="first").sum())
else:
    tree_population["possible_duplicate_record"] = False
    _n_dup_extra = 0

pop_summary = (tree_population["zero_semantics"].value_counts(dropna=False)
               .rename_axis("population_status").reset_index(name="records"))
show(pop_summary, title="Analytical population classification")
kpis([
    kpi("Tree rows retained", len(tree_population)),
    kpi("Height observations", int(tree_population["height_observed"].sum())),
    kpi("DBH observations", int(tree_population["dbh_observed"].sum())),
    kpi("Structure-eligible", int(tree_population["eligible_structure_analysis"].sum())),
])
assert len(tree_population) == len(trees_aseer), "Population classification must never drop rows."
tree_population.to_csv(OUT/"audit"/"tree_analytical_population.csv", index=False)
STAGES.save("08_population", "tree_population", tree_population.head(5000),
            note="All tree rows retained; eligibility and zero/null semantics only")
def _population_findings():
    n = len(tree_population); vc = tree_population["zero_semantics"].value_counts()
    pos = int(vc.get("MEASURED_POSITIVE", 0))
    out = [f"<b>{n_(pos)}</b> of {n_(n)} tree records ({pct(pos, n)}) carry a positive height and a positive DBH. "
           f"Missing measurements: {n_(int(vc.get('MISSING_MEASUREMENTS', 0)))}; recorded zeros to review: {n_(int(vc.get('RECORDED_ZERO_REVIEW', 0)))}; "
           f"partial records: {n_(int(vc.get('PARTIAL_MEASUREMENT', 0)))}."]
    if _n_dup_extra:
        n_flag = int(tree_population["possible_duplicate_record"].sum())
        out.append(f"<b>{n_(n_flag)}</b> records ({pct(n_flag, n)}) share plot, species, DBH, height and crown values with another record in the same plot. "
                   f"If they are double entries, tree counts, density and volume are inflated by up to <b>{n_(_n_dup_extra)}</b> records ({pct(_n_dup_extra, n)}). "
                   "They may also be genuine multi-stem trees. The records are flagged in <code>possible_duplicate_record</code> and kept; confirm against the field forms.")
    else:
        out.append("No record shares plot, species, DBH, height and crown values with another record.")
    return out


report(_population_findings)
callout("warning", "No automatic zero conversion",
        "Zeros are not converted to missing and missing values are not converted to zero. "
        "Records labelled <code>RECORDED_ZERO_REVIEW</code> require domain review.")
done()


<a id="s9"></a>

## 9 · Statistical validation gate

Profiles continuous variables without deleting outliers. It provides robust statistics and flags distributions that need careful treatment before forest calculations or GeoAI.

**About the code below**

| | |
|:--|:--|
| **Reads** | `tree_population` (from section 8). |
| **Does** | (1) computes robust statistics per variable, (2) grades a QA flag, (3) writes findings. |
| **Writes** | `audit/statistical_numeric_profile.csv`, StageStore `09_statistical_qa`. |
| **Check before moving on** | Every variable has a QA flag. Read the graded flags in the table below. |

> **Why it matters.** Skewed variables such as DBH and height make means misleading. Medians and percentiles are safer for reporting.

| QA flag | Meaning |
|:--------|:--------|
| `LOW_QA_CONCERN` | Under 5% missing and no IQR outliers. |
| `NOTE_TAIL_OUTLIERS` | Under 5% missing, some IQR outliers, but under 10% of values. Normal for right-skewed field data. |
| `REVIEW_HEAVY_TAIL` | 10% or more of the values are IQR outliers. Check the distribution before modelling. |
| `REVIEW_MISSING` | 5% or more of the values are missing. Decide how to treat the gaps before use. |

In [ ]:
# =============================================================================
#  SECTION 9 · STATISTICAL QA BEFORE FOREST CALCULATIONS
#  Reads  : tree_population (section 8)
#  Does   : computes robust statistics per continuous variable, grades a QA flag, writes findings
#  Writes : audit/statistical_numeric_profile.csv, StageStore 09_statistical_qa
#  Result : stat_profile (one row per variable)
# =============================================================================
banner(9, "Statistical QA before forest calculations",
       "Completeness · robust distribution metrics · outlier flags · no deletion")

from scipy import stats

# ---- 1. robust profile: median, MAD, percentiles, skewness and IQR outlier counts ----------------
def robust_numeric_profile(df, columns):
    rows = []
    for col in [c for c in columns if c and c in df.columns]:
        x = pd.to_numeric(df[col], errors="coerce")
        z = x.dropna()
        if len(z) == 0:
            continue
        q1, q3 = z.quantile([.25,.75]); iqr = q3-q1
        med = z.median()
        mad = np.median(np.abs(z-med))
        rows.append({
            "field": col, "n": len(z), "missing_n": int(x.isna().sum()),
            "missing_pct": float(x.isna().mean()*100), "zero_n": int((z==0).sum()),
            "mean": z.mean(), "median": med, "sd": z.std(ddof=1), "cv_pct": (z.std(ddof=1)/z.mean()*100) if z.mean()!=0 else np.nan,
            "mad": mad, "q1": q1, "q3": q3, "iqr": iqr,
            "p01": z.quantile(.01), "p05": z.quantile(.05), "p95": z.quantile(.95), "p99": z.quantile(.99),
            "min": z.min(), "max": z.max(),
            "skewness": stats.skew(z, bias=False) if len(z)>2 else np.nan,
            "kurtosis": stats.kurtosis(z, bias=False) if len(z)>3 else np.nan,
            "iqr_outlier_n": int(((z < q1-1.5*iqr) | (z > q3+1.5*iqr)).sum()),
        })
    return pd.DataFrame(rows)

# ---- 2. choose the variables to profile (height, DBH, plus any volume/biomass/carbon columns) ----
candidate_numeric = [HEIGHT_COL, DBH_COL]
for hint in ["basal", "vol/tr", "vol/ha", "agb", "bgb", "total_biomass", "carbon"]:
    c = _first_col(tree_population, [hint])
    if c and c not in candidate_numeric: candidate_numeric.append(c)

stat_profile = robust_numeric_profile(tree_population, candidate_numeric)
# ---- 3. graded QA flag (see the table in the guide above) ------------------------------------
if len(stat_profile):
    stat_profile["iqr_outlier_pct"] = stat_profile["iqr_outlier_n"] / stat_profile["n"] * 100
    stat_profile["qa_flag"] = np.select(
        [stat_profile["missing_pct"] >= 5, stat_profile["iqr_outlier_pct"] >= 10, stat_profile["iqr_outlier_n"] > 0],
        ["REVIEW_MISSING", "REVIEW_HEAVY_TAIL", "NOTE_TAIL_OUTLIERS"], default="LOW_QA_CONCERN")
else:
    stat_profile["qa_flag"] = "REVIEW"
show(stat_profile, title="NFI continuous-variable statistical QA", max_rows=50)
stat_profile.to_csv(OUT/"audit"/"statistical_numeric_profile.csv", index=False)
STAGES.save("09_statistical_qa", "numeric_profile", stat_profile, note="Outliers flagged, never auto-deleted")
def _stat_findings():
    out = []
    limit = float(CFG.get("settings", {}).get("height_unusual_max_m", 80))
    for r in stat_profile.itertuples():
        out.append(f"<b>{r.field}</b>: n = {n_(r.n)}, median {r.median:.2f} (IQR {r.q1:.2f} to {r.q3:.2f}), maximum {r.max:.2f}, skewness {r.skewness:.2f}; "
                   f"{n_(r.iqr_outlier_n)} IQR outliers ({r.iqr_outlier_pct:.1f}%) → <code>{r.qa_flag}</code>.")
    skewed = [r.field for r in stat_profile.itertuples() if r.skewness > 1]
    if skewed:
        out.append("Right-skewed (skewness above 1): <b>" + ", ".join(skewed) + "</b>. The mean overstates the typical tree, so report medians and percentiles, "
                   "and transform before parametric tests.")
    if HEIGHT_COL in set(stat_profile["field"]):
        hmax = float(stat_profile.loc[stat_profile["field"] == HEIGHT_COL, "max"].iloc[0])
        out.append(f"The tallest recorded height is {hmax:.1f} m, {'within' if hmax <= limit else '<b>above</b>'} the plausibility limit of {limit:.0f} m set in the configuration.")
    return out


report(_stat_findings)
callout("tip", "Interpretation rule",
        "A statistical outlier is a review flag, not proof of an erroneous tree. "
        "Biologically valid large trees must remain unless independent evidence supports correction.")
done()


<a id="s10"></a>

## 10 · Exploratory data analysis and statistical figure pack

Works on the **ready clean** frames from section 7 (`trees_aseer` / `plot_aseer`). Figures use corrected, engineered, joined data, with dense grids, legend-matched region colours where relevant, and smooth density curves.

**Covered:** structure and missingness · univariate and bivariate statistics · outliers (flagged, never deleted) · accessibility, ownership and coordinate trust · species and language codes · land cover, condition, agents and decomposition · years since cut, stump versus tree, bole height · metric-by-attribute panels · limitations.

**About the code below**

| | |
|:--|:--|
| **Reads** | `trees_aseer`, `plot_aseer`, the lattice. |
| **Does** | Sub-steps A to L (see the `sub()` headings in the output). Each ends with a findings panel or a figure card with a finding. |
| **Writes** | `audit/*.csv` tables, `figures/*.png`, `figures/stat_pack/`, `validated/aseer_trees_fv_metrics_provisional.csv`, StageStore `08_eda`. |
| **Check before moving on** | Read every findings panel. Three deserve attention: the plot-area assumption behind the per-hectare metrics, stumps inside the volume frame, and the sparse crown-diameter columns. |

> **Why it matters.** Descriptive analysis shows what the data can and cannot support before any model or map is trusted.

In [ ]:
# =============================================================================
#  SECTION 10 · EXPLORATORY DATA ANALYSIS AND STATISTICAL FIGURE PACK
#  Reads  : trees_aseer, plot_aseer (ready clean), longform
#  Does   : structure, missingness, FV metrics, univariate and bivariate statistics, outliers, categories, figure pack
#  Writes : audit/*.csv, figures/*.png, figures/stat_pack/, validated/aseer_trees_fv_metrics_provisional.csv, StageStore 08_eda
#  Result : trees_fv, trees_vol, stat_paths and the tables used by the executive summary
# =============================================================================
banner(10, "Exploratory data analysis & statistical figure pack",
       "Structure · missingness · distributions · relationships · outliers · species · condition · limitations")

# ---- 0. choose the frames: prefer the ready clean tables, otherwise build a fresh Asir slice -----
universe = set(asir_su["SU_UID"].astype(str))
_trees_pref = globals().get("trees_aseer")
_plot_pref = globals().get("plot_aseer")
_trees_ok = isinstance(_trees_pref, pd.DataFrame) and len(_trees_pref) > 0 and (
    "_cleaning_layer" in _trees_pref.columns or "SU_UID" in _trees_pref.columns
)
_plot_ok = isinstance(_plot_pref, pd.DataFrame) and len(_plot_pref) > 0 and (
    "_cleaning_layer" in _plot_pref.columns or "SU_UID" in _plot_pref.columns
)
if _trees_ok:
    trees_aseer = _trees_pref
else:
    trees_aseer = trees.loc[trees["SU_UID"].isin(universe)].copy()
if _plot_ok:
    plot_aseer = _plot_pref
    if "coord_source" not in plot_aseer.columns or "coord_displacement_m" not in plot_aseer.columns:
        plot_qa = plot_coordinate_qa(plot, CFG)
        qa_cols = [c for c in [
            "PLOT_UID", "coord_source", "coord_confidence", "coord_displacement_m",
            "lat_validated", "lon_validated",
        ] if c in plot_qa.columns]
        plot_aseer = plot_aseer.merge(
            plot_qa[qa_cols].drop_duplicates("PLOT_UID"),
            on="PLOT_UID", how="left", suffixes=("", "_qa"),
        )
else:
    plot_qa = plot_coordinate_qa(plot, CFG)
    plot_aseer = plot_qa.loc[plot_qa["SU_UID"].isin(universe)].copy()
_layer = (
    str(plot_aseer["_cleaning_layer"].iloc[0])
    if "_cleaning_layer" in plot_aseer.columns and len(plot_aseer)
    else "raw/unmarked"
)
apply_plot_style()
apply_fancy_theme()          # brand palette layered on top of the project plot style

sub("A", "Frames and structure")
kpis([
    kpi("Tree records", len(trees_aseer)), kpi("Tree columns", trees_aseer.shape[1]),
    kpi("Plot records", len(plot_aseer)), kpi("Plot columns", plot_aseer.shape[1]),
    kpi("Cleaning layer", _layer, kind="ok" if _layer != "raw/unmarked" else "warn"),
])
if _layer == "raw/unmarked":
    callout("warning", "EDA is running on unmarked frames",
            "No <code>_cleaning_layer</code> marker was found. Run sections 6 and 7 first to use the ready clean tables.")
if "tree_or_stump_label" in trees_aseer.columns:
    show(trees_aseer["tree_or_stump_label"].value_counts(dropna=False).rename("n").to_frame(),
         title="Tree versus stump records", bars=["n"], index=True)

sub("B", "Missingness")
miss_trees = missingness_report(trees_aseer, "trees_aseer")
miss_plot = missingness_report(plot_aseer, "plot_aseer")
miss_trees.to_csv(OUT/"audit"/"missingness_trees_aseer.csv", index=False)
miss_plot.to_csv(OUT/"audit"/"missingness_plot_aseer.csv", index=False)
show(miss_trees.head(12), title="Trees · most-missing columns", max_rows=12)
show(miss_plot.head(12), title="Plots · most-missing columns", max_rows=12)

# Compact side-by-side missingness board (plasma severity colours)
p_miss = plot_missingness_pair(
    miss_trees, miss_plot,
    OUT / "figures" / "missingness_trees_plots_pair.png",
    title_a="Asir trees", title_b="Asir plots",
    top_n=12,
    title="Missingness · trees vs plots (top 12 columns)",
)
show_fig(p_miss, "Missingness · trees | plots",
         read="Each bar is the share of missing values in one column (top 12 shown). Sparse columns must not be read as true zeros.")


def _miss_findings():
    out = []
    for nm, m, ncols in [("tree", miss_trees, trees_aseer.shape[1]), ("plot", miss_plot, plot_aseer.shape[1])]:
        empty = int((m["null_pct"] >= 100).sum()); sparse = int(((m["null_pct"] >= 50) & (m["null_pct"] < 100)).sum())
        out.append(f"<b>{n_(empty)}</b> of {n_(ncols)} {nm} columns are completely empty and another <b>{n_(sparse)}</b> are more than half empty. "
                   "Leave them out of models and charts; they carry no information for this slice.")
    return out


report(_miss_findings)

sub("C", "Tree-grain engineering (NFI-FV formulas)")
trees_fv = engineer_tree_volume_features_fv(trees_aseer, CFG, trees_only=False)
trees_vol = trees_fv.loc[trees_fv["eligible_volume"]].copy()
kpis([kpi("Tree records", len(trees_fv)),
      kpi("Eligible for volume (DBH + height)", len(trees_vol), kind="ok"),
      kpi("Not eligible", len(trees_fv) - len(trees_vol), kind="warn")])
progress_bar(len(trees_vol) / max(len(trees_fv), 1), "Share of tree records with both DBH and height")
STAGES.save(
    "08_eda", "trees_fv_provisional",
    trees_fv.head(5000) if len(trees_fv) > 5000 else trees_fv,
    note="Provisional tree-grain FV features (sample if large)",
)

dbh_col = CFG["columns"]["dbh"]
ht_col = CFG["columns"]["height"]
tree_num_raw = [c for c in [dbh_col, ht_col, "crown_diameterl1", "crown_diameterl2"] if c in trees_aseer.columns]
tree_num_eng = [c for c in FV_DELIVERABLE_COLS if c in trees_vol.columns]
fv_out = fv_deliverable_frame(trees_fv)
fv_path = OUT / "validated" / "aseer_trees_fv_metrics_provisional.csv"
fv_path.parent.mkdir(parents=True, exist_ok=True)
fv_out.to_csv(fv_path, index=False)
say("NFI-FV tree metrics written to", "save", code=fv_path)
say("Columns: " + ", ".join(f"<code>{c}</code>" for c in FV_DELIVERABLE_COLS), "note")
show(fv_out.loc[fv_out["eligible_volume"]].head(8), title="FV metrics preview", caption="First 8 eligible trees", max_rows=8)
show(univariate_numeric(trees_vol, tree_num_eng), title="FV metrics · univariate summary")


def _fv_findings():
    assumed = pd.to_numeric(fv_out["fv_plot_area_ha_assumed"], errors="coerce").dropna().unique()
    status = [str(x) for x in fv_out["fv_equation_status"].dropna().unique()]
    out = [f"The volume and biomass metrics are <b>provisional</b>. They use an assumed plot area of <b>{', '.join(f'{a:g}' for a in assumed)} ha</b> "
           f"and the equation status <code>{', '.join(status)}</code>.",
           "Because of that assumption, per-hectare figures (Vol/Ha and anything expanded per hectare) must not be reported or mapped as final. "
           "Section 2 keeps stems per hectare blocked until <code>plot_area_ha</code> is confirmed; this needs the same treatment."]
    if "tree_or_stump_label" in trees_vol.columns:
        n_st = int((trees_vol["tree_or_stump_label"].astype(str) == "Stump").sum())
        if n_st:
            out.append(f"<b>{n_(n_st)}</b> stump records ({pct(n_st, len(trees_vol))}) sit inside the frame used for volume, biomass and the figures titled "
                       "“trees-only”. A stump describes a cut stem, not a standing tree, so filter them out before volume totals are reported.")
    return out


report(_fv_findings, "Provisional metrics: what to keep in mind")

sub("D", "Univariate numeric analysis")
uni_trees = univariate_numeric(trees_aseer, tree_num_raw)
uni_eng = univariate_numeric(trees_vol, tree_num_eng)
uni_long = univariate_numeric(longform, [c for c in ["n_tree_records", "Area_H_attr"] if c in longform.columns])
read_uni = eda_readout_numeric(pd.concat([uni_trees, uni_eng], ignore_index=True))
show(uni_trees, title="Raw field variables")
show(uni_eng, title="Engineered FV variables")
show(read_uni, title="Plain-language readout", caption="Automatic interpretation of the numeric summaries")
show(uni_long, title="Long-form lattice variables")


def _dist_findings():
    u = uni_trees.set_index("variable"); out = []
    for v, unit in [(dbh_col, "cm"), (ht_col, "m")]:
        if v in u.index:
            r = u.loc[v]
            out.append(f"<b>{v}</b>: median {r['median']:.1f} {unit} (middle half {r['p25']:.1f} to {r['p75']:.1f}), maximum {r['max']:.1f} {unit}, skewness {r['skew']:.2f}. "
                       f"The mean ({r['mean']:.1f}) is above the median, so the typical tree is smaller than the mean suggests. Report medians and percentiles.")
    sparse = [v for v in u.index if u.loc[v, "n"] < 0.05 * len(trees_aseer) and u.loc[v, "median"] == 0]
    for v in sparse:
        out.append(f"<b>{v}</b> has only {n_(u.loc[v, 'n'])} values ({pct(u.loc[v, 'n'], len(trees_aseer))} of rows) and a median of 0. "
                   "That looks like placeholder zeros, not measurements. Exclude it from analysis unless the field manual says otherwise.")
    return out


report(_dist_findings)
uni_trees.to_csv(OUT/"audit"/"univariate_trees.csv", index=False)
uni_eng.to_csv(OUT/"audit"/"univariate_trees_fv_provisional.csv", index=False)
read_uni.to_csv(OUT/"audit"/"eda_readout_univariate.csv", index=False)

p_uni = plot_univariate_grid(trees_aseer, tree_num_raw, OUT/"figures"/"univariate_trees.png", "Asir trees: univariate (raw field)")
p_uni_e = plot_univariate_grid(trees_vol, tree_num_eng, OUT/"figures"/"univariate_trees_fv.png", "Trees-only provisional FV metrics (tree grain)")
p_box = plot_boxplots(trees_aseer, tree_num_raw, OUT/"figures"/"boxplots_trees.png", "Asir trees: boxplots (outliers flagged later)")
p_ecdf = plot_ecdf(trees_vol, ["dbh_cm", "height_m", "vol_tree_m3"], OUT/"figures"/"ecdf_trees_fv.png", "Empirical CDF: trees-only DBH / height / Vol/tree")
p_long = plot_univariate_grid(longform, [c for c in ["n_tree_records", "Area_H_attr"] if c in longform.columns],
                              OUT/"figures"/"univariate_longform.png", "Long-form lattice: univariate")
show_figs([(p_uni, "Distributions · raw field variables"), (p_uni_e, "Distributions · provisional FV metrics"),
           (p_box, "Boxplots · raw field variables"), (p_ecdf, "Empirical CDF · DBH, height and volume per tree"),
           (p_long, "Distributions · long-form lattice")])

sub("E", "Outliers (flagged, never deleted)")
outliers = outlier_iqr_flags(trees_aseer, tree_num_raw, k=1.5)
outliers.to_csv(OUT/"audit"/"outlier_iqr_flags_trees.csv", index=False)
kpis([kpi("IQR outlier flags", len(outliers), kind="warn", sub="rows kept in all tables")])
show(outliers.groupby("variable").size().rename("n_flags").reset_index(), title="Outlier flags by variable", bars=["n_flags"])


def _outlier_findings():
    c = outliers.groupby("variable").size().sort_values(ascending=False)
    out = [f"The IQR rule flags <b>{n_(len(outliers))}</b> values in {len(c)} variables. Nothing is deleted."]
    for v, k in c.items():
        if v in trees_aseer.columns:
            n_obs = int(pd.to_numeric(trees_aseer[v], errors="coerce").notna().sum())
            out.append(f"<code>{v}</code>: {n_(k)} flags, {pct(k, n_obs)} of {n_(n_obs)} values.")
    out.append("A right-skewed variable always flags its upper tail. A flag is a prompt to review the record, not proof of an error; large trees are biologically valid.")
    return out


report(_outlier_findings)

sub("F", "Categorical univariate analysis")
cat_cols = [c for c in [
    "tree_or_stump_label", "tree_species_scientific_name", "tree_species_vernacular_name",
    "tree_species_language_code", "lucc_label",
    "stem_quality_label", "crown_condition_label", "overall_tree_condition_label",
    "causative_agent_label[1]", "decomposition_status_label", "years_since_cut_label",
] if c in trees_aseer.columns]
uni_cat = univariate_categorical(trees_aseer, cat_cols, top_n=12)
show(uni_cat.head(40), title="Top categories per variable", caption="Up to 12 levels per column; first 40 rows shown", max_rows=40)
uni_cat.to_csv(OUT/"audit"/"univariate_categorical_trees.csv", index=False)
# One compact grid: stump/tree · scientific species · language (+ lucc if present).
# Vernacular omitted when scientific is present (same rank order).
_cat_show = [c for c in [
    "tree_or_stump_label", "tree_species_scientific_name",
    "tree_species_language_code", "lucc_label",
] if c in trees_aseer.columns]
p_cat = plot_categorical_grid(
    trees_aseer, _cat_show,
    OUT / "figures" / "categorical_grid.png",
    top_n=10, ncols=2,
    title="Categorical distributions (compact grid)",
)
show_fig(p_cat, "Categories · compact grid",
         read="Each panel ranks the categories of one attribute (top 10). Dominant classes set map symbology; rare classes stay visible for QA.")


def _species_findings():
    u = uni_cat; out = []
    sp = u[u["variable"] == "tree_species_scientific_name"].sort_values("rank")
    if len(sp):
        top = sp.iloc[0]
        out.append(f"<b>{n_(int(sp['n_levels_total'].iloc[0]))}</b> species were recorded. <i>{top['level']}</i> dominates with {top['pct']:.1f}% of records, "
                   f"and the five most frequent species make up {sp.head(5)['pct'].sum():.1f}%. Species-level statistics are only stable for the leading few.")
    st = u[(u["variable"] == "tree_or_stump_label") & (u["level"].astype(str) == "Stump")]
    if len(st):
        out.append(f"Stumps are {st['pct'].iloc[0]:.1f}% of records ({n_(int(st['n'].iloc[0]))}).")
    return out


report(_species_findings)

sub("G", "Bivariate numeric analysis (Spearman and Pearson)")
biv_s = bivariate_numeric(trees_vol, tree_num_eng, method="spearman")
biv_p = bivariate_numeric(trees_vol, tree_num_eng, method="pearson")
read_biv = eda_readout_bivariate(biv_s)
show(biv_s.head(12), title="Strongest Spearman relationships", max_rows=12)
show(read_biv.head(12), title="Plain-language readout", max_rows=12)
biv_s.to_csv(OUT/"audit"/"bivariate_spearman_trees_fv.csv", index=False)
biv_p.to_csv(OUT/"audit"/"bivariate_pearson_trees_fv.csv", index=False)
p_corr = plot_corr_pair(
    trees_vol, tree_num_eng,
    OUT / "figures" / "bivariate_corr_pair_trees_fv.png",
    title="Bivariate correlations · Spearman | Pearson",
)
p_heat = None
try:
    p_heat = plot_corr_heatmap(
        trees_vol, tree_num_eng,
        OUT / "figures" / "bivariate_corr_heatmap_trees_fv.png",
        method="spearman",
        title="Spearman correlation heatmap · provisional FV metrics",
    )
except Exception as _he:
    say(f"Correlation heatmap skipped: {_he}", "warning")
p_scat = plot_scatter_matrix(
    trees_vol, ["dbh_cm", "height_m", "basal_area_m2", "vol_tree_m3"],
    OUT / "figures" / "scatter_matrix_trees_fv.png",
    title="Scatter matrix (trees-only provisional FV grain)",
)
show_figs([
    (p_corr, "Correlations · Spearman | Pearson"),
    (p_heat, "Correlation heatmap · Spearman") if p_heat else (None, None),
    (p_scat, "Scatter matrix · DBH, height, basal area, volume"),
])


def _biv_findings():
    d = trees_vol[["dbh_cm", "height_m"]].apply(pd.to_numeric, errors="coerce").dropna()
    rho = float(d["dbh_cm"].corr(d["height_m"], method="spearman"))
    word = "strongly" if abs(rho) >= .7 else "moderately" if abs(rho) >= .4 else "weakly"
    strong = int((biv_s["corr"].abs() >= 0.99).sum())
    return [f"DBH and height are {word} related (Spearman ρ = {rho:.2f}, n = {n_(len(d))}).",
            f"<b>{n_(strong)}</b> of {n_(len(biv_s))} metric pairs have |ρ| of at least 0.99. Basal area, volume, biomass and carbon are all calculated from DBH and height, "
            "so these near-perfect links are built in by the formulas and add no new information. Judge real relationships with DBH and height, not with the derived metrics."]


report(_biv_findings)

sub("H", "Categorical × numeric analysis")
if "lucc_label" in trees_vol.columns:
    biv_lucc = bivariate_cat_numeric(trees_vol, "lucc_label", "dbh_cm")
    show(biv_lucc, title="DBH (cm) by land-cover class")
    p_bx = plot_grouped_boxplots(trees_vol, "lucc_label", "dbh_cm", OUT/"figures"/"dbh_by_lucc.png",
                                 title="DBH (cm) by land-cover class (trees-only)")
    show_fig(p_bx, "DBH by land-cover class",
         read="One box per land-cover class: the line is the median, the box the middle half, the whiskers the usual range. Compare medians to see where larger stems concentrate.")
if "tree_species_scientific_name" in trees_vol.columns:
    p_sp = plot_grouped_boxplots(trees_vol, "tree_species_scientific_name", "dbh_cm", OUT/"figures"/"dbh_by_species.png",
                                 top_levels=10, title="DBH (cm) by top species (trees-only)")
    show_fig(p_sp, "DBH by top 10 species",
         read="One box per species (the ten most frequent). Wider boxes mean more variable stem size; points beyond the whiskers are candidates for measurement QA.")

def _group_findings():
    out = []
    for col, lab in [("lucc_label", "land-cover class"), ("tree_species_scientific_name", "species")]:
        if col in trees_vol.columns:
            g = trees_vol.groupby(col)["dbh_cm"].agg(n="count", median="median").query("n >= 30").sort_values("median")
            if len(g) >= 2:
                lo, hi = g.iloc[0], g.iloc[-1]
                out.append(f"Among {lab} groups with at least 30 records, median DBH is highest in <b>{g.index[-1]}</b> ({hi['median']:.1f} cm, n = {n_(int(hi['n']))}) "
                           f"and lowest in <b>{g.index[0]}</b> ({lo['median']:.1f} cm, n = {n_(int(lo['n']))}).")
    if out:
        out.append("Groups with fewer than 30 records are left out of this comparison because their medians are unstable.")
    return out


report(_group_findings)

sub("I", "Biological plausibility flags")
bio = biological_tree_flags(trees_aseer, CFG)
bio.to_csv(OUT/"audit"/"tree_biological_flags_aseer.csv", index=False)
say(f"Biological flag table written ({len(bio):,} rows).", "save", code=OUT/"audit"/"tree_biological_flags_aseer.csv")
show(bio.head(10), title="Biological flags preview", max_rows=10)
# Lexical cleaning lives in section 6.

sub("J", "Statistical figure pack")
# Full statistical figure pack: FV metrics + accessibility/ownership/condition/agents/species/stumps
stat_dir = OUT / "figures" / "stat_pack"
stat_paths = build_asir_statistical_figure_pack(trees_fv, plot_df=plot_aseer, out_dir=stat_dir)
say("Statistical figure pack written to", "save", code=stat_dir)
# Show key dashboards
for key in ["tree_attributes", "plot_access_ownership", "stump_vs_tree", "species_bubbles",
            "vernacular_names", "language_codes",
            "causative_agents", "causative_agents_secondary", "dbh_by_agent",
            "decomposition", "years_since_cut",
            "plot_notes_and_qualifiers", "notes_by_access", "notes_by_ownership",
            "coord_displacement_by_access", "bole_height_by_stump"]:
    p = stat_paths.get(key)
    if p:
        show_fig(p, _pretty(key))
# Show all FV distribution suites
for col, p in (stat_paths.get("fv_distributions") or {}).items():
    show_fig(p, f"Distribution suite · {_pretty(col)}")
# Show a sample of metric-by-attribute panels
for p in (stat_paths.get("metric_by_attribute") or []):
    show_fig(p)
_n_attr = len(stat_paths.get("metric_by_attribute") or [])
callout("info", "Metric-by-attribute panels",
        f"<b>{_n_attr}</b> derived attribute panels are shown above (full pack under <code>{stat_dir}</code>).")

sub("K", "Limitations (methods honesty)")
lim = eda_limitations_nfi()
show(lim, title="Known limitations of this EDA", max_rows=40)
lim.to_csv(OUT/"audit"/"eda_limitations.csv", index=False)

STAGES.save("08_eda", "missingness_trees", miss_trees)
STAGES.save("08_eda", "univariate_trees", uni_trees)
STAGES.save("08_eda", "univariate_fv", uni_eng)
STAGES.save("08_eda", "outlier_flags", outliers if len(outliers) else pd.DataFrame({"msg": ["none"]}))
STAGES.save("08_eda", "bivariate_spearman", biv_s)
STAGES.save("08_eda", "eda_readouts", read_uni)
# ---- L. figure index: where every figure lives and what it shows (figures were shown above) -----
sub("L", "Figure index")
_eda_index = [
    (OUT/"figures"/"univariate_trees.png", "Univariate · raw field variables", "Distributions of raw tree measurements; long tails flag QA candidates."),
    (OUT/"figures"/"univariate_trees_fv.png", "Univariate · provisional FV metrics", "Volume and biomass-related metrics at tree grain, before modelling."),
    (OUT/"figures"/"boxplots_trees.png", "Boxplots · raw field variables", "Boxes and whiskers; points beyond them are flagged, not deleted."),
    (OUT/"figures"/"ecdf_trees_fv.png", "ECDF · DBH, height, volume", "Cumulative distributions to compare medians and upper tails."),
    (OUT/"figures"/"categorical_grid.png", "Categorical distributions", "Top categories for stump/tree, species, language and land cover."),
    (OUT/"figures"/"bivariate_corr_pair_trees_fv.png", "Bivariate correlations", "Spearman and Pearson strengths among FV metrics."),
    (OUT/"figures"/"scatter_matrix_trees_fv.png", "Scatter matrix · FV grain", "Pairwise scatter of DBH, height, basal area and volume."),
    (OUT/"figures"/"dbh_by_lucc.png", "DBH by land-cover class", "DBH stratified by land-cover class."),
    (OUT/"figures"/"dbh_by_species.png", "DBH by top species", "DBH for the ten most frequent species."),
]
show(pd.DataFrame([{"figure": t, "what it shows": d, "file": Path(p).name, "written": Path(p).exists()} for p, t, d in _eda_index]),
     title="EDA figure index", caption=f"All files are under <code>{OUT / 'figures'}</code>", max_rows=20)

flush_saved()
done()


<a id="s11"></a>

## 11 · Integrity scorecard: clip/pivot workflow versus anti-clip long-form

Quantifies what the older clip/pivot workflow loses compared with the anti-clip long-form approach, and scores each against the baked Gate targets.

**About the code below**

| | |
|:--|:--|
| **Reads** | `asir_su`, `trees`, the lattice, the Gate targets. |
| **Does** | (1) compares the two workflows metric by metric, (2) scores them, (3) draws the comparison, (4) writes findings. |
| **Writes** | `audit/old_vs_new_*.csv`, `figures/old_vs_new_integrity.png`, StageStore `08_comparison`. |
| **Check before moving on** | The anti-clip workflow matches every target; the old workflow does not. |

> **Why it matters.** This is the evidence for using the anti-clip method in all NFI reporting.

In [ ]:
# =============================================================================
#  SECTION 11 · INTEGRITY SCORECARD (CLIP/PIVOT VS ANTI-CLIP)
#  Reads  : asir_su, trees, longform, the Gate targets
#  Does   : compares the two workflows metric by metric, scores them, draws the comparison, writes findings
#  Writes : audit/old_vs_new_*.csv, figures/old_vs_new_integrity.png, StageStore 08_comparison
#  Result : cmp (metric comparison) and score (one row per workflow)
# =============================================================================
banner(11, "Integrity scorecard",
       "Clip/pivot workflow versus anti-clip long-form")

# ---- 1. compare the two workflows and score them against the Gate targets ----------------
cmp = compare_old_vs_new_method(asir_su, trees, longform, GATE)
score = accuracy_scorecard(cmp)
show(cmp, title="Old versus new method", caption="Counts from the clip/pivot workflow compared with the anti-clip long-form workflow")
show(score, title="Accuracy scorecard", max_rows=40)
cmp.to_csv(OUT/"audit"/"old_vs_new_method_comparison.csv", index=False)
score.to_csv(OUT/"audit"/"old_vs_new_accuracy_scorecard.csv", index=False)
STAGES.save("08_comparison", "old_vs_new", cmp)
STAGES.save("08_comparison", "scorecard", score)
p_cmp = plot_old_vs_new_bars(score, OUT/"figures"/"old_vs_new_integrity.png")
show_fig(p_cmp, "Integrity: clip/pivot versus anti-clip",
         read="Each pair of bars compares what the older clip/pivot workflow returns with what the anti-clip lattice returns, against the Gate target. "
              "Bars that reach the target agree with the validation packages.")

# ---- 2. findings written from the comparison table ------------------------------------------------
def _score_findings():
    c = cmp.set_index("metric")

    def val(m, col):
        return c.loc[m, col] if m in c.index else np.nan
    out = []
    o_su, n_su_ = val("su_universe_complete", "old_value"), val("su_universe_complete", "new_value")
    o_g1, n_g1 = val("gate1_sus_recoverable_for_validation", "old_value"), val("gate1_sus_recoverable_for_validation", "new_value")
    o_pl, n_pl_ = val("expected_plot_lattice_rows", "old_value"), val("expected_plot_lattice_rows", "new_value")
    n_zero = val("zero_tree_plot_cells_visible", "new_value")
    out.append(f"The clip/pivot workflow sees only <b>{n_(o_su)}</b> of {n_(n_su_)} sampling units and <b>{n_(o_g1)}</b> of the {n_(n_g1)} Gate 1 units. "
               "Every empty unit disappears, so it can never be classified in the field.")
    out.append(f"It returns {n_(o_pl)} plot rows where the design has {n_(n_pl_)}, and hides <b>{n_(n_zero)}</b> zero-tree plot cells. "
               "Any density or mean computed from it treats absence as if it were not part of the sample.")
    same = val("plots_with_trees_recovered", "old_value")
    if pd.notna(same) and same == val("plots_with_trees_recovered", "new_value"):
        out.append(f"Both workflows recover the same <b>{n_(same)}</b> plots with trees. The anti-clip lattice adds the empty cells without changing any observed value.")
    sc = score.set_index("method")
    if len(sc) >= 2:
        best = sc["metrics_matching_gate_targets"].idxmax(); worst = sc["metrics_matching_gate_targets"].idxmin()
        out.append(f"Targets matched: <b>{best}</b> {n_(sc.loc[best, 'metrics_matching_gate_targets'])}, <b>{worst}</b> {n_(sc.loc[worst, 'metrics_matching_gate_targets'])} "
                   f"(integrity score {sc.loc[best, 'integrity_score_pct']:.0f}% versus {sc.loc[worst, 'integrity_score_pct']:.0f}%). Use the anti-clip method for all NFI reporting.")
    return out


report(_score_findings)
flush_saved()
done()


<a id="s12"></a>

## 12 · Cartography: interactive grid-review maps

Builds interactive Leaflet maps of the Asir grid (Esri and Google imagery, no OSM) and registers them in the HTML atlas portal. Static PNG exports are written for print reports but are not embedded here.

**About the code below**

| | |
|:--|:--|
| **Reads** | `asir_grid`, `asir_su`, `su_tree`. |
| **Does** | (1) builds the interactive maps, (2) attaches them to the atlas portal, (3) writes optional print PNGs, (4) explains each map and links to the portal. |
| **Writes** | `maps/interactive/*.html`, `maps/html_atlas/`, print PNGs under `maps/`. |
| **Check before moving on** | The portal link opens and shows five grid-review maps. |

> **Where to review the maps:** all interactive maps open in your browser (zoom and pan) through the atlas portal. Set `OPEN_MAPS_IN_BROWSER = False` in section 0 to get links only.

In [ ]:
# =============================================================================
#  SECTION 12 · CARTOGRAPHY (INTERACTIVE GRID-REVIEW MAPS)
#  Reads  : asir_grid, asir_su, su_tree
#  Does   : builds interactive Leaflet maps, attaches them to the atlas portal, writes optional print PNGs
#  Writes : maps/interactive/*.html, maps/html_atlas/, print PNGs under maps/
#  Result : imap_paths (map files) and atlas_leaf (portal paths)
# =============================================================================
banner(12, "Cartography with basemaps",
       "Interactive Leaflet maps · zoom/pan in the HTML atlas (no OSM)")

callout("info", "Where to review the maps",
        "<b>All interactive Leaflet maps</b> (sampling-unit labels, region, governorate, Area_H, Gate 1) "
        "are registered in the HTML atlas for zoom and pan:<br>"
        f"<code>{(OUT / 'maps' / 'html_atlas' / 'index.html').resolve().as_uri()}</code><br>"
        "Static PNG exports may still be written for print reports, but the notebook no longer embeds them.")

# ---- 1. build the interactive Leaflet maps and attach them to the atlas portal -------------------
from nfi_beautiful_maps import attach_interactive_leaflet_to_atlas

imap_dir = OUT / "maps" / "interactive"
atlas_dir = OUT / "maps" / "html_atlas"
imap_paths = interactive_asir_maps(asir_grid, asir_su, su_tree=su_tree, out_dir=imap_dir)
atlas_leaf = attach_interactive_leaflet_to_atlas(atlas_dir, imap_paths)

# ---- 2. optional static PNG exports for print (not displayed here) --------------------------------
map_paths = map_asir_grid_regions(
    asir_grid, asir_su, su_tree=su_tree, out_dir=OUT / "maps"
)

# ---- 3. explain each map, then link to the portal -------------------------------------------------
_map_explain = [
    ("Sampling-unit labels",
     "Interactive markers for each SU on Esri/Google imagery. Click a point for SU_UID, region, governorate and whether trees were recorded."),
    ("Sampling units by region",
     "Grid cells coloured by administrative region. Use this to see how the Asir analytical universe spans neighbouring regions."),
    ("Sampling units by governorate",
     "Grid cells coloured by governorate. Zoom in to compare contiguous governorate blocks along the escarpment and coast."),
    ("Cell area (Area_H)",
     "Inventory cell area joined by SU key (not Mercator-derived). Darker cells have larger Area_H; inspect outliers by zooming."),
    ("Gate 1 · tree-record presence",
     "Maroon = SU with at least one tree record; orange = Gate 1 zero-tree SU. Spatial pattern of presence versus absence across the grid."),
]
show(pd.DataFrame(_map_explain, columns=["Map", "What it shows and how to use it"]),
     title="Grid-review maps in the atlas", max_rows=10, index=False)
report(lambda: [
    f"<b>{n_(len(imap_paths))}</b> interactive grid-review maps were built and registered in the atlas portal.",
    "Start with <b>Gate 1 · tree-record presence</b>: it shows whether the empty sampling units cluster in space. Clusters point to a regional cause "
    "(for example arid country or access limits) and should be classified together in the field.",
    "Check <b>Cell area (Area_H)</b> next: cells far from the nominal size are the ones section 2 asked you to verify.",
])

show_html_map(atlas_leaf["index"], title="Open HTML atlas · Grid review maps (zoom / pan)",
              open_browser=globals().get("OPEN_MAPS_IN_BROWSER", True))

show_code_details(
    "Map generation code (collapsible) · interactive Leaflet + static PNG exports",
    "imap_paths = interactive_asir_maps(asir_grid, asir_su, su_tree=su_tree, out_dir=OUT/'maps'/'interactive')\n"
    "attach_interactive_leaflet_to_atlas(OUT/'maps'/'html_atlas', imap_paths)\n"
    "map_paths = map_asir_grid_regions(asir_grid, asir_su, su_tree=su_tree, out_dir=OUT/'maps')  # print PNGs only\n"
    "# Atlas portal: maps/html_atlas/index.html  ·  Grid review branch",
)
done()


<a id="s13"></a>

## 13 · Beautiful HTML atlas (PDF-theme parity)

Interactive HTML maps aligned with the NCVC/FAO PDF set: tree density, species diversity, tree health, DBH, volume per tree, volume per hectare, basal area, AGB, BGB, total biomass, carbon, access, ownership, LUCC, stump share, species and DBH points.

**Basemaps (no API key):** OpenStreetMap, Esri World Imagery, **Google Satellite / Hybrid** through public `mt*.google.com` tile URLs (not the paid Maps JavaScript API).

Each map includes a title, north arrow, legend, scale, mouse position, minimap, fullscreen, layer control, and flexible symbolisation (quantile choropleth, categorical polygons and points, graduated points).

**About the code below**

| | |
|:--|:--|
| **Reads** | The ready clean trees and plots, validated coordinates, the grid. |
| **Does** | (1) attaches validated coordinates to the plots, (2) builds the atlas, (3) attaches the grid-review maps, (4) lists the map links. |
| **Writes** | `maps/html_atlas/index.html` and the individual maps; StageStore `11b_html_atlas`. |
| **Check before moving on** | The number of maps built matches your expectation, and the portal opens. |

> **Why it matters.** Maps use only the ready clean tables, so what you see on a map matches what the tables and graphs say.

> **Note:** per-hectare maps (volume per hectare, biomass, carbon) rest on the provisional FV metrics and an assumed plot area (see section 10). Treat them as provisional.

In [ ]:
# =============================================================================
#  SECTION 13 · BEAUTIFUL HTML ATLAS (PDF-THEME PARITY)
#  Reads  : trees_aseer, plot_aseer (ready clean), validated plot coordinates, asir_grid
#  Does   : attaches validated coordinates, builds the NFI poster-style atlas, attaches the grid-review maps
#  Writes : maps/html_atlas/index.html and individual maps, StageStore 11b_html_atlas
#  Result : atlas, atlas_paths, su_map_metrics
# =============================================================================
banner(13, "Beautiful HTML atlas",
       "NFI poster-style interactive maps with satellite basemaps")

# ---- 1. choose the frames and attach validated plot coordinates (needed for point maps) ------------
say("Building the atlas from the ready clean frames.", "info")
_trees_map = globals().get("trees_aseer", trees)
_plots_map = globals().get("plot_aseer", plot)
# Always ensure validated coordinates for point maps (health / species)
try:
    _qa = plot_coordinate_qa(plot, CFG)
    _qa = _qa.loc[_qa["SU_UID"].isin(set(asir_su["SU_UID"].astype(str)))].copy()
    _xy = [c for c in [
        "PLOT_UID", "lat_validated", "lon_validated",
        "coord_source", "coord_confidence", "coord_displacement_m",
    ] if c in _qa.columns]
    if isinstance(_plots_map, pd.DataFrame) and "PLOT_UID" in _plots_map.columns:
        _plots_map = _plots_map.drop(
            columns=[c for c in _xy if c != "PLOT_UID" and c in _plots_map.columns],
            errors="ignore",
        ).merge(_qa[_xy].drop_duplicates("PLOT_UID"), on="PLOT_UID", how="left")
    else:
        _plots_map = _qa
    say(f"Plot coordinates attached for the atlas: {'yes' if 'lat_validated' in _plots_map.columns else 'NO'} (n = {len(_plots_map):,}).", "info")
except Exception as _e:
    say(f"Plot coordinate QA for the atlas was skipped: {_e}", "warning")

atlas_dir = OUT / "maps" / "html_atlas"
# ---- 2. build the atlas (basemaps: OSM, Esri imagery, Google Satellite/Hybrid; no API key) ----------
atlas = build_beautiful_html_atlas(
    asir_grid, asir_su, _trees_map, plot_df=_plots_map, out_dir=atlas_dir,
)
atlas_paths = atlas["paths"]
su_map_metrics = atlas["su_metrics"]

show(su_map_metrics.head(20), title="SU map metrics · first 20 rows", caption="The metrics that colour the atlas maps, one row per sampling unit", max_rows=20)
_maps_built = pd.Series({k: v for k, v in atlas_paths.items() if k != "su_metrics_csv"}, name="html_map").to_frame()
show(_maps_built, title="Atlas maps built", max_rows=40, index=True)

def _atlas_findings():
    n_html = int(_maps_built["html_map"].astype(str).str.endswith(".html").sum())
    out = [f"<b>{n_(n_html)}</b> HTML maps were written to the atlas, plus the portal <code>index.html</code>."]
    num = su_map_metrics.select_dtypes(include="number")
    for c in [c for c in num.columns if c.lower() in ("n_tree_records", "n_trees", "tree_density")][:1]:
        z = int((num[c] == 0).sum())
        out.append(f"<b>{n_(z)}</b> of {n_(len(num))} sampling units have <code>{c}</code> = 0. They appear as empty cells on the density maps; "
                   "read them as <b>unclassified</b> until the Gate 1 sign-off, not as zero forest.")
    out.append("Per-hectare, biomass and carbon maps use the provisional FV metrics (see section 10). Treat them as indicative.")
    return out


report(_atlas_findings)

STAGES.save("11b_html_atlas", "su_map_metrics", su_map_metrics.head(2000), note="SU metrics for beautiful HTML maps")
STAGES.save(
    "11b_html_atlas", "paths",
    pd.Series(atlas_paths, name="path").reset_index().rename(columns={"index": "map"}),
    note="HTML atlas paths",
)

# ---- 3. make sure the grid-review maps are inside the atlas portal -----------------------------------
# (the atlas HTML is outside the Jupyter root, so links open in the browser instead of an IFrame)
from nfi_beautiful_maps import attach_interactive_leaflet_to_atlas
_imap_dir = OUT / "maps" / "interactive"
if not list(_imap_dir.glob("interactive_*.html")):
    _imap = interactive_asir_maps(asir_grid, asir_su, su_tree=su_tree, out_dir=_imap_dir)
else:
    _imap = {p.stem.replace("interactive_", "", 1): str(p) for p in _imap_dir.glob("interactive_*.html")}
    # normalize keys
    _key_map = {
        "asir_su_labels": "su_labels",
        "gate1_su_status": "gate1",
        "area_h": "area_h",
        "by_region": "by_region",
        "by_governorate": "by_governorate",
    }
    _imap = {_key_map.get(k, k): v for k, v in _imap.items()}
attach_interactive_leaflet_to_atlas(atlas_dir, _imap)

show_html_atlas(atlas_paths, open_index=globals().get("OPEN_MAPS_IN_BROWSER", True))
say("Full atlas folder", "save", code=atlas_dir)
done()


<a id="s14"></a>

## 14 · Spatial left-join QA of plot coordinates to the analytical grid

Joins validated plot coordinates to the analytical grid with a LEFT join, so plots outside every cell are reported rather than dropped.

**About the code below**

| | |
|:--|:--|
| **Reads** | The ready plots and the Asir grid. |
| **Does** | (1) turns plots into points, (2) left-joins them to the grid, (3) counts the join statuses, (4) writes findings. |
| **Writes** | StageStore `10_spatial`. |
| **Check before moving on** | Most plots fall inside their own cell. Any other status is listed and needs a decision. |

> **Why it matters.** A plot that falls outside its own SU cell contradicts the sampling design, or its coordinate is wrong.

In [ ]:
# =============================================================================
#  SECTION 14 · SPATIAL LEFT-JOIN QA OF PLOT COORDINATES
#  Reads  : plot_aseer (ready plots with validated coordinates), asir_grid
#  Does   : turns plots into points, left-joins them to the grid, counts the join statuses
#  Writes : StageStore 10_spatial
#  Result : plot_spatial (one row per plot with spatial_join_status)
# =============================================================================
banner(14, "Spatial left-join QA of plot coordinates",
       "Plots joined to the analytical grid with a LEFT join, so nothing is silently dropped")

# ---- 1. points-in-polygon join; plots outside every cell are kept and labelled -------------------
plot_spatial = left_sjoin_plots_to_su_grid(plots_to_geodataframe(plot_aseer), asir_grid)
_sj = plot_spatial["spatial_join_status"].value_counts()
kpis([kpi(str(k), int(v), kind="ok" if str(k).lower().startswith(("match", "ok", "inside", "joined")) else "warn")
      for k, v in _sj.items()][:6])
show(_sj.to_frame("n"), title="Spatial-join status", bars=["n"], index=True)
# ---- 2. findings: how many plots sit where the design says they should? ---------------------------
def _spatial_findings():
    n = int(_sj.sum()); good = sum(int(v) for k, v in _sj.items() if str(k).lower().startswith(("match", "ok", "inside", "joined")))
    out = [f"<b>{n_(good)}</b> of {n_(n)} plots ({pct(good, n)}) join to a grid cell."]
    bad = {k: int(v) for k, v in _sj.items() if not str(k).lower().startswith(("match", "ok", "inside", "joined"))}
    for k, v in bad.items():
        out.append(f"<b>{n_(v)}</b> plots ({pct(v, n)}) have status <code>{k}</code>. Plots without a valid coordinate cannot be mapped or checked against their cell; "
                   "list them and recover the coordinate from the field forms.")
    if "SU_UID_plot" in plot_spatial.columns and "SU_UID_grid" in plot_spatial.columns:
        mism = int((plot_spatial["SU_UID_plot"].astype(str) != plot_spatial["SU_UID_grid"].astype(str)).sum())
        out.append(f"{n_(mism)} plots fall in a different cell from the one in their ID.")
    return out


report(_spatial_findings)
STAGES.save("10_spatial", "plot_left_sjoin", plot_spatial.drop(columns="geometry", errors="ignore"))
flush_saved()
done()


<a id="s15"></a>

## 15 · GEDI L2A LiDAR validation (primary remote height product)

**Product:** NASA **GEDI L2A** (`GEDI02_A`) footprint elevation and height metrics (RH95 / RH98 / RH100).

**Purpose:** cross-check NFI field-recorded tree height (plot-level upper-canopy summary) against quality-filtered GEDI L2A shots near each plot.

**This section does not impute, replace, correct, or overwrite any NFI measurement.**

Comparisons are at **plot / footprint support**. An interactive **3D scene** places field stems and GEDI L2A RH spikes on a local metres plane for visual review. The section also reports **tree-height and terrain-elevation gaps and potential errors**, and lists **imputation candidate fields** with the **row counts that would be affected**. It **does not impute** unless the optional flags `ENABLE_TREE_HEIGHT_IMPUTATION` or `ENABLE_TERRAIN_HEIGHT_IMPUTATION` are switched on.

**About the code below**

| | |
|:--|:--|
| **Reads** | The ready trees and plots, the GEDI footprint table `H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet` (or raw `.h5` granules). |
| **Does** | (1) prepares plot-level field heights, (2) loads and quality-filters GEDI, (3) pairs each plot with its nearest shot, (4) computes agreement metrics and graphs, (5) builds the 3D scene, (6) audits height and terrain gaps. |
| **Writes** | `lidar/` (paired table, metrics, figures, imputation-candidate tables), StageStore `15_lidar`. |
| **Check before moving on** | The status card at the top says whether the comparison ran. If it says *not executed*, the GEDI file is missing; run the helper cell below and re-run this one. |

> **Why it matters.** Field heights are the reference, and GEDI is evidence about them. A large, systematic gap points to a footprint-scale mismatch or to a measurement problem worth a field check.

> **Read the agreement with care.** The median field height in this inventory is only a few metres. GEDI relative-height metrics are least reliable for low vegetation and on steep terrain, where vertical errors of a few metres are common. A modest R² does not by itself mean the field data are poor.


In [ ]:
# =============================================================================
#  SECTION 15 · GEDI L2A LIDAR CROSS-CHECK (HEIGHT AND ELEVATION)
#  Reads  : trees_aseer, plot_aseer, H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet (or raw .h5 granules)
#  Does   : prepares plot-level field heights, filters GEDI shots, pairs plots with the nearest shot, measures agreement, audits gaps
#  Writes : lidar/ (paired table, metrics, figures, 3D scene, imputation-candidate tables), StageStore 15_lidar
#  Result : plot_lidar_base, gedi_pack, lidar_compare, lidar_metrics (when GEDI is present), tree_h_audit, terrain_audit
# =============================================================================
banner(15, "GEDI L2A LiDAR cross-check: height & elevation",
       "GEDI02_A footprints · RH98 vs plot upper-canopy · 3D scene · NO IMPUTATION")

from pathlib import Path
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import geopandas as gpd

from nfi_gedi_l2a import (
    load_gedi_l2a,
    plot_field_vs_gedi_l2a_3d,
    download_gedi_l2a_easy,
    ensure_gedi_l2a_table,
    GEDI_PRODUCT_LABEL,
    GEDI_L2A_SHORT_NAME,
)
# ---- USER SWITCHES · edit here ---------------------------------------------------------
AUTO_DOWNLOAD_GEDI_L2A = True   # easiest path: search + download + parse when the table is missing
GEDI_MAX_GRANULES = 8           # keep first download small

LIDAR_DIR = OUT / "lidar"
LIDAR_DIR.mkdir(parents=True, exist_ok=True)
(Path("data") / "gedi" / "raw").mkdir(parents=True, exist_ok=True)

# ---- POLICY · quality rules for GEDI shots and pairing --------------------------------------
GEDI_MAX_DISTANCE_M = 30.0
GEDI_SENSITIVITY_MIN = 0.90
GEDI_HEIGHT_METRIC = "rh98"          # L2A upper-canopy proxy within the footprint
import os as _os
GEDI_DIR = Path(_os.environ.get("NFI_GEDI_DIR", "H:/3.NFI/GEDIImages"))   # override with the NFI_GEDI_DIR variable
if not GEDI_DIR.exists() and Path("data/gedi").exists():
    GEDI_DIR = Path("data/gedi").resolve()
GEDI_LOCAL_FILE = GEDI_DIR / "gedi_l2a_footprints.parquet"
GEDI_RAW_DIR = GEDI_DIR / "raw"  # optional GEDI02_A *.h5 granules
USE_META_CHM_FALLBACK = False         # LiDAR comparisons use GEDI L2A only

sub("A", "GEDI product and the support mismatch")
callout("info", "LiDAR product",
        f"<b>{GEDI_PRODUCT_LABEL}</b> (<code>{GEDI_L2A_SHORT_NAME}</code>) is the primary remote height source.<br>"
        "RH98/RH95 describe the return distribution <i>inside a ~25 m footprint</i>, not an individual tree. "
        "Field heights are aggregated to plot-level upper-canopy summaries before pairing. "
        "<b>No NFI values are overwritten.</b>")

callout("warning", "Support mismatch is explicit",
        "GEDI L2A RH metrics are footprint-scale. NFI tree height is stem-scale. "
        "We retain distance-to-shot, quality flags and sensitivity for every pair.")

# ---- 1. NFI plot support: one upper-canopy height summary per plot ---------------------------------
_hcol = next((c for c in ["tree_height", "height_m"] if c in (trees_aseer.columns if "trees_aseer" in dir() else trees.columns)), None)
_trees = trees_aseer if "trees_aseer" in dir() else trees
_plots = plot_aseer if "plot_aseer" in dir() else plot
if "lat_validated" not in _plots.columns or "lon_validated" not in _plots.columns:
    _plots = plot_coordinate_qa(_plots, CFG)

plot_lidar_base = _plots.copy()
if _hcol and "PLOT_UID" in _trees.columns and "PLOT_UID" in plot_lidar_base.columns:
    _agg = (
        _trees.assign(_h=pd.to_numeric(_trees[_hcol], errors="coerce"))
        .groupby("PLOT_UID", as_index=False)
        .agg(
            nfi_height_mean=("_h", "mean"),
            nfi_height_p95=("_h", lambda s: np.nanpercentile(s.dropna(), 95) if s.notna().any() else np.nan),
            n_trees=("_h", "size"),
        )
    )
    plot_lidar_base = plot_lidar_base.merge(_agg, on="PLOT_UID", how="left")
plot_lidar_base["nfi_latitude"] = pd.to_numeric(plot_lidar_base.get("lat_validated"), errors="coerce")
plot_lidar_base["nfi_longitude"] = pd.to_numeric(plot_lidar_base.get("lon_validated"), errors="coerce")
if "elevation" in plot_lidar_base.columns:
    plot_lidar_base["nfi_elevation_m"] = pd.to_numeric(plot_lidar_base["elevation"], errors="coerce")
elif "elev" in plot_lidar_base.columns:
    plot_lidar_base["nfi_elevation_m"] = pd.to_numeric(plot_lidar_base["elev"], errors="coerce")
else:
    plot_lidar_base["nfi_elevation_m"] = np.nan

plot_lidar_base = plot_lidar_base.dropna(subset=["nfi_latitude", "nfi_longitude"]).copy()
say(f"NFI plots prepared for GEDI L2A pairing: {len(plot_lidar_base):,}", "info")

# ---- 2. load GEDI L2A and apply the quality filter ----------------------------------------------
sub("B", "Load GEDI, pair with plots, measure agreement")
bbox = None
if len(plot_lidar_base):
    bbox = (
        float(plot_lidar_base["nfi_longitude"].min()) - 0.05,
        float(plot_lidar_base["nfi_latitude"].min()) - 0.05,
        float(plot_lidar_base["nfi_longitude"].max()) + 0.05,
        float(plot_lidar_base["nfi_latitude"].max()) + 0.05,
    )

# Easiest path: if the footprint parquet is missing, download a few GEDI02_A granules
# for the Asir bbox (NASA Earthdata login once), then parse to parquet.
if AUTO_DOWNLOAD_GEDI_L2A and bbox is not None and not Path(GEDI_LOCAL_FILE).exists():
    callout("info", "Downloading GEDI L2A (easy path)",
            "Searching NASA CMR for <code>GEDI02_A</code>, downloading up to "
            f"<b>{GEDI_MAX_GRANULES}</b> granules, and writing "
            f"<code>{GEDI_LOCAL_FILE}</code>. First run may ask for an Earthdata login.")
    _dl = download_gedi_l2a_easy(
        bbox,
        out_table=Path(GEDI_LOCAL_FILE),
        raw_dir=Path(GEDI_RAW_DIR),
        max_granules=GEDI_MAX_GRANULES,
        temporal=("2019-04-01", "2023-12-31"),
    )
    if _dl.get("status") in {"OK", "EXISTS"}:
        callout("success", "GEDI L2A ready", _dl.get("message", ""))
    elif _dl.get("status") == "NEED_LOGIN":
        callout("warning", "NASA Earthdata login needed (free)",
                _dl.get("message", "") + "<br>Or run: <code>python download_gedi_l2a.py</code>")
    else:
        callout("warning", "GEDI L2A download did not complete",
                f"{_dl.get('status')}: {_dl.get('message', '')}")

gedi_pack = load_gedi_l2a(
    table_path=GEDI_LOCAL_FILE,
    raw_dir=GEDI_RAW_DIR if GEDI_RAW_DIR.exists() else None,
    sensitivity_min=GEDI_SENSITIVITY_MIN,
    bbox=bbox,
)
lidar_compare = pd.DataFrame()
gedi = gedi_pack["all"]
gedi_filtered = gedi_pack["filtered"]

if gedi_pack["status"] != "OK":
    callout("warning", "GEDI L2A not yet present",
            f"{gedi_pack['message']}<br>"
            f"Expected table: <code>{GEDI_LOCAL_FILE}</code><br>"
            f"Or HDF5 granules in: <code>{GEDI_RAW_DIR}</code><br>"
            "Use the optional earthaccess cell below to search/download <code>GEDI02_A</code>, "
            "export footprints, then re-run this section.")
else:
    callout("success", "GEDI L2A loaded", gedi_pack["message"] + f"<br>Source: <code>{gedi_pack['source']}</code>")
    gedi.to_parquet(LIDAR_DIR / "gedi_l2a_standardized_all.parquet", index=False)
    gedi_filtered.to_parquet(LIDAR_DIR / "gedi_l2a_quality_filtered.parquet", index=False)
    show(gedi["gedi_quality_keep"].value_counts(dropna=False).rename("shots").to_frame(),
         title="GEDI L2A quality filter")

    # ---- 3. pair each plot with its nearest quality shot (metres, UTM 37N) and measure agreement --------
    if len(gedi_filtered) and len(plot_lidar_base):
        pg = gpd.GeoDataFrame(
            plot_lidar_base.copy(),
            geometry=gpd.points_from_xy(plot_lidar_base["nfi_longitude"], plot_lidar_base["nfi_latitude"]),
            crs=4326,
        ).to_crs(32637)
        gg = gpd.GeoDataFrame(
            gedi_filtered.copy(),
            geometry=gpd.points_from_xy(gedi_filtered["longitude"], gedi_filtered["latitude"]),
            crs=4326,
        ).to_crs(32637)
        joined = gpd.sjoin_nearest(pg, gg, how="left", distance_col="gedi_distance_m")
        lidar_compare = pd.DataFrame(joined.drop(columns="geometry", errors="ignore"))
        lidar_compare["support_band_m"] = np.select(
            [lidar_compare["gedi_distance_m"].le(30), lidar_compare["gedi_distance_m"].le(60),
             lidar_compare["gedi_distance_m"].le(100)],
            ["≤30 m", "≤60 m", "≤100 m"], default=">100 m",
        )
        if GEDI_HEIGHT_METRIC in lidar_compare.columns and "nfi_height_p95" in lidar_compare.columns:
            lidar_compare["height_diff_gedi_minus_nfi_m"] = (
                lidar_compare[GEDI_HEIGHT_METRIC] - lidar_compare["nfi_height_p95"]
            )
        if "elev_ground" in lidar_compare.columns:
            lidar_compare["elevation_diff_gedi_minus_nfi_m"] = (
                lidar_compare["elev_ground"] - pd.to_numeric(lidar_compare["nfi_elevation_m"], errors="coerce")
            )
        lidar_compare.to_csv(LIDAR_DIR / "nfi_gedi_l2a_disparity_by_plot.csv", index=False)

        strong = lidar_compare.loc[lidar_compare["gedi_distance_m"] <= GEDI_MAX_DISTANCE_M].copy()
        kpis([
            kpi("GEDI L2A quality shots", len(gedi_filtered), sub="after L2A filters"),
            kpi("Plot-shot pairs ≤30 m", len(strong), sub="primary support"),
            kpi("Height metric", GEDI_HEIGHT_METRIC.upper(), sub="L2A RH"),
        ])

        def _agreement_metrics(obs, ref, label):
            d = pd.DataFrame({
                "obs": pd.to_numeric(obs, errors="coerce"),
                "ref": pd.to_numeric(ref, errors="coerce"),
            }).dropna()
            if len(d) < 2:
                return pd.DataFrame([{"comparison": label, "n": len(d)}])
            e = d["ref"] - d["obs"]
            return pd.DataFrame([{
                "comparison": label, "n": len(d),
                "bias_gedi_minus_nfi": float(e.mean()),
                "mae": float(np.abs(e).mean()),
                "rmse": float(np.sqrt(np.mean(e ** 2))),
                "r2": float(r2_score(d["obs"], d["ref"])) if len(d) > 1 else np.nan,
                "pearson_r": float(d["obs"].corr(d["ref"], method="pearson")),
                "spearman_rho": float(d["obs"].corr(d["ref"], method="spearman")),
            }])

        metrics = []
        if GEDI_HEIGHT_METRIC in strong.columns and "nfi_height_p95" in strong.columns:
            metrics.append(_agreement_metrics(
                strong["nfi_height_p95"], strong[GEDI_HEIGHT_METRIC],
                f"NFI plot height P95 vs GEDI L2A {GEDI_HEIGHT_METRIC.upper()} (≤{GEDI_MAX_DISTANCE_M:.0f} m)",
            ))
        if "elev_ground" in strong.columns and strong["nfi_elevation_m"].notna().any():
            metrics.append(_agreement_metrics(
                strong["nfi_elevation_m"], strong["elev_ground"],
                f"NFI elevation vs GEDI L2A ground (≤{GEDI_MAX_DISTANCE_M:.0f} m)",
            ))
        if metrics:
            lidar_metrics = pd.concat(metrics, ignore_index=True)
            show(lidar_metrics, title="GEDI L2A ↔ NFI agreement metrics")
            lidar_metrics.to_csv(LIDAR_DIR / "nfi_gedi_l2a_agreement_metrics.csv", index=False)

            def _lidar_findings():
                out = [f"<b>{n_(len(strong))}</b> of {n_(len(plot_lidar_base))} plots ({pct(len(strong), len(plot_lidar_base))}) have a quality GEDI shot within "
                       f"{GEDI_MAX_DISTANCE_M:.0f} m. Only these pairs are scored."]
                for r in lidar_metrics.itertuples():
                    if pd.isna(getattr(r, "rmse", np.nan)):
                        out.append(f"<b>{r.comparison}</b>: only {n_(r.n)} pairs, too few to score.")
                        continue
                    line = (f"<b>{r.comparison}</b>: n = {n_(r.n)}, bias {r.bias_gedi_minus_nfi:+.2f} m (GEDI minus NFI), MAE {r.mae:.2f} m, "
                            f"RMSE {r.rmse:.2f} m, R² {r.r2:.2f}, Spearman ρ {r.spearman_rho:.2f}.")
                    if "height" in str(r.comparison).lower() and "nfi_height_p95" in strong.columns:
                        mean_h = float(pd.to_numeric(strong["nfi_height_p95"], errors="coerce").mean())
                        line += f" Relative RMSE is {100 * r.rmse / mean_h:.0f}% of the mean field height ({mean_h:.1f} m)."
                    out.append(line)
                out.append("GEDI footprints are about 25 m wide and see the whole vertical profile; field values describe stems. "
                           "Over low, sparse vegetation and steep slopes, GEDI heights carry errors of a few metres, which is a large share of a 3 to 4 m canopy. "
                           "Read the agreement as evidence about the footprint match first, and about field quality second.")
                return out
            report(_lidar_findings, "GEDI agreement · what the numbers say")

        figdir = OUT / "figures" / "lidar"
        figdir.mkdir(parents=True, exist_ok=True)
        if GEDI_HEIGHT_METRIC in strong.columns and "nfi_height_p95" in strong.columns:
            d = strong[["nfi_height_p95", GEDI_HEIGHT_METRIC]].dropna()
            if len(d) >= 5:
                fig, ax = plt.subplots(figsize=(6.2, 5.2))
                ax.scatter(d["nfi_height_p95"], d[GEDI_HEIGHT_METRIC], alpha=0.55, c="#0e7490", edgecolors="white", s=36)
                lo = float(min(d.min())); hi = float(max(d.max()))
                ax.plot([lo, hi], [lo, hi], ls="--", color="#b91c1c", lw=1.5, label="1:1")
                ax.set(xlabel="NFI plot height P95 (m)", ylabel=f"GEDI L2A {GEDI_HEIGHT_METRIC.upper()} (m)",
                       title="Field upper-canopy vs GEDI L2A")
                ax.legend()
                fig.tight_layout()
                _gp = figdir / "nfi_vs_gedi_l2a_height.png"
                fig.savefig(_gp, dpi=180); plt.close(fig)
                show_fig(_gp, "NFI vs GEDI L2A height", read=(
                    "Plot-level field height P95 paired with the nearest quality GEDI L2A RH98 shot (≤30 m). "
                    "Points near the 1:1 line agree; systematic offset indicates footprint vs stem-scale mismatch or bias."
                ))
                dd = d.copy()
                dd["mean_pair"] = (dd["nfi_height_p95"] + dd[GEDI_HEIGHT_METRIC]) / 2
                dd["difference"] = dd[GEDI_HEIGHT_METRIC] - dd["nfi_height_p95"]
                fig, ax = plt.subplots(figsize=(6.2, 4.6))
                ax.scatter(dd["mean_pair"], dd["difference"], alpha=0.55, c="#6366f1", edgecolors="white", s=32)
                mdiff, sd = float(dd["difference"].mean()), float(dd["difference"].std(ddof=1))
                ax.axhline(mdiff, color="#0f172a")
                ax.axhline(mdiff + 1.96 * sd, color="#64748b", ls="--")
                ax.axhline(mdiff - 1.96 * sd, color="#64748b", ls="--")
                ax.set(xlabel="Pair mean height (m)", ylabel="GEDI L2A - NFI (m)", title="Bland-Altman · GEDI L2A")
                fig.tight_layout()
                _gp = figdir / "nfi_gedi_l2a_height_bland_altman.png"
                fig.savefig(_gp, dpi=180); plt.close(fig)
                show_fig(_gp, "NFI-GEDI L2A height · Bland-Altman", read=(
                    "Difference (GEDI L2A RH - field P95) against pair mean. Flat cloud around zero supports agreement."
                ).replace("-", "-"))

        STAGES.save("15_lidar", "nfi_gedi_l2a_compare", lidar_compare.head(10000),
                    note="GEDI L2A disparity evidence only; no NFI values replaced")
        callout("success", "GEDI L2A comparison generated",
                "Outputs include support distance, L2A quality metadata, paired disparities. <b>No imputation.</b>")

# ---- 4. field versus GEDI comparison board (plus optional Meta canopy-height fallback) --------------
sub("C", "Field versus GEDI graphs and the 3D scene")
from nfi_lidar_global import plot_field_lidar_comparison_board

figdir_lidar = LIDAR_DIR / "figures"
figdir_lidar.mkdir(parents=True, exist_ok=True)

chm_joined = None
if USE_META_CHM_FALLBACK and (gedi_pack["status"] != "OK" or len(lidar_compare) == 0):
    callout("info", "Meta CHM fallback enabled",
            "GEDI L2A unavailable - sampling Meta CHM only because USE_META_CHM_FALLBACK=True.")
    try:
        from nfi_lidar_global import extract_meta_chm_at_plots, PRODUCT_NAME
        _base = plot_lidar_base.dropna(subset=["nfi_latitude", "nfi_longitude", "nfi_height_mean"]).copy()
        _base = _base.rename(columns={"nfi_latitude": "lat_validated", "nfi_longitude": "lon_validated",
                                      "nfi_height_mean": "mean_height_m"})
        if len(_base) > 120:
            _base = _base.sample(120, random_state=42)
        if len(_base) >= 20:
            chm_joined = extract_meta_chm_at_plots(
                _base, cache_dir=LIDAR_DIR / "meta_chm_cache",
                lon_col="lon_validated", lat_col="lat_validated", progress_every=40,
            )
            chm_joined.to_csv(LIDAR_DIR / "field_vs_meta_chm_sample.csv", index=False)
    except Exception as _e:
        callout("warning", "Meta CHM sampling skipped", str(_e))
else:
    callout("note", "Meta CHM skipped",
            "Comparisons use <b>GEDI L2A</b> only. Set <code>USE_META_CHM_FALLBACK = True</code> to enable CHM when GEDI is absent.")

_gedi_cmp = lidar_compare if len(lidar_compare) else None
lidar_board = plot_field_lidar_comparison_board(
    trees=_trees,
    plot_df=_plots,
    gedi_compare=_gedi_cmp,
    chm_joined=chm_joined,
    out_dir=figdir_lidar,
    gedi_height_col=GEDI_HEIGHT_METRIC,
)
callout("success", "Field versus GEDI L2A graphs", lidar_board["narrative"])
_lidar_analysis = {
    "field_height_summary": "Distribution of field-measured tree heights.",
    "field_elevation_summary": "Plot elevation from field or database sources.",
    "gedi_height_board": "Field upper-canopy height versus GEDI L2A RH. Scatter, residual KDE, Bland-Altman.",
    "gedi_elevation_board": "Field elevation versus GEDI L2A ground elevation at footprint support.",
}
for _k, _p in lidar_board["paths"].items():
    if not _p or not Path(_p).exists():
        continue
    if str(_k).startswith("chm_") and not USE_META_CHM_FALLBACK:
        continue
    show_fig(_p, f"Field vs GEDI L2A · {_pretty(_k)}",
             read=_lidar_analysis.get(_k, "Field measurement versus GEDI L2A evidence. No NFI overwrite."))

# ---- 5. 3D scene: field stems and GEDI L2A relative-height spikes on a local metres plane ----------
if len(gedi_filtered):
    _html3d = LIDAR_DIR / "figures" / "field_vs_gedi_l2a_3d.html"
    _scene = plot_field_vs_gedi_l2a_3d(
        trees=_trees,
        gedi_filtered=gedi_filtered,
        plot_df=_plots,
        out_html=_html3d,
        max_distance_m=60.0,
        height_col=_hcol or "tree_height",
        gedi_rh_col=GEDI_HEIGHT_METRIC,
    )
    if _scene.get("status") == "OK":
        callout("success", "3D field vs GEDI L2A scene",
                f"Interactive Plotly scene with {_scene['n_trees']} field stems and {_scene['n_gedi']} GEDI L2A footprints.<br>"
                f"<a href='{_html3d.resolve().as_uri()}' target='_blank'>Open 3D scene</a>")
        try:
            from IPython.display import IFrame
            display(IFrame(_html3d.resolve().as_uri(), width="100%", height=520))
        except Exception:
            show_html_map(_html3d, title="3D · field trees vs GEDI L2A", open_browser=False)
    else:
        callout("warning", "3D scene not built", _scene.get("reason", "unknown"))
else:
    callout("tip", "Next step for GEDI L2A",
            "Download <code>GEDI02_A</code> granules (optional cell below), export footprints to "
            f"<code>{GEDI_LOCAL_FILE}</code>, then re-run this section for graphs and the 3D scene.")

STAGES.save("15_lidar", "comparison_figure_index",
            pd.DataFrame([{"figure": k, "path": v} for k, v in lidar_board["paths"].items()]),
            note="Field height/elevation vs GEDI L2A")

# ---------------------------------------------------------------------------
# Discrepancies + imputation CANDIDATES (tree height & terrain)
# Reports gaps / potential errors and row counts that WOULD be affected.
# Imputation is OPTIONAL and OFF by default - never overwrites NFI source fields.
# ---------------------------------------------------------------------------
from nfi_imputation_candidates import (
    audit_tree_height_discrepancies,
    audit_terrain_height_discrepancies,
    optional_impute_tree_height,
    optional_impute_terrain_height,
    write_imputation_audit,
)

ENABLE_TREE_HEIGHT_IMPUTATION = False   # set True only after explicit approval
ENABLE_TERRAIN_HEIGHT_IMPUTATION = False

sub("D", "Height & terrain discrepancies · imputation candidates (no overwrite)")

_gedi_for_audit = lidar_compare if "lidar_compare" in dir() and len(lidar_compare) else None
_trees_aud = trees_aseer if "trees_aseer" in dir() else trees
_plots_aud = plot_aseer if "plot_aseer" in dir() else plot

tree_h_audit = audit_tree_height_discrepancies(
    _trees_aud,
    height_col=_hcol or "tree_height",
    gedi_compare=_gedi_for_audit,
    gedi_rh_col=GEDI_HEIGHT_METRIC,
    gedi_support_m=GEDI_MAX_DISTANCE_M,
    height_unusual_max_m=float(CFG.get("settings", {}).get("height_unusual_max_m", 80)),
)
terrain_audit = audit_terrain_height_discrepancies(
    _plots_aud,
    gedi_compare=_gedi_for_audit,
    gedi_elev_col="elev_ground",
    gedi_support_m=GEDI_MAX_DISTANCE_M,
)

show(tree_h_audit["summary"], title="Tree height · gaps, potential errors, imputation candidates",
     caption="would_impute_if_enabled=True rows are counted only; values are NOT written unless the optional flag is on.")
show(terrain_audit["summary"], title="Terrain elevation · gaps, potential errors, imputation candidates",
     caption="Field plot tables currently lack elevation; GEDI L2A elev_ground is the optional donor.")

kpis([
    kpi("Tree-height imputation candidates", tree_h_audit.get("n_imputation_candidates", 0),
        sub="rows if imputation enabled", kind="warn"),
    kpi("Tree-height potential errors", tree_h_audit.get("n_potential_errors", 0),
        sub="review only · not auto-filled", kind="warn"),
    kpi("Terrain imputation candidates", terrain_audit.get("n_imputation_candidates", 0),
        sub="plots if imputation enabled", kind="warn"),
    kpi("Terrain potential errors", terrain_audit.get("n_potential_errors", 0),
        sub="|GEDI-field| large", kind="warn"),
])

def _imputation_findings():
    c_h, e_h = int(tree_h_audit.get("n_imputation_candidates", 0)), int(tree_h_audit.get("n_potential_errors", 0))
    c_t, e_t = int(terrain_audit.get("n_imputation_candidates", 0)), int(terrain_audit.get("n_potential_errors", 0))
    out = [f"Tree height: <b>{n_(c_h)}</b> records could be imputation candidates and <b>{n_(e_h)}</b> look like potential errors "
           f"({pct(e_h, len(_trees_aud))} of {n_(len(_trees_aud))} tree records).",
           f"Terrain elevation: <b>{n_(c_t)}</b> plots could be candidates and <b>{n_(e_t)}</b> show a large GEDI-minus-field gap "
           f"({pct(c_t, len(_plots_aud))} of {n_(len(_plots_aud))} plots)."]
    if c_t and c_t >= 0.9 * len(_plots_aud):
        out.append("Almost every plot lacks a field elevation, so terrain candidates reflect a missing column, not measurement gaps. Supply plot elevation from the field forms or a DEM before any comparison.")
    out.append("Imputation stays <b>OFF</b>. These counts show what a later, separately approved study would touch; no NFI value is written or replaced.")
    return out


report(_imputation_findings, "Discrepancies · what a later imputation study would touch")

_audit_dir = LIDAR_DIR / "imputation_candidates"
_audit_paths = write_imputation_audit(tree_h_audit, terrain_audit, _audit_dir)
say("Imputation-candidate tables written", "save", code=_audit_dir)
if tree_h_audit.get("candidates") is not None and len(tree_h_audit["candidates"]):
    show(tree_h_audit["candidates"].head(25), title="Tree-height issues · preview (first 25)", max_rows=25)
if terrain_audit.get("candidates") is not None and len(terrain_audit["candidates"]):
    show(terrain_audit["candidates"].head(25), title="Terrain elevation issues · preview (first 25)", max_rows=25)

callout("tip", "Optional imputation (currently OFF)",
        f"<code>ENABLE_TREE_HEIGHT_IMPUTATION = {ENABLE_TREE_HEIGHT_IMPUTATION}</code><br>"
        f"<code>ENABLE_TERRAIN_HEIGHT_IMPUTATION = {ENABLE_TERRAIN_HEIGHT_IMPUTATION}</code><br>"
        "When enabled, sibling columns <code>tree_height_imputed</code> / <code>elevation_imputed</code> are filled "
        "from nearby GEDI L2A only. Source NFI columns are never overwritten.")

_tree_imp = optional_impute_tree_height(
    _trees_aud, enabled=ENABLE_TREE_HEIGHT_IMPUTATION,
    height_col=_hcol or "tree_height",
    gedi_compare=_gedi_for_audit, gedi_rh_col=GEDI_HEIGHT_METRIC,
    gedi_support_m=GEDI_MAX_DISTANCE_M,
)
_terr_imp = optional_impute_terrain_height(
    _plots_aud, enabled=ENABLE_TERRAIN_HEIGHT_IMPUTATION,
    gedi_compare=_gedi_for_audit, gedi_elev_col="elev_ground",
    gedi_support_m=GEDI_MAX_DISTANCE_M,
)
callout("info" if _tree_imp["status"] == "DISABLED" else "success",
        "Tree-height imputation status", _tree_imp["message"])
callout("info" if _terr_imp["status"] == "DISABLED" else "success",
        "Terrain-height imputation status", _terr_imp["message"])

if _tree_imp["status"] != "DISABLED":
    _tp = LIDAR_DIR / "optional_tree_height_imputed.csv"
    _tree_imp["frame"].to_csv(_tp, index=False)
    say("Optional tree-height sibling columns exported", "save", code=_tp)
    STAGES.save("15_lidar", "trees_height_imputed_sibling", _tree_imp["frame"].head(5000),
                note="Sibling imputation only; source height untouched")
if _terr_imp["status"] != "DISABLED":
    _ep = LIDAR_DIR / "optional_elevation_imputed.csv"
    _terr_imp["frame"].to_csv(_ep, index=False)
    say("Optional elevation sibling columns exported", "save", code=_ep)
    STAGES.save("15_lidar", "plots_elevation_imputed_sibling", _terr_imp["frame"].head(5000),
                note="Sibling imputation only; no source overwrite")

STAGES.save("15_lidar", "tree_height_imputation_candidate_summary", tree_h_audit["summary"],
            note="Counts only; imputation disabled by default")
STAGES.save("15_lidar", "terrain_imputation_candidate_summary", terrain_audit["summary"],
            note="Counts only; imputation disabled by default")

done()


### Optional helper · GEDI L2A download

If section 15 reports that the GEDI L2A table is missing, run the next cell. It uses `earthaccess` to search **GEDI02_A**, download a few granules for the Asir bounding box, and write `H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet`. It needs a free [NASA Earthdata](https://urs.earthdata.nasa.gov/) account (you log in once). If the table already exists, the cell reports `EXISTS` and does not download again.

| | |
|:--|:--|
| **Reads** | The plot coordinates (for the bounding box); falls back to the Asir extent. |
| **Writes** | `H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet` and raw granules under `H:/3.NFI/GEDIImages/raw/`. |
| **Check** | The cell prints `OK` or `EXISTS`. Then re-run section 15. |

> **Path.** The folder defaults to `H:/3.NFI/GEDIImages`. Set the `NFI_GEDI_DIR` environment variable before section 15 to use another folder (for example on Colab), or place the files under `data/gedi` inside the package.


In [ ]:
import os as _os
from pathlib import Path as _P
if "GEDI_LOCAL_FILE" not in globals():            # section 15 has not run in this session
    GEDI_DIR = _P(_os.environ.get("NFI_GEDI_DIR", "H:/3.NFI/GEDIImages"))
    GEDI_LOCAL_FILE, GEDI_RAW_DIR = GEDI_DIR / "gedi_l2a_footprints.parquet", GEDI_DIR / "raw"
# =============================================================================
#  OPTIONAL HELPER · GEDI L2A DOWNLOAD
#  Reads  : plot coordinates (for the bounding box); falls back to the Asir extent
#  Does   : searches NASA CMR for GEDI02_A, downloads a few granules, writes the footprint table
#  Writes : H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet and raw granules under H:/3.NFI/GEDIImages/raw/
#  Result : prints OK or EXISTS; then re-run section 15
# =============================================================================
# Easiest GEDI L2A download (run this cell if Section 15 says the file is missing).
# Needs a free NASA Earthdata account: https://urs.earthdata.nasa.gov/
# First run prompts for login once; later runs reuse saved credentials.

from pathlib import Path
from nfi_gedi_l2a import download_gedi_l2a_easy

# ---- 1. bounding box: the plot extent plus a 0.05 degree margin (Asir extent as fallback) ----------
_bbox = None
if "plot_lidar_base" in dir() and len(plot_lidar_base):
    _bbox = (
        float(plot_lidar_base["nfi_longitude"].min()) - 0.05,
        float(plot_lidar_base["nfi_latitude"].min()) - 0.05,
        float(plot_lidar_base["nfi_longitude"].max()) + 0.05,
        float(plot_lidar_base["nfi_latitude"].max()) + 0.05,
    )
else:
    # Asir fallback
    _bbox = (41.5, 17.5, 44.5, 21.0)

# ---- 2. search and download (first run asks for the NASA Earthdata login) -----------------------
say("Searching and downloading GEDI L2A for the bounding box:", "info", code=str(tuple(round(x, 3) for x in _bbox)))
_gedi_dl = download_gedi_l2a_easy(
    _bbox,
    out_table=GEDI_LOCAL_FILE,
    raw_dir=GEDI_RAW_DIR,
    max_granules=8,
    temporal=("2019-04-01", "2023-12-31"),
)
say(f"Status <b>{_gedi_dl.get('status')}</b>: {_gedi_dl.get('message')}", "note")
if _gedi_dl.get("status") in {"OK", "EXISTS"}:
    callout("success", "GEDI L2A on disk",
            f"Table: <code>{_gedi_dl.get('table')}</code><br>Re-run Section 15 to build graphs and the 3D scene.")
elif _gedi_dl.get("status") == "NEED_LOGIN":
    callout("warning", "Login required",
            "Create a free Earthdata account, then re-run this cell. "
            "Or in a terminal: <code>python download_gedi_l2a.py</code>")
else:
    callout("danger", "Download failed", str(_gedi_dl.get("message")))


<a id="s16"></a>

## 16 · GeoAI modelling protocol: frozen before modelling

This section records the validation design **before** Sentinel/GEDI feature modelling. It prevents model-selection leakage and makes later imputation research auditable. No model is trained here.

**About the code below**

| | |
|:--|:--|
| **Reads** | Nothing from the data; this is a written protocol. |
| **Does** | Builds a table of 12 controls, each with the rule and the reason for it. |
| **Writes** | `audit/geoai_validation_protocol.csv`, StageStore `16_geoai_protocol`. |
| **Check before moving on** | Everyone who will train models has read the table. Later model comparisons must use identical spatial folds and untouched outer validation data. |

> **Why it matters.** Random train/test splits on spatial data overstate accuracy because neighbouring plots resemble each other.

In [ ]:
# =============================================================================
#  SECTION 16 · GEOAI VALIDATION AND FINE-TUNING PROTOCOL
#  Reads  : nothing from the data; this is a written protocol
#  Does   : records twelve validation controls, each with its rule and its reason, before any model is trained
#  Writes : audit/geoai_validation_protocol.csv, StageStore 16_geoai_protocol
#  Result : geoai_protocol (control, policy, why_it_matters)
# =============================================================================
banner(16, "GeoAI validation & fine-tuning protocol",
       "Spatial CV · nested tuning · uncertainty · applicability · no imputation")

geoai_protocol = pd.DataFrame([
    ("analysis_grain","Plot/SU for EO modelling; tree level only for biological/allometric analysis"),
    ("response_freeze","Validated NFI response variables frozen before EO feature selection"),
    ("split_strategy","Spatially blocked outer CV; no random-only train/test split"),
    ("tuning","Nested spatial CV inside outer training folds"),
    ("baseline","Statistical/null baseline before RF/XGBoost/advanced models"),
    ("regression_metrics","RMSE; MAE; R²; bias; relative RMSE; CCC; calibration slope/intercept"),
    ("residual_spatial_test","Moran's I on out-of-fold residuals"),
    ("stratified_performance","Stratum; governorate; forest type; response range; elevation where available"),
    ("uncertainty","Prediction interval / ensemble uncertainty plus empirical coverage"),
    ("applicability","Environmental novelty / area-of-applicability mask for wall-to-wall maps"),
    ("lidar_role","GEDI is validation/feature evidence; never silently substitutes field height"),
    ("imputation_status","DISABLED - disparity analysis only in this notebook"),
],columns=["control","policy"])
# ---- 1. add the reason behind each control, so readers see why it is there -----------------------
geoai_protocol["why_it_matters"] = geoai_protocol["control"].map({
    "analysis_grain": "Trees are nested in plots; pooling trees overstates the sample size for EO models.",
    "response_freeze": "Choosing responses after seeing the EO features leaks information into the model.",
    "split_strategy": "Neighbouring plots resemble each other, so random splits overstate accuracy.",
    "tuning": "Tuning on the outer test folds would inflate the reported skill.",
    "baseline": "A model is only useful if it beats a simple, transparent benchmark.",
    "regression_metrics": "One metric alone hides bias, scale error or poor calibration.",
    "residual_spatial_test": "Spatially clustered residuals show structure the model has missed.",
    "stratified_performance": "Average skill can hide poor performance in specific zones or forest types.",
    "uncertainty": "Maps without uncertainty invite over-interpretation.",
    "applicability": "Predictions outside the training environment are unreliable.",
    "lidar_role": "Remote heights are evidence with their own error, not ground truth.",
    "imputation_status": "Filling gaps needs its own approved and independently validated study.",
})
# ---- 2. show and save the frozen protocol -------------------------------------------------------
show(geoai_protocol, title="Frozen GeoAI validation protocol", max_rows=30)
geoai_protocol.to_csv(OUT/"audit"/"geoai_validation_protocol.csv",index=False)
STAGES.save("16_geoai_protocol","protocol",geoai_protocol,note="Pre-model validation design")
callout("success","Protocol frozen",
        "Future model comparison must use identical spatial folds and untouched outer validation data. "
        "Any later imputation study must be a separate, explicitly approved analysis.")
done()


<a id="s17"></a>

## 17 · StageStore inventory of intermediate products

Lists every intermediate saved during the run so reviewers can open any stage without re-running the notebook.

**About the code below**

| | |
|:--|:--|
| **Reads** | The StageStore folder. |
| **Does** | (1) lists all stages and products, (2) saves the manifest, (3) loads one product as a sample review. |
| **Writes** | `stages/stage_manifest.csv`. |
| **Check before moving on** | Each section from 1 to 16 that saves products appears in the manifest. |

In [ ]:
# =============================================================================
#  SECTION 17 · STAGESTORE INVENTORY
#  Reads  : the StageStore folder
#  Does   : lists every stage and product saved during the run, saves the manifest, loads one product as a sample review
#  Writes : stages/stage_manifest.csv
#  Result : manifest (one row per saved product)
# =============================================================================
banner(17, "StageStore inventory",
       "Every intermediate product saved during this run")

manifest = STAGES.list_stages()
manifest.to_csv(OUT/"stages"/"stage_manifest.csv", index=False)
kpis([kpi("Products saved", len(manifest)), kpi("Stages", manifest["stage"].nunique())])
show(manifest[["stage", "name", "rows", "note"]], title="Stage manifest", bars=["rows"], max_rows=80)
def _stage_findings():
    n_st = manifest["stage"].nunique(); rows = pd.to_numeric(manifest["rows"], errors="coerce")
    samp = manifest.loc[rows.isin([2000, 5000, 10000])]
    out = [f"<b>{n_(len(manifest))}</b> products were saved across <b>{n_(n_st)}</b> stages, so every intermediate result can be opened without re-running the notebook."]
    if len(samp):
        out.append(f"<b>{n_(len(samp))}</b> products are capped samples ({', '.join(samp['name'].astype(str).head(6))}"
                   f"{'...' if len(samp) > 6 else ''}). Use the full CSVs under <code>validated/</code> for analysis; StageStore keeps samples for quick review.")
    return out


report(_stage_findings)

# ---- sample review: load one saved product back to prove the round trip works -------------------
REVIEW_STAGE, REVIEW_NAME = "08_comparison", "scorecard"
show(STAGES.load(REVIEW_STAGE, REVIEW_NAME), title=f"Sample review · {REVIEW_STAGE} / {REVIEW_NAME}")
done()


<a id="s18"></a>

## 18 · Row-conservation ledger, readiness checklist and validated exports

Confirms that no rows were lost between stages, summarises what is (and is not) ready for mapping and reporting, and writes the validated exports.

**About the code below**

| | |
|:--|:--|
| **Reads** | Everything built in sections 1 to 17. |
| **Does** | (A) checks row conservation stage by stage, (B) builds the readiness checklist and lists the action for each open item, (C) writes the validated exports, (D) shows the output folder overview. |
| **Writes** | `validated/aseer_*` exports, `audit/row_conservation.csv`, `audit/geoai_readiness_gate.csv`. |
| **Check before moving on** | Every ledger row shows ✔ pass. Items marked **OPEN** are sign-off or policy blockers, not code failures. |

> **Why it matters.** A ledger that balances is the proof that no join silently dropped or duplicated rows.

In [ ]:
# =============================================================================
#  SECTION 18 · LEDGER, READINESS AND VALIDATED EXPORTS
#  Reads  : everything built in sections 1 to 17
#  Does   : checks row conservation, builds the readiness checklist with an action per open item, writes the validated exports
#  Writes : validated/aseer_* exports, audit/row_conservation.csv, audit/geoai_readiness_gate.csv
#  Result : ledger, readiness, OPEN_ITEMS (used by the executive summary)
# =============================================================================
banner(18, "Ledger, readiness & validated exports",
       "Row conservation · sign-off checklist · export inventory")

# ---- 1. row-conservation ledger: expected versus observed rows at each stage ------------------------
sub("A", "Row-conservation ledger")
ledger = pd.DataFrame([
    row_conservation("asir_su", GATE["total_sus"], len(asir_su)),
    row_conservation("longform_plots", GATE["expected_plots"], len(longform)),
    row_conservation("anti_clip_su", GATE["total_sus"], len(su_tree)),
    row_conservation("gate1", GATE["sus_no_trees"], len(gate1["zero_su_ids"])),
    row_conservation("gate2", GATE["gate2_plots"], len(gate2["gate2_plots"])),
    row_conservation("ready_plots", len(plot_aseer), len(globals().get("plots_ready", plot_aseer))),
    row_conservation("ready_trees", len(trees_aseer), len(globals().get("trees_ready", trees_aseer))),
])
show(ledger, title="Row conservation", caption="Expected versus observed rows at each stage", max_rows=20)
ledger.to_csv(OUT/"audit"/"row_conservation.csv", index=False)
report(lambda: [
    f"<b>{int(ledger['pass'].sum())}</b> of {len(ledger)} stages conserve their rows (delta = 0). "
    + ("No join dropped or duplicated a row between sections." if bool(ledger['pass'].all()) else "Investigate every row marked fail before exporting."),
])

# ---- 2. readiness checklist: PASS = done, OPEN = needs a human sign-off or an input ----------------
sub("B", "Readiness checklist")
readiness = pd.DataFrame([
    {"item": "gates_match_baked_targets", "pass": bool(check["match"].all())},
    {"item": "anti_clip_preserves_430", "pass": len(su_tree) == 430},
    {"item": "eda_missingness_univariate_bivariate_done",
     "pass": (OUT/"audit"/"statistical_numeric_profile.csv").exists()},
    {"item": "outliers_flagged_not_deleted",
     "pass": ("tree_population" in dir() and len(tree_population) == len(trees_aseer))},
    {"item": "old_vs_new_comparison_done",
     "pass": "scorecard" in dir() or (OUT/"stages").exists()},
    {"item": "tree_fv_metrics_calculated", "pass": "trees_fv" in dir() and len(trees_fv) > 0},
    {"item": "gate1_field_class_complete", "pass": False},
    {"item": "gate2_field_class_complete", "pass": False},
    {"item": "html_atlas_and_static_maps_built",
     "pass": (OUT/"maps").exists() and any((OUT/"maps").rglob("*"))},
    {"item": "biomass_maps_use_provisional_fv_only",
     "pass": True},
    {"item": "analytical_population_preserves_all_tree_rows",
     "pass": "tree_population" in dir() and len(tree_population) == len(trees_aseer)},
    {"item": "lidar_disparity_section_present_no_imputation",
     "pass": True},
    {"item": "lidar_observations_available",
     "pass": "lidar_compare" in dir() and len(lidar_compare) > 0},
    {"item": "geoai_validation_protocol_frozen",
     "pass": (OUT/"audit"/"geoai_validation_protocol.csv").exists()},
])
checklist(readiness, "item", "pass", title="Data-preparation readiness")
_open = readiness.loc[~readiness["pass"].astype(bool), "item"].tolist()
_ACTIONS = {
    "gate1_field_class_complete": "Field team completes <code>gates/gate1_field_classification_TEMPLATE.csv</code> (survey_status, validated_by, date, comments).",
    "gate2_field_class_complete": "Field team completes <code>gates/gate2_field_classification_TEMPLATE.csv</code>.",
    "lidar_observations_available": "Place the GEDI L2A footprint table at the path held in <code>GEDI_LOCAL_FILE</code> (default <code>H:/3.NFI/GEDIImages/gedi_l2a_footprints.parquet</code>; change it with the <code>NFI_GEDI_DIR</code> environment variable) (helper cell after section 15), then re-run section 15.",
}
OPEN_ITEMS = [(i, _ACTIONS.get(i, "Review and sign off.")) for i in _open]
if _open:
    callout("warning", f"{len(_open)} item(s) still open",
            "These are sign-off or policy blockers, not code failures. What to do next:")
    show(pd.DataFrame(OPEN_ITEMS, columns=["open item", "action"]).assign(**{"open item": lambda d: d["open item"].map(_pretty)}),
         title="Open items and their actions", index=False)
readiness.to_csv(OUT/"audit"/"geoai_readiness_gate.csv", index=False)

# ---- 3. write the validated exports ------------------------------------------------------------
sub("C", "Validated exports")
val = OUT / "validated"
longform.to_csv(val/"aseer_longform_su_plot_tree_area.csv", index=False)
su_tree.drop(columns="geometry", errors="ignore").to_csv(val/"aseer_su_anti_clip_tree_area.csv", index=False)
trees_aseer.to_csv(val/"aseer_trees_long.csv", index=False)
if "trees_fv" in dir():
    fv_deliverable_frame(trees_fv).to_csv(val/"aseer_trees_fv_metrics_provisional.csv", index=False)
plot_aseer.to_csv(val/"aseer_plots_coord_qa.csv", index=False)
try:
    su_tree.to_file(val/"aseer_su_anti_clip_tree_area.gpkg", layer="asir_su", driver="GPKG")
except Exception as e:
    say(f"GeoPackage export skipped: {e}", "warning")
STAGES.save("13_validated", "longform_final", longform)
flush_saved()

file_inventory(val, "*", title="Validated exports (newest first)", limit=25)
sub("D", "Output folder overview")
output_overview(OUT)
callout("success", "Run complete",
        f"Validated exports and diagnostics are under <code>{OUT}</code>.")
done()


<a id="s19"></a>

## 19 · Executive summary and report export

Collects the headline numbers, every finding recorded during the run, the open items and the recommended next steps. It also writes a stand-alone HTML report and a findings CSV that can be shared without the notebook.

**About the code below**

| | |
|:--|:--|
| **Reads** | Everything computed above, plus the findings recorded by each section. |
| **Does** | (A) headline cards, (B) charts of what is kept at each step, (C) open items and next steps, (D) all findings by section, (E) run times, (F) report export. |
| **Writes** | `documents/NFI_analysis_report.html`, `documents/NFI_analysis_findings.csv`. |
| **Check before moving on** | Open the HTML report and read the open items first. |

> **Why it matters.** Reviewers should not need to scroll through 19 sections to learn where the data stand.

In [ ]:
# =============================================================================
#  SECTION 19 · EXECUTIVE SUMMARY AND REPORT EXPORT
#  Reads  : everything computed above, plus every findings panel recorded during the run
#  Does   : headline cards, an overview chart, open items, next steps, run times, report export
#  Writes : documents/NFI_analysis_report.html and documents/NFI_analysis_findings.csv
#  Result : a stand-alone report that can be shared without the notebook
# =============================================================================
banner(19, "Executive summary & report export",
       "Headline numbers · what the data can support · open items · next steps · shareable report")


def _g(name, default=None):
    """Read a variable created in an earlier section; return `default` if that section was not run."""
    return globals().get(name, default)


# ---- A. headline numbers -----------------------------------------------------------------------------
sub("A", "Headline numbers")
_su_n, _pl_n, _tr_n = len(_g("asir_su", [])), len(_g("plots_ready", [])), len(_g("trees_ready", []))
_g1 = len(_g("gate1", {}).get("zero_su_ids", [])) if _g("gate1") else None
_g2 = len(_g("gate2", {}).get("gate2_plots", [])) if _g("gate2") else None
_lf = _g("longform")
_pl_tree = int(_lf["has_tree_record"].sum()) if _lf is not None else None
kpis([
    kpi("Sampling units", _su_n, sub="Asir universe"),
    kpi("Gate 1 · SUs without trees", _g1 if _g1 is not None else "n/a", sub=pct(_g1, _su_n) + " of SUs" if _g1 is not None else "", kind="warn"),
    kpi("Design plots", len(_lf) if _lf is not None else "n/a", sub="SU × plot lattice"),
    kpi("Plots with tree records", _pl_tree if _pl_tree is not None else "n/a",
        sub=pct(_pl_tree, len(_lf)) + " of design plots" if _lf is not None else ""),
    kpi("Tree records", _tr_n, sub="ready clean"),
    kpi("Readiness", f"{int(_g('readiness')['pass'].sum())} / {len(_g('readiness'))}" if _g("readiness") is not None else "n/a",
        sub="items passed", kind="ok"),
])

# ---- B. overview chart: where the trees are, and where the gaps are ---------------------------------------
sub("B", "Where the data are, and where the gaps are")
_reg = _g("by_region_ready")
if _reg is not None and len(_reg):
    _r = _reg.sort_values("n_su", ascending=True)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})
    axs[0].barh(_r["region"], _r["n_su"] - _r["n_gate1_zero_tree_su"], color=CHART_COLORS[2], label="SUs with tree records")
    axs[0].barh(_r["region"], _r["n_gate1_zero_tree_su"], left=_r["n_su"] - _r["n_gate1_zero_tree_su"], color=CHART_COLORS[3], label="Gate 1 · no tree records")
    axs[0].set(xlabel="Sampling units", title="Sampling units by region"); axs[0].legend(fontsize=9, loc="lower right"); axs[0].grid(axis="y", visible=False)
    axs[1].barh(_r["region"], _r["n_tree_records"], color=CHART_COLORS[0])
    for _i, _v in enumerate(_r["n_tree_records"]):
        axs[1].text(_v, _i, f" {int(_v):,}", va="center", fontsize=9)
    axs[1].set(xlabel="Tree records", title="Tree records by region"); axs[1].grid(axis="y", visible=False)
    axs[1].set_yticklabels([])
    fig.tight_layout()
    _ex = OUT / "figures" / "executive_summary.png"
    fig.savefig(_ex, dpi=170); plt.close(fig)
    show_fig(_ex, "Executive overview · sampling units and tree records by region",
             read="Left: each bar is a region's sampling units, split into units with tree records and Gate 1 units without. "
                  "Right: the tree records that each region contributes. Regions with long red segments or short blue bars cannot support stand-alone estimates.")

# ---- C. what the data can and cannot support, then open items and next steps ------------------------------
sub("C", "What the data can support, open items and next steps")


def _summary_findings():
    out = []
    if _g1 is not None and _pl_tree is not None:
        out.append(f"<b>Coverage.</b> Only {n_(_pl_tree)} of {n_(len(_lf))} design plots ({pct(_pl_tree, len(_lf))}) hold tree records, and {pct(_g1, _su_n)} of the sampling units are empty. "
                   "Empty cells are unclassified, not zero forest, until Gate 1 and Gate 2 are signed off.")
    if _reg is not None and len(_reg):
        big = _reg.sort_values("n_tree_records", ascending=False).iloc[0]
        out.append(f"<b>Regional balance.</b> {big['region']} supplies {pct(big['n_tree_records'], _reg['n_tree_records'].sum(), 0)} of all tree records. "
                   "Estimates for the other regions rest on very few records.")
    out.append("<b>Metrics.</b> Volume, biomass and carbon are provisional (assumed plot area, stumps in the frame, a rule that makes derived metrics collinear). "
               "Report DBH and height statistics with confidence; report per-hectare figures only after <code>plot_area_ha</code> is confirmed.")
    out.append("<b>Integrity.</b> Row conservation holds at every stage, the Gate targets reconcile, and no measurement was overwritten or imputed.")
    return out


report(_summary_findings, "Headline findings")

if _g("OPEN_ITEMS"):
    show(pd.DataFrame(OPEN_ITEMS, columns=["open item", "action"]).assign(**{"open item": lambda d: d["open item"].map(_pretty)}),
         title="Open items awaiting sign-off", index=False)

_steps = []
if _g("gate1"):
    _steps.append(f"Field-verify the {n_(_g1)} Gate 1 sampling units and the {n_(_g2)} Gate 2 plots using the two sign-off templates in <code>gates/</code>.")
_d_ = _g("_d")
if _d_ is not None and len(_d_):
    _far = int((_d_ > 1000).sum())
    if _far:
        _steps.append(f"Check the {n_(_far)} plots whose field and office coordinates differ by more than 1 km (paired table in <code>figures/coord_crosscheck/</code>).")
_steps.append("Confirm with the data owner how <code>Area_H</code> was computed, and supply <code>plot_area_ha</code> so per-hectare estimates can be unblocked.")
if "possible_duplicate_record" in _g("tree_population", pd.DataFrame()).columns and int(_g("tree_population")["possible_duplicate_record"].sum()):
    _steps.append(f"Review the {n_(int(_g('tree_population')['possible_duplicate_record'].sum()))} tree records flagged as possible duplicates against the field forms.")
_steps.append("Remove stump records from the volume and biomass frame before totals are reported.")
if not (_g("lidar_compare") is not None and len(_g("lidar_compare"))):
    _steps.append("Provide the GEDI L2A footprint table (helper cell after section 15) and re-run section 15 for the agreement metrics.")
_steps.append("Only after these steps, start the GeoAI phase with the frozen protocol in section 16.")
findings(_steps, "Recommended next steps")

# ---- D. every finding, by section -------------------------------------------------------------------------
sub("D", "All findings by section")
_fl = pd.DataFrame(FINDINGS_LOG)
if len(_fl):
    _fl["finding"] = _fl["finding"].str.slice(0, 400)
    show(_fl[["section", "section_name", "topic", "finding"]], title="Findings recorded during this run",
         caption="Full text is in the exported report.", max_rows=200, index=False)

# ---- E. run times ----------------------------------------------------------------------------------------
sub("E", "Run times")
_rt = run_times_table()
show(_rt[["section", "name", "seconds"]], title="Section run times (seconds)", bars=["seconds"], max_rows=30, index=False)
say(f"Total run time so far: <b>{_rt['seconds'].sum() / 60:,.1f} min</b>.", "info")

# ---- F. export the report --------------------------------------------------------------------------------
sub("F", "Report export")
_docs = OUT / "documents"
_docs.mkdir(parents=True, exist_ok=True)
_hl = [("Sampling units", n_(_su_n)), ("Gate 1 SUs", n_(_g1) if _g1 is not None else "n/a"),
       ("Design plots", n_(len(_lf)) if _lf is not None else "n/a"), ("Plots with trees", n_(_pl_tree) if _pl_tree is not None else "n/a"),
       ("Tree records", n_(_tr_n))]
_rp = export_report_html(_docs / "NFI_analysis_report.html", "Asir NFI data preparation · analysis report", _hl,
                         open_items=[(a, _plain(b)) for a, b in (_g("OPEN_ITEMS") or [])],
                         next_steps=[_plain(x) for x in _steps])
_fl.to_csv(_docs / "NFI_analysis_findings.csv", index=False)
say("Stand-alone report written:", "save", code=_rp)
say("Findings table written:", "save", code=_docs / "NFI_analysis_findings.csv")
show_html_map(_rp, title="Open the analysis report", open_browser=False)
done()


---
### Scientific safeguard

This notebook intentionally separates **measurement validation**, **LiDAR disparity evidence**, and any future **imputation**. A later imputation model should only be considered after demonstrating adequate spatial support, temporal compatibility, low bias, acceptable error, stable performance across ecological strata, and independent, spatially blocked validation. This notebook performs no imputation.

# Part B · Expansion block

Everything **above** this heading is Part A, the preserved Asir pipeline (Gates, anti-clip lattice, cleaning, EDA, cartography, GEDI, GeoAI protocol, ledger). Everything **below** extends it with national evidence and the calculation-document formulas.

| Block | Sections | What it adds |
|:--|:--|:--|
| Environment and design | E0, E1 | Package root, modules, output folders, constants, research design |
| National LULC Validation | E2, E3, E4 | Key numbers, rule counts, confusion matrices, restored figures |
| Missingness | E5 | Completeness by table, handling policy, imputation benchmarks (diagnostic only) |
| Calculation document | E6, E7, E8 | Tree and shrub biomass and carbon (provisional), land-use proportions by SU |
| Statistics and spatial | E9, E10, E11 | Correlations and PCA, Moran's I and Gi\*, height model with SU-blocked validation |
| Maps and close-out | E12, E13 | National maps, run summary, limitations |

**How to run.** Part B reloads its own inputs from `config.yaml`, so it can run after a kernel restart. Run the bootstrap cell and the presentation kit cell first, because the banners, tables and findings panels come from the kit. Part B stores its tables under `nat_*` names and never overwrites the Part A frames (`trees`, `su`, `trees_aseer`, `plot_aseer`).


<a id="e0"></a>

## E0 · Expansion environment

Locates the package root, reloads the configuration, imports the expansion modules, creates the output folders and checks that every input exists.

**About the code below**

| | |
|:--|:--|
| **Reads** | `config.yaml`; modules under `src/`. |
| **Does** | (1) finds the package root, (2) clears stale `nfi_*` modules, (3) loads the config, (4) creates `outputs/expansion/`, (5) lists inputs and the calculation-document constants. |
| **Writes** | `outputs/expansion/{tables,figures,maps,logs}`. Defines `EXP_ROOT`, `CFG`, `OUT`, `EXP_OUT`, `SEED`. |
| **Check before moving on** | Every row of the *Input files* checklist shows PASS. A missing file stops the matching section later with a less helpful message. |

> **Why it matters.** The constants table is the single place where the formula assumptions are visible. Read it before trusting any biomass or carbon number below.


In [ ]:
# =============================================================================
#  SECTION E0 · EXPANSION ENVIRONMENT
#  Reads  : config.yaml and the project modules under src/
#  Does   : locates the package root, imports the expansion modules, creates output folders, checks inputs
#  Writes : outputs/expansion/{tables, figures, maps, logs}
#  Result : EXP_ROOT, CFG, OUT, EXP_OUT, SEED, plus empty defaults that later cells fill in
# =============================================================================
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
banner("E0", "Expansion environment", "Config reload · module import · output folders · input check")


def _find_package_root():
    candidates = []
    if "ROOT" in globals():                      # set by the bootstrap cell
        candidates.append(Path(ROOT))
    candidates += [Path.cwd(), Path(r"F:/1.KSA/29.NFI/12.NFI_Comprehensive_Expanded_V5")]
    for p in candidates:
        if (p / "config.yaml").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("config.yaml and src/ were not found. Run the bootstrap cell first, "
                            "or change into the package folder.")


EXP_ROOT = _find_package_root()
os.chdir(EXP_ROOT)
if str(EXP_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(EXP_ROOT / "src"))
for _m in [m for m in sys.modules if m.startswith("nfi_")]:     # pick up edits to src/ without a restart
    del sys.modules[_m]

from nfi_audit import load_config, ensure_output_dirs, read_first_sheet, attach_keys
from nfi_lulc_html import (
    load_lulc_validated, lulc_key_numbers, validation_rule_counts, final_lulc_distribution,
    load_confusion_csv, compute_oa_from_confusion, plot_validation_true_false,
    plot_final_lulc_bars, plot_national_points, forest_priority_summary,
)
from nfi_calc_biomass import (
    apply_tree_metrics, exclude_by_substrings, shrub_agb_mg, tree_bgb_mg, carbon_mg,
    total_biomass_mg, land_use_proportion_by_su, FORM_FACTOR, WOOD_DENSITY, BEF,
    ROOT_SHOOT, CARBON_FRACTION, CONVERSION_FACTOR_HA, TREE_EXCLUDE, SHRUB_EXCLUDE,
)
from nfi_missingness_expanded import (
    missingness_by_column, classify_variable_types, simple_imputation_benchmarks, try_mice_numeric,
)
from nfi_esda_expanded import points_gdf, global_moran_i, getis_ord_gi_star

CFG = load_config(EXP_ROOT / "config.yaml")
OUT = ensure_output_dirs(CFG)
EXP_OUT = Path(CFG["paths"]["output_root"]) / "expansion"
for _s in ("tables", "figures", "maps", "logs"):
    (EXP_OUT / _s).mkdir(parents=True, exist_ok=True)
SEED = int(CFG["project"].get("random_seed", 42))
np.random.seed(SEED)

# Values that later sections fill in. The defaults keep the summary cell safe if a section is skipped.
kn, shrub_note, mi, ML_GAP = {}, "not processed", {}, None

# ---- inputs ---------------------------------------------------------------------------
_inputs = [(f"table: {k}", Path(v)) for k, v in CFG["tables"].items() if isinstance(v, (str, Path))]
_inputs.append(("LULC validated CSV", EXP_ROOT / CFG["paths"]["lulc_validated_csv"]))
inputs = pd.DataFrame({"item": [n for n, _ in _inputs],
                       "pass": [p.exists() for _, p in _inputs],
                       "path": [str(p) for _, p in _inputs]})
checklist(inputs[["item", "pass"]], item_col="item", pass_col="pass", title="Input files")
if not inputs["pass"].all():
    say("Missing inputs: " + ", ".join(inputs.loc[~inputs["pass"], "item"]) +
        ". Fix the paths in config.yaml before running the sections that need them.", "warning")

# ---- constants ------------------------------------------------------------------------
const = pd.DataFrame({
    "constant": ["Form factor", "Wood density", "Biomass expansion factor (BEF)", "Root to shoot ratio",
                 "Carbon fraction", "Conversion factor to per hectare"],
    "value": [FORM_FACTOR, WOOD_DENSITY, BEF, ROOT_SHOOT, CARBON_FRACTION, CONVERSION_FACTOR_HA]})
const["value"] = const["value"].map(lambda v: str(v)[:90])
show(const, title="Calculation-document constants in use",
     caption="Every provisional biomass and carbon number in Part B is built from these values.")
say("Expansion outputs go to", "save", code=EXP_OUT)
done()


<a id="e1"></a>

## E1 · Research design note

| Element | Statement |
|:--|:--|
| **Population and frame** | KSA NFI sampling units (SU) and plots from the national cleansed tables. The Asir 430-SU frame stays the Gate 1 and Gate 2 validation case study from Part A. |
| **Estimands** | These are sample quantities, not design-based estimates, unless weights and SU areas are supplied: (1) final LULC class at office coordinates, (2) tree size distributions and provisional carbon from the calculation document, (3) the height to DBH relationship, judged by SU-blocked validation. |
| **Causal claims** | None. Nothing here identifies a causal effect. Do not read model importance or spatial clustering as cause. |
| **Policy** | Remote height imputation is **off** for official carbon (`policy.remote_height_imputation: false`). Imputation in E5 is a diagnostic only. |
| **Unit of analysis** | Trees are nested in plots, plots in SUs. Any test or model that ignores this nesting overstates its precision, so validation in E10 and E11 works at SU level. |


<a id="e2"></a>

## E2 · National LULC Validation ingest

Loads the validated office-point land-cover table and summarises how many points were accepted, how many were updated, and what the final class mix looks like.

**About the code below**

| | |
|:--|:--|
| **Reads** | The validated LULC CSV named in `config.yaml`. |
| **Does** | (A) computes headline numbers, (B) counts the validation rules and the final class distribution, (C) writes findings. |
| **Writes** | `expansion/tables/lulc_key_numbers.csv`, `lulc_validation_rule_counts.csv`, `lulc_final_distribution.csv`, `lulc_forest_priority_summary.csv`. |
| **Check before moving on** | The row count matches the validation report, and the accepted share is plausible. A very low share points to a rule problem, not to bad land cover. |

> **Why it matters.** These points are the land-cover reference for everything national. An error here carries into the land-use proportions in E7.


In [ ]:
# =============================================================================
#  SECTION E2 · NATIONAL LULC VALIDATION INGEST
#  Reads  : the validated LULC CSV (CFG["paths"]["lulc_validated_csv"])
#  Does   : headline numbers, validation rule counts, final class distribution, written findings
#  Writes : expansion/tables/lulc_*.csv
#  Result : lulc, kn, rules, dist, forest
# =============================================================================
banner("E2", "National LULC Validation ingest", "Office-point land cover · validation rules · final class mix")

lulc_path = EXP_ROOT / CFG["paths"]["lulc_validated_csv"]
lulc = load_lulc_validated(lulc_path)
kn = lulc_key_numbers(lulc)
pd.DataFrame([kn]).to_csv(EXP_OUT / "tables" / "lulc_key_numbers.csv", index=False)

sub("A", "Headline numbers")
_num = [(k, v) for k, v in kn.items()
        if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_))]
kpis([kpi(_pretty(k), v) for k, v in _num[:6]])
_shown = {k for k, _ in _num[:6]}
_rest = [(k, v) for k, v in kn.items() if k not in _shown]
if _rest:
    show(pd.DataFrame({"measure": [_pretty(k) for k, _ in _rest], "value": [str(v)[:120] for _, v in _rest]}),
         title="Other key numbers")

sub("B", "Validation rules and final class distribution")
rules = validation_rule_counts(lulc)
show(rules, title="Validation rule counts")
rules.to_csv(EXP_OUT / "tables" / "lulc_validation_rule_counts.csv")

dist = final_lulc_distribution(lulc)
_dist_num = list(dist.select_dtypes("number").columns)[:1]
show(dist.head(20), title="Final LULC distribution (top 20)", bars=_dist_num or None)
dist.to_csv(EXP_OUT / "tables" / "lulc_final_distribution.csv")

forest = forest_priority_summary(lulc)
show(forest, title="Forest priority summary")
forest.to_csv(EXP_OUT / "tables" / "lulc_forest_priority_summary.csv", index=False)


def _lulc_findings():
    out = []
    keyed = lulc.drop_duplicates(["su_no", "plot_no"])
    out.append(f"The file holds <b>{n_(len(lulc))}</b> office-point rows for <b>{n_(len(keyed))}</b> unique plots "
               f"in <b>{n_(lulc['su_no'].nunique())}</b> sampling units.")
    ok = lulc["validation_true"].astype(int)
    out.append(f"<b>{pct(ok.sum(), len(ok))}</b> of rows were accepted as they stood. The other "
               f"<b>{n_(len(ok) - ok.sum())}</b> rows were changed during validation, so their final class is the "
               "validated one, not the first office reading.")
    vc = lulc["final_lulc"].value_counts()
    top3 = vc.head(3)
    out.append(f"The most frequent final class is <b>{vc.index[0]}</b> ({pct(vc.iloc[0], vc.sum())} of rows). "
               f"The top three classes together hold {pct(top3.sum(), vc.sum())}, so the national mix is dominated by few classes.")
    if "inside_forest_reserved" in lulc.columns:
        s = lulc["inside_forest_reserved"].astype(str).str.lower().isin(["1", "true", "yes", "y"])
        out.append(f"<b>{pct(s.sum(), len(s))}</b> of points fall inside a forest reserve ({n_(s.sum())} rows).")
    return out


report(_lulc_findings)
done()


<a id="e3"></a>

## E3 · Multi-class accuracy products

The HTML validation report dropped its multi-class charts and matrices from the body. The underlying CSV and PNG files still exist, so this section brings them back.

**About the code below**

| | |
|:--|:--|
| **Reads** | Confusion-matrix CSVs and error-matrix PNGs under `data/lulc/`. |
| **Does** | (A) loads each confusion matrix that exists, (B) computes overall accuracy per comparison, (C) shows the two error-matrix figures. |
| **Writes** | `expansion/tables/cm_*.csv`, `accuracy_field_vs_final_lulc.csv`, `lulc_oa_from_confusion.csv`. |
| **Check before moving on** | The *Products found* table shows every file as found. A missing file is skipped, not guessed. |

> **Why it matters.** Overall accuracy against field data is the main evidence that the final LULC class can be trusted.


In [ ]:
# =============================================================================
#  SECTION E3 · MULTI-CLASS ACCURACY PRODUCTS
#  Reads  : data/lulc/confusion_*.csv, accuracy_field_vs_final_lulc.csv, error-matrix PNGs
#  Does   : loads each product that exists, computes overall accuracy, shows the figures
#  Writes : expansion/tables/cm_*.csv, accuracy_field_vs_final_lulc.csv, lulc_oa_from_confusion.csv
#  Result : acc (overall accuracy per comparison)
# =============================================================================
banner("E3", "Multi-class accuracy products", "Confusion matrices · overall accuracy · error-matrix figures")

cm_dir = EXP_ROOT / "data" / "lulc"
cm_files = {
    "field_vs_final": cm_dir / "confusion_field_vs_final_lulc.csv",
    "field_vs_esa": cm_dir / "confusion_field_vs_esa.csv",
    "field_vs_esri": cm_dir / "confusion_field_vs_esri.csv",
    "accuracy_field_vs_final": cm_dir / "accuracy_field_vs_final_lulc.csv",
}
acc_rows, status_rows = [], []
for name, path in cm_files.items():
    if not path.exists():
        status_rows.append({"product": name, "found": False, "rows": 0, "columns": 0})
        continue
    df = load_confusion_csv(path)
    status_rows.append({"product": name, "found": True, "rows": df.shape[0], "columns": df.shape[1]})
    if name.startswith("field_vs"):                       # a confusion matrix
        oa = compute_oa_from_confusion(df)
        oa["comparison"] = name
        acc_rows.append(oa)
        df.to_csv(EXP_OUT / "tables" / f"cm_{name}.csv", index=False)
    else:                                                 # the accuracy summary table
        df.to_csv(EXP_OUT / "tables" / f"{name}.csv", index=False)
    show(df, title=_pretty(name), caption="First 12 rows", max_rows=12)

sub("A", "Products found")
show(pd.DataFrame(status_rows), title="Products found")

sub("B", "Overall accuracy by comparison")
acc = pd.DataFrame(acc_rows)
if len(acc):
    show(acc, title="Overall accuracy from the confusion matrices")
    acc.to_csv(EXP_OUT / "tables" / "lulc_oa_from_confusion.csv", index=False)

    def _oa_findings():
        col = next((c for c in acc.columns if "overall" in c.lower() or c.lower() in ("oa", "overall_accuracy")), None)
        if col is None:
            return ["Overall accuracy columns were not recognised. Read the table above directly."]
        v = pd.to_numeric(acc[col], errors="coerce")
        scale = 100 if v.max() <= 1 else 1
        hide = ("Overall accuracy hides weak classes. Read the matrices for the producer and user accuracy of the classes "
                "that matter most for forest reporting.")
        if v.notna().sum() < 2 or v.max() == v.min():
            names = ", ".join(_pretty(c) for c in acc["comparison"])
            return [f"Overall accuracy is <b>{v.max() * scale:.1f}%</b> for {names}.", hide]
        hi, lo = acc.loc[v.idxmax(), "comparison"], acc.loc[v.idxmin(), "comparison"]
        return [f"Overall accuracy is highest for <b>{_pretty(hi)}</b> at <b>{v.max() * scale:.1f}%</b> and lowest for "
                f"<b>{_pretty(lo)}</b> at <b>{v.min() * scale:.1f}%</b>.", hide]
    report(_oa_findings)
else:
    say("No confusion matrices were found, so overall accuracy was not computed.", "warning")

sub("C", "Error-matrix figures")
for fname, title, read in [
    ("17_error_matrix_field_vs_final_lulc.png", "Error matrix, field versus final LULC",
     "Rows are field classes and columns are map classes. Counts on the diagonal agree. Dark off-diagonal cells show which classes are confused."),
    ("18_error_matrices_compact.png", "Error matrices, compact set",
     "The same reading as above, for each comparison product side by side."),
]:
    p = cm_dir / fname
    show_fig(p, title=title, read=read) if p.exists() else say("Figure file not found:", "warning", code=p)
done()


<a id="e4"></a>

## E4 · LULC validation figures

Redraws the three standard validation figures from the loaded table.

**About the code below**

| | |
|:--|:--|
| **Reads** | `lulc` and `dist` from E2. |
| **Does** | Draws accepted versus updated counts, the final class bars and the national office-point map. |
| **Writes** | `expansion/figures/lulc_accepted_vs_updated.png`, `lulc_final_distribution.png`, `expansion/maps/map_national_lulc_office_points.png`. |
| **Check before moving on** | All three figure cards appear. A warning line means that one drawing function failed; the others are unaffected. |


In [ ]:
# =============================================================================
#  SECTION E4 · LULC VALIDATION FIGURES
#  Reads  : lulc, dist (section E2)
#  Does   : draws accepted versus updated, final class bars, national office-point map
#  Writes : expansion/figures/lulc_*.png, expansion/maps/map_national_lulc_office_points.png
#  Result : three figure files
# =============================================================================
banner("E4", "LULC validation figures", "Accepted versus updated · class bars · national point map")

_jobs = [
    ("Accepted versus updated points", plot_validation_true_false, lulc,
     EXP_OUT / "figures" / "lulc_accepted_vs_updated.png",
     "Two bars: points accepted as first read, and points changed in validation. A tall second bar means the first reading needed much correction."),
    ("Final LULC class distribution", plot_final_lulc_bars, dist,
     EXP_OUT / "figures" / "lulc_final_distribution.png",
     "Bar length is the number of points per final class. Short bars are rare classes with weak statistics."),
    ("National office points by final class", plot_national_points, lulc,
     EXP_OUT / "maps" / "map_national_lulc_office_points.png",
     "Each dot is one office point, coloured by its final class. Look for gaps in coverage and for class clusters."),
]
for title, fn, data, path, read in _jobs:
    try:
        fig = fn(data, path)
        if fig is not None:
            plt.close(fig)
        show_fig(path, title=title, read=read)
    except Exception as e:
        say(f"{title} could not be drawn ({type(e).__name__}: {e}).", "warning")
done()


<a id="e5"></a>

## E5 · Missingness diagnostics, handling policy and imputation benchmarks

Measures how incomplete each national table is, states how each kind of gap is handled, and benchmarks simple imputation so the cost of filling gaps is visible.

**About the code below**

| | |
|:--|:--|
| **Reads** | The national trees, plots, SU master and shrubs tables, plus `lulc`. |
| **Does** | (A) summarises completeness per table, (B) lists the most incomplete columns, (C) draws the tree-table chart, (D) states the handling policy, (E) benchmarks imputation, (F) writes findings. |
| **Writes** | `expansion/tables/types_*.csv`, `missingness_*.csv`, `missingness_summary_by_table.csv`, `imputation_benchmarks_trees.csv`, `figures/missingness_trees_top15.png`. |
| **Check before moving on** | Read the handling policy table. It says which gaps are kept as gaps, which are modelled, and which are reported only. |

> **Why it matters.** Treating *missing*, *not applicable* and *true zero* as the same thing is the most expensive error in an inventory. Imputation here never feeds official carbon.


In [ ]:
# =============================================================================
#  SECTION E5 · MISSINGNESS DIAGNOSTICS, HANDLING POLICY, IMPUTATION BENCHMARKS
#  Reads  : national trees, plot, SU master and shrubs tables, plus lulc
#  Does   : completeness per table, most incomplete columns, handling policy, imputation benchmarks
#  Writes : expansion/tables/types_*.csv, missingness_*.csv, imputation_benchmarks_trees.csv, one figure
#  Result : nat_trees, nat_plot, nat_su, nat_shrubs (national frames used by E6 to E11)
# =============================================================================
banner("E5", "Missingness diagnostics", "Completeness · handling policy · imputation benchmarks (diagnostic only)")

# Independent of Part A memory state, and stored under nat_* names so Part A frames are never overwritten.
nat_trees = read_first_sheet(CFG["tables"]["trees"])
nat_plot = read_first_sheet(CFG["tables"]["plot"])
nat_su = read_first_sheet(CFG["tables"]["su"])
nat_shrubs = read_first_sheet(CFG["tables"]["shrubs"])
frames = {"trees": nat_trees, "plot": nat_plot, "su": nat_su, "shrubs": nat_shrubs, "lulc": lulc}

sub("A", "Completeness of each national table")
rows, miss_tables = [], {}
for name, df in frames.items():
    classify_variable_types(df).to_csv(EXP_OUT / "tables" / f"types_{name}.csv", index=False)
    miss = missingness_by_column(df)
    miss.to_csv(EXP_OUT / "tables" / f"missingness_{name}.csv", index=False)
    miss_tables[name] = miss
    share = df.isna().mean()
    rows.append({"table": name, "rows": len(df), "columns": df.shape[1],
                 "cells_missing_pct": float(df.isna().to_numpy().mean() * 100),
                 "columns_with_any_missing": int((share > 0).sum()),
                 "columns_over_50pct_missing": int((share > 0.5).sum()),
                 "most_incomplete_column": str(share.idxmax()) if len(share) else "-",
                 "its_missing_pct": float(share.max() * 100) if len(share) else 0.0})
miss_summary = pd.DataFrame(rows)
miss_summary.to_csv(EXP_OUT / "tables" / "missingness_summary_by_table.csv", index=False)
show(miss_summary, title="Completeness by table", bars=["cells_missing_pct"])

sub("B", "Most incomplete columns, table by table")
for name, miss in miss_tables.items():
    show(miss.head(10), title=f"{_pretty(name)}: ten most incomplete columns")

sub("C", "Tree table chart")
_m = (nat_trees.isna().mean().sort_values(ascending=False).head(15)[::-1]) * 100
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(_m.index.astype(str), _m.values, color=CHART_COLORS[0])
ax.set_xlabel("Records with no value (%)")
ax.set_title("Tree table: most incomplete columns")
_p = EXP_OUT / "figures" / "missingness_trees_top15.png"
fig.tight_layout(); fig.savefig(_p); plt.close(fig)
show_fig(_p, title="Tree table: most incomplete columns",
         read="Each bar is the share of tree records with no value. A long bar means the variable cannot support a complete-case analysis.")

sub("D", "Handling policy")
policy_tbl = pd.DataFrame([
    ["Not applicable by design (for example height of a stump)", "Structural gap",
     "Keep as a gap and add an eligibility flag. Exclude from the estimand, never fill.", "No"],
    ["Zero recorded as a measurement", "True zero",
     "Keep as zero. Count it in the denominator of means and proportions.", "Yes"],
    ["Height missing, DBH present, plausibly random given species and size", "Missing at random (assumed)",
     "Model-based fill (SU-blocked) as a sensitivity run only. Report results with and without it.", "No"],
    ["Whole plots or SUs unreachable or restricted", "Possibly not at random",
     "Qualitative review with field teams. Report as unobserved area, not as zero.", "No"],
    ["Coordinates missing or outside the grid", "Positional gap",
     "Flag and list. Do not place by guess. Resolve through the Part A coordinate checks.", "No"],
], columns=["situation", "reading", "method", "feeds official carbon"])
show(policy_tbl, title="How each kind of gap is handled",
     caption="Whether a gap is random cannot be proven from the data alone. The policy therefore keeps official results free of imputed values.")

sub("E", "Imputation benchmarks (diagnostic)")
num_cols = [c for c in ["tree_dbh", "tree_height", "tree_bole_height"] if c in nat_trees.columns]
if num_cols:
    bench = simple_imputation_benchmarks(nat_trees, num_cols, random_state=SEED)
    show(bench, title="Simple imputation benchmarks on the tree table")
    bench.to_csv(EXP_OUT / "tables" / "imputation_benchmarks_trees.csv", index=False)
    mice = try_mice_numeric(nat_trees, num_cols, random_state=SEED)
    if mice.get("status") == "ok":
        mice["imputed"].head().to_csv(EXP_OUT / "tables" / "mice_trees_preview.csv", index=False)
        say("MICE ran on the numeric tree block; a preview was saved.", "success")
    else:
        say(f"MICE was not run: {mice.get('reason')}", "note")
else:
    say("None of tree_dbh, tree_height, tree_bole_height were found, so the benchmarks were skipped.", "warning")
callout("warning", "Imputation is diagnostic here",
        "Official carbon uses field heights only. Remote height imputation is set to "
        f"<code>{CFG.get('policy', {}).get('remote_height_imputation')}</code> in <code>config.yaml</code>.")


def _miss_findings():
    out = []
    worst = miss_summary.sort_values("cells_missing_pct", ascending=False).iloc[0]
    out.append(f"The most incomplete table is <b>{worst['table']}</b> with <b>{worst['cells_missing_pct']:.1f}%</b> of cells empty. "
               f"Its emptiest column is <code>{worst['most_incomplete_column']}</code> ({worst['its_missing_pct']:.1f}% missing).")
    if "tree_height" in nat_trees.columns:
        h = nat_trees["tree_height"].isna()
        out.append(f"<b>{pct(h.sum(), len(h))}</b> of tree records ({n_(h.sum())}) have no height. "
                   "Those records cannot enter volume, biomass or carbon, so any national total built from them is a lower bound.")
    if "tree_dbh" in nat_trees.columns:
        d = nat_trees["tree_dbh"].isna()
        out.append(f"<b>{pct(d.sum(), len(d))}</b> of tree records have no DBH.")
    return out


report(_miss_findings)
done()


<a id="e6"></a>

## E6 · Tree biomass and carbon (provisional)

Applies the calculation-document formulas to every tree with a usable DBH and height, keeps a ledger of every record removed, and builds the anti-clip SU status table.

**About the code below**

| | |
|:--|:--|
| **Reads** | `nat_trees`, `nat_su` from E5. |
| **Does** | (A) applies the species rule and the numeric checks, (B) computes basal area, volume, biomass and carbon, (C) builds the SU frame with a status per unit. |
| **Writes** | `expansion/tables/national_trees_provisional_biomass.csv`, `national_su_tree_density_anticlip.csv`. |
| **Check before moving on** | The row ledger adds up, and the SU status table has no SU lost. Units with records but no usable DBH and height are listed as such, never as empty. |

> **Why it matters.** A unit whose trees all lack a height is not a zero-tree unit. Counting only measured trees would turn missing data into false zeros.


In [ ]:
# =============================================================================
#  SECTION E6 · TREE BIOMASS AND CARBON (PROVISIONAL) AND ANTI-CLIP SU STATUS
#  Reads  : nat_trees, nat_su (section E5)
#  Does   : species rule, numeric checks, calculation-document metrics, row ledger, SU status
#  Writes : expansion/tables/national_trees_provisional_biomass.csv, national_su_tree_density_anticlip.csv
#  Result : trees_m (trees with metrics), su_tree (one row per SU), dbh_col, ht_col, su_col
# =============================================================================
banner("E6", "Tree biomass and carbon (provisional)", "Calculation-document formulas · row ledger · anti-clip SU status")

dbh_col, ht_col = "tree_dbh", "tree_height"
_need = [c for c in (dbh_col, ht_col) if c not in nat_trees.columns]
if _need:
    raise KeyError(f"The tree table lacks required column(s) {_need}. Check config.yaml and the cleansed trees workbook.")
su_col = CFG["columns"]["su_id_in_child_tables"]

sub("A", "Species rule and numeric checks")
sp_col = "tree_species_scientific_name" if "tree_species_scientific_name" in nat_trees.columns else None
keep = exclude_by_substrings(nat_trees[sp_col], TREE_EXCLUDE) if sp_col else pd.Series(True, index=nat_trees.index)
trees_live = nat_trees.loc[keep].copy()
for _c in (dbh_col, ht_col):
    trees_live[_c] = pd.to_numeric(trees_live[_c], errors="coerce")
dbh_ok, ht_ok = trees_live[dbh_col].notna(), trees_live[ht_col].notna()
valid = dbh_ok & ht_ok

sub("B", "Calculation-document metrics")
trees_m = apply_tree_metrics(trees_live.loc[valid].copy(), dbh_col, ht_col)
trees_m["su_key"] = trees_m[su_col].astype(str).str.strip()

ledger = pd.DataFrame({
    "step": ["All tree records", "Removed by the species rule", "No usable DBH (after species rule)",
             "No usable height (after species rule)", "Retained with DBH and height"],
    "records": [len(nat_trees), int((~keep).sum()), int((~dbh_ok).sum()), int((~ht_ok).sum()), int(valid.sum())]})
show(ledger, title="Row ledger", bars=["records"],
     caption="The two missing-value rows can overlap, so they need not add up to the removed total.")
kpis([kpi("Tree records", len(nat_trees)), kpi("Retained with metrics", int(valid.sum()),
      sub=pct(valid.sum(), len(nat_trees)) + " of all records"),
      kpi("Removed by species rule", int((~keep).sum()), kind="info")])

_metric_cols = [c for c in [dbh_col, ht_col, "basal_area_m2", "volume_m3", "agb_mg", "carbon_mg_provisional"] if c in trees_m.columns]
show(trees_m[_metric_cols].describe().T.reset_index().rename(columns={"index": "variable"}),
     title="Distribution of the derived tree metrics")
trees_m.to_csv(EXP_OUT / "tables" / "national_trees_provisional_biomass.csv", index=False)

sub("C", "Anti-clip SU status")
su_ids = nat_su[CFG["columns"]["su_id_in_su_master"]].astype(str).str.strip()
frame = pd.DataFrame({"su_key": su_ids}).drop_duplicates()
n_all = nat_trees[su_col].astype(str).str.strip().value_counts().rename("n_tree_records_all")
n_use = trees_m["su_key"].value_counts().rename("n_tree_records_with_metrics")
su_tree = (frame.merge(n_all, left_on="su_key", right_index=True, how="left")
                .merge(n_use, left_on="su_key", right_index=True, how="left"))
for _c in ("n_tree_records_all", "n_tree_records_with_metrics"):
    su_tree[_c] = su_tree[_c].fillna(0).astype(int)
su_tree["n_tree_records"] = su_tree["n_tree_records_with_metrics"]          # name used by earlier versions
su_tree["su_status"] = np.select(
    [su_tree["n_tree_records_all"].eq(0), su_tree["n_tree_records_with_metrics"].eq(0)],
    ["No tree records", "Records but no usable DBH and height"], default="Usable tree records")
su_tree["tree_density_ha"] = (su_tree["n_tree_records"] / CFG["project"]["plots_per_su_design"]) * CONVERSION_FACTOR_HA
su_tree.to_csv(EXP_OUT / "tables" / "national_su_tree_density_anticlip.csv", index=False)
status = su_tree["su_status"].value_counts().rename_axis("su_status").reset_index(name="n_su")
show(status, title="Sampling-unit status", bars=["n_su"])


def _tree_findings():
    out = [f"<b>{pct(valid.sum(), len(nat_trees))}</b> of the {n_(len(nat_trees))} tree records carry both DBH and height and enter the metrics. "
           f"The species rule removes {n_((~keep).sum())} records, and {n_((~ht_ok).sum())} more lack a height."]
    s = su_tree["su_status"].value_counts()
    out.append(f"Of <b>{n_(len(su_tree))}</b> sampling units, <b>{n_(s.get('Usable tree records', 0))}</b> have usable trees, "
               f"<b>{n_(s.get('No tree records', 0))}</b> have no tree records, and "
               f"<b>{n_(s.get('Records but no usable DBH and height', 0))}</b> have records that cannot be used. "
               "The last group is a measurement gap, not an empty stand.")
    if "carbon_mg_provisional" in trees_m.columns and len(trees_m):
        q = trees_m["carbon_mg_provisional"].quantile([0.5, 0.99])
        if q.iloc[0] > 0:
            out.append(f"Provisional carbon per tree has a median of <b>{q.iloc[0]:.3f} Mg</b> and a 99th percentile of "
                       f"<b>{q.iloc[1]:.2f} Mg</b>, {q.iloc[1] / q.iloc[0]:.0f} times the median. Report medians and percentiles beside means.")
    flag = int((trees_m[dbh_col] <= 0).sum() + (trees_m[ht_col] <= 0).sum())
    if flag:
        out.append(f"<b>{n_(flag)}</b> retained values are zero or negative. They were flagged, not removed. Review them against the Part A zero and null rules.")
    return out


report(_tree_findings)
done()


<a id="e7"></a>

## E7 · Land-use proportions by SU (Final LULC)

Turns the final land-cover class at each office point into a land-use proportion per sampling unit, as set out in section 5 of the calculation document.

**About the code below**

| | |
|:--|:--|
| **Reads** | `lulc` from E2. |
| **Does** | (A) keeps one row per SU and plot, counting what is dropped, (B) computes proportions per SU, (C) draws the plot share by class. |
| **Writes** | `expansion/tables/national_land_use_proportion_by_su.csv`, `national_final_lulc_plot_share.csv`, `figures/lulc_plot_share.png`. |
| **Check before moving on** | The dropped-duplicate count is small, and most SUs have the designed number of plots. |

> **Why it matters.** SU area is not stated in the calculation document, so any total that needs area stays blocked. These are proportions, not hectares.


In [ ]:
# =============================================================================
#  SECTION E7 · LAND-USE PROPORTIONS BY SU (FINAL LULC)
#  Reads  : lulc (section E2)
#  Does   : one row per SU and plot, proportion per SU, plot share by class
#  Writes : expansion/tables/national_land_use_proportion_by_su.csv, national_final_lulc_plot_share.csv, a figure
#  Result : plot_lulc (one row per plot), lu (proportions per SU)
# =============================================================================
banner("E7", "Land-use proportions by SU", "Final LULC at office points · one row per plot · proportions per SU")

_cols = ["su_no", "plot_no", "final_lulc", "lon", "lat", "validation_true", "inside_forest_reserved"]
plot_lulc = lulc.sort_values(["su_no", "plot_no"]).drop_duplicates(["su_no", "plot_no"], keep="first")[_cols]
n_dropped = len(lulc) - len(plot_lulc)
design_plots = CFG["project"]["plots_per_su_design"]

sub("A", "One row per plot")
kpis([kpi("Rows in LULC table", len(lulc)), kpi("Unique plots kept", len(plot_lulc)),
      kpi("Duplicate rows dropped", n_dropped, kind="warn" if n_dropped else "ok")])

sub("B", "Proportions per SU")
lu = land_use_proportion_by_su(plot_lulc, "su_no", "final_lulc", design_plots)
show(lu.head(20), title="Land-use proportion by SU (first 20)")
lu.to_csv(EXP_OUT / "tables" / "national_land_use_proportion_by_su.csv", index=False)
share = plot_lulc["final_lulc"].value_counts(normalize=True).rename("plot_share").to_frame()
share["n_plots"] = plot_lulc["final_lulc"].value_counts()
show(share.head(15), title="Plot share by final class", bars=["plot_share"])
share.to_csv(EXP_OUT / "tables" / "national_final_lulc_plot_share.csv")

sub("C", "Plot share chart")
_top = share.head(12)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(_top.index.astype(str), _top["plot_share"] * 100, color=CHART_COLORS[2])
ax.set_xlabel("Share of plots (%)")
ax.set_title("Final LULC class share of plots")
_p = EXP_OUT / "figures" / "lulc_plot_share.png"
fig.tight_layout(); fig.savefig(_p); plt.close(fig)
show_fig(_p, title="Final LULC class share of plots",
         read="Bar length is the percentage of plots in each final class, largest at the top.")


def _lu_findings():
    per_su = plot_lulc.groupby("su_no").size()
    out = [f"<b>{n_(len(plot_lulc))}</b> plots in <b>{n_(len(per_su))}</b> sampling units enter the proportions. "
           f"{n_(n_dropped)} duplicate rows were dropped (first row kept)."]
    short = int((per_su < design_plots).sum())
    out.append(f"<b>{n_(short)}</b> of {n_(len(per_su))} units have fewer than the designed {design_plots} plots, so their proportions rest on fewer points.")
    top = share.iloc[0]
    out.append(f"<b>{share.index[0]}</b> covers {top['plot_share'] * 100:.1f}% of plots.")
    out.append("SU area is not stated in the calculation document. Totals that need area stay blocked until Area_H or another area source is supplied.")
    return out


report(_lu_findings)
done()


<a id="e8"></a>

## E8 · Shrub biomass (provisional)

Applies the shrub equations to records with crown diameters and a height. The column choices and the height unit are shown, because both are assumptions.

**About the code below**

| | |
|:--|:--|
| **Reads** | `nat_shrubs` from E5. |
| **Does** | (A) detects the species, crown and height columns, (B) applies the species rule and unit check, (C) computes biomass and carbon, (D) writes findings. |
| **Writes** | `expansion/tables/national_shrubs_provisional_biomass.csv`. |
| **Check before moving on** | The detected columns are the ones you expect. If several height columns exist, a warning lists them. Confirm the unit. |

> **Why it matters.** A wrong height column or unit changes biomass by orders of magnitude without raising an error.


In [ ]:
# =============================================================================
#  SECTION E8 · SHRUB BIOMASS (PROVISIONAL)
#  Reads  : nat_shrubs (section E5)
#  Does   : detects columns, applies the species rule and unit check, computes biomass and carbon
#  Writes : expansion/tables/national_shrubs_provisional_biomass.csv
#  Result : sh_ok (processed shrubs, when possible), shrub_note
# =============================================================================
banner("E8", "Shrub biomass (provisional)", "Column detection · unit check · calculation-document equations")

sh = nat_shrubs.copy()
sp = next((c for c in sh.columns if "species" in c.lower()), None)
cd1 = next((c for c in sh.columns if "crown_diameterl1" in c.lower() or c.lower().endswith("diam1")), None)
cd2 = next((c for c in sh.columns if "crown_diameterl2" in c.lower() or c.lower().endswith("diam2")), None)
ht_candidates = [c for c in sh.columns if c.lower() in ("sh_height", "tree_height", "height") or "height" in c.lower()]
sh_h = ht_candidates[0] if ht_candidates else None

sub("A", "Detected columns")
show(pd.DataFrame({"role": ["Species", "Crown diameter 1", "Crown diameter 2", "Height"],
                   "column used": [sp or "-", cd1 or "-", cd2 or "-", sh_h or "-"]}),
     title="Columns used for the shrub equations")
if len(ht_candidates) > 1:
    callout("warning", "More than one height column",
            "Used <code>" + str(sh_h) + "</code>. Other candidates: " + ", ".join(f"<code>{c}</code>" for c in ht_candidates[1:]) +
            ". Confirm that the first one is the shrub height.")

sub("B", "Species rule, unit check and biomass")
n_all_sh = len(sh)
if sp:
    sh = sh.loc[exclude_by_substrings(sh[sp], SHRUB_EXCLUDE)]
n_after_rule = len(sh)
shrub_note = "not processed"
sh_ok = pd.DataFrame()
_absent = [r for r, c in [("species", sp), ("crown diameter 1", cd1), ("crown diameter 2", cd2), ("height", sh_h)] if c is None]
if _absent or not n_after_rule:
    callout("warning", "Shrub biomass was not computed",
            "Missing: " + (", ".join(_absent) if _absent else "no records left after the species rule") + ".")
else:
    sh = sh.copy()
    sh["mean_CD_m"] = (pd.to_numeric(sh[cd1], errors="coerce") + pd.to_numeric(sh[cd2], errors="coerce")) / 2.0
    h = pd.to_numeric(sh[sh_h], errors="coerce")
    unit_is_m = bool(h.median(skipna=True) < 20)
    sh["height_cm"] = np.where(unit_is_m, h * 100.0, h)
    say(f"Median raw height is {h.median(skipna=True):.2f}. It is read as <b>{'metres, converted to cm' if unit_is_m else 'centimetres'}</b> "
        "(rule: a median below 20 means metres). Confirm against the field manual.", "info")
    ok = sh["mean_CD_m"].gt(0) & sh["height_cm"].gt(0)
    sh_ok = sh.loc[ok].copy()
    sh_ok["agb_mg"] = shrub_agb_mg(sh_ok["mean_CD_m"], sh_ok["height_cm"])
    sh_ok["bgb_mg"] = tree_bgb_mg(sh_ok["agb_mg"])
    sh_ok["total_biomass_mg"] = total_biomass_mg(sh_ok["agb_mg"], sh_ok["bgb_mg"])
    sh_ok["carbon_mg_provisional"] = carbon_mg(sh_ok["total_biomass_mg"])
    sh_ok.to_csv(EXP_OUT / "tables" / "national_shrubs_provisional_biomass.csv", index=False)
    shrub_note = f"processed n={len(sh_ok)}"
    show(pd.DataFrame({"step": ["All shrub records", "After the species rule", "With positive crown and height (processed)"],
                       "records": [n_all_sh, n_after_rule, len(sh_ok)]}),
         title="Shrub row ledger", bars=["records"])


def _shrub_findings():
    if not len(sh_ok):
        return []
    c = sh_ok["carbon_mg_provisional"]
    return [f"<b>{n_(len(sh_ok))}</b> of {n_(n_all_sh)} shrub records ({pct(len(sh_ok), n_all_sh)}) were processed. "
            f"Provisional carbon per shrub has a median of <b>{c.median():.4f} Mg</b>.",
            "The shrub equation uses a logarithm whose base is an open question in the calculation document. The code assumes the natural log. "
            "Treat shrub carbon as provisional until the author confirms the base."]


report(_shrub_findings)
done(shrub_note)


<a id="e9"></a>

## E9 · Bivariate and multivariate checks on the provisional metrics

Looks at how DBH, height and the derived metrics move together, and how much of the variation a few components capture.

**About the code below**

| | |
|:--|:--|
| **Reads** | `trees_m` from E6. |
| **Does** | (A) Pearson and Spearman correlations, (B) the height to DBH relationship, (C) a descriptive PCA, (D) findings. |
| **Writes** | `expansion/tables/corr_pearson_tree_metrics.csv`, `corr_spearman_tree_metrics.csv`, `pca_tree_metrics.csv`, two figures. |
| **Check before moving on** | Read the finding on built-in correlation before using any of these coefficients as evidence. |

> **Why it matters.** Basal area, volume, biomass and carbon are computed from DBH and height by formula. Their high correlations with each other are built in, not discovered.


In [ ]:
# =============================================================================
#  SECTION E9 · BIVARIATE AND MULTIVARIATE CHECKS ON PROVISIONAL METRICS
#  Reads  : trees_m (section E6)
#  Does   : Pearson and Spearman correlations, height to DBH fit, descriptive PCA
#  Writes : expansion/tables/corr_*.csv, pca_tree_metrics.csv, figures/corr_heatmaps_tree_metrics.png, height_dbh_hexbin.png
#  Result : corr_p, corr_s, ols_r2, pca_ratio
# =============================================================================
banner("E9", "Bivariate and multivariate checks", "Correlations · height to DBH · descriptive PCA")
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

cols = [c for c in [dbh_col, ht_col, "basal_area_m2", "volume_m3", "agb_mg", "carbon_mg_provisional"] if c in trees_m.columns]
if len(cols) < 2 or len(trees_m) < 10:
    say("Too few metric columns or records for this section.", "warning")
    corr_p = corr_s = pd.DataFrame()
    ols_r2, pca_ratio = float("nan"), []
else:
    sub("A", "Correlations")
    corr_p = trees_m[cols].corr(method="pearson")
    corr_s = trees_m[cols].corr(method="spearman")
    corr_p.to_csv(EXP_OUT / "tables" / "corr_pearson_tree_metrics.csv")
    corr_s.to_csv(EXP_OUT / "tables" / "corr_spearman_tree_metrics.csv")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, mat, ttl in zip(axes, (corr_p, corr_s), ("Pearson", "Spearman")):
        im = ax.imshow(mat.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
        ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
        ax.set_title(ttl); ax.grid(False)
        for i in range(len(cols)):
            for j in range(len(cols)):
                ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=axes, shrink=0.8, label="Correlation")
    _p = EXP_OUT / "figures" / "corr_heatmaps_tree_metrics.png"
    fig.savefig(_p); plt.close(fig)
    show_fig(_p, title="Correlation of tree metrics",
             read="Red is positive and blue is negative. Compare the two panels: where Spearman is higher than Pearson, the link is monotone but not straight.")

    sub("B", "Height to DBH")
    X = trees_m[[dbh_col]].to_numpy(dtype=float)
    y = trees_m[ht_col].to_numpy(dtype=float)
    ols = LinearRegression().fit(X, y)
    ols_r2 = float(ols.score(X, y))
    fig, ax = plt.subplots(figsize=(7, 5.5))
    hb = ax.hexbin(X[:, 0], y, gridsize=40, cmap="viridis", mincnt=1, bins="log")
    xs = np.linspace(np.nanmin(X), np.nanmax(X), 50)
    ax.plot(xs, ols.predict(xs.reshape(-1, 1)), color=CHART_COLORS[3], label=f"Linear fit, R² = {ols_r2:.2f}")
    ax.set_xlabel("DBH"); ax.set_ylabel("Height"); ax.set_title("Height against DBH")
    ax.legend(); fig.colorbar(hb, label="Trees (log scale)")
    _p = EXP_OUT / "figures" / "height_dbh_hexbin.png"
    fig.tight_layout(); fig.savefig(_p); plt.close(fig)
    show_fig(_p, title="Height against DBH",
             read="Each hexagon counts trees. Brighter hexagons hold more trees. The line is a straight-line fit; a curved cloud means a log-log form fits better (see E11).")

    sub("C", "Descriptive PCA")
    Z = StandardScaler().fit_transform(trees_m[cols].dropna())
    pca = PCA(n_components=min(3, Z.shape[1]), random_state=SEED).fit(Z)
    pca_ratio = list(pca.explained_variance_ratio_)
    pca_tbl = pd.DataFrame({"pc": range(1, len(pca_ratio) + 1), "var_ratio": pca_ratio})
    pca_tbl["cumulative"] = pca_tbl["var_ratio"].cumsum()
    pca_tbl.to_csv(EXP_OUT / "tables" / "pca_tree_metrics.csv", index=False)
    show(pca_tbl, title="Variance explained by the first components", bars=["var_ratio"])

    def _bi_findings():
        rho = corr_s.loc[dbh_col, ht_col]
        return [f"DBH and height have a Spearman correlation of <b>{rho:.2f}</b>, and a straight-line fit of height on DBH explains <b>{ols_r2 * 100:.0f}%</b> of the variance.",
                "Basal area, volume, biomass and carbon are calculated from DBH and height, so their correlations with each other and with DBH are built in. "
                "They add no independent evidence and should not be used together as model inputs.",
                f"The first principal component holds <b>{pca_ratio[0] * 100:.0f}%</b> of the variance. Because the inputs are formula-linked, this mostly restates tree size."]
    report(_bi_findings)
done()


<a id="e10"></a>

## E10 · Spatial autocorrelation (ESDA) at SU level

Tests whether sampling units with high mean tree carbon sit near other high ones, using one point per sampling unit.

**About the code below**

| | |
|:--|:--|
| **Reads** | `plot_lulc` (E7) and `trees_m` (E6). |
| **Does** | (A) builds one point per SU from the mean plot coordinates, (B) computes Global Moran's I for several neighbour counts, (C) runs Getis-Ord Gi\* when available, (D) writes findings. |
| **Writes** | `expansion/tables/esda_global_moran_mean_c_su.csv`, `esda_moran_k_sensitivity.csv`, `esda_gi_star_mean_c_su.csv`. |
| **Check before moving on** | Moran's I keeps the same sign and a similar size across the neighbour counts. If it flips, do not report a pattern. |

> **Why it matters.** Plots inside one SU carry the same value. Using plot points would repeat each value five times and make clustering look stronger than it is.


In [ ]:
# =============================================================================
#  SECTION E10 · ESDA AT SU LEVEL (GLOBAL MORAN'S I, GETIS-ORD GI*)
#  Reads  : plot_lulc (section E7), trees_m (section E6)
#  Does   : one point per SU, Moran's I for k = 4, 8, 12, Gi* when the packages exist
#  Writes : expansion/tables/esda_global_moran_mean_c_su.csv, esda_moran_k_sensitivity.csv, esda_gi_star_mean_c_su.csv
#  Result : su_pts (one point per SU), mi (Moran's I, k = 8), moran_rows
# =============================================================================
banner("E10", "Spatial autocorrelation at SU level", "One point per SU · Moran's I sensitivity · Gi*")

sub("A", "One point per sampling unit")
_ll = (plot_lulc.assign(su_key=plot_lulc["su_no"].astype(str).str.strip())
                .groupby("su_key")
                .agg(lon=("lon", "mean"), lat=("lat", "mean"), n_plots=("plot_no", "nunique"))
                .reset_index())
su_c = trees_m.groupby("su_key")["carbon_mg_provisional"].mean().rename("mean_c_su")
su_pts = _ll.merge(su_c, left_on="su_key", right_index=True, how="left")
su_pts["su_no"] = su_pts["su_key"]
su_pts = su_pts.dropna(subset=["lon", "lat"])
_esda = su_pts.dropna(subset=["mean_c_su"])
kpis([kpi("SU points", len(su_pts)), kpi("With a carbon value", len(_esda)),
      kpi("Without trees or metrics", len(su_pts) - len(_esda), kind="warn" if len(su_pts) > len(_esda) else "ok")])

sub("B", "Global Moran's I by neighbour count")
moran_rows = []
gdf = points_gdf(_esda, "lon", "lat") if len(_esda) else None
for k in (4, 8, 12):
    if gdf is None or len(gdf) <= k:
        continue
    try:
        r = global_moran_i(gdf, "mean_c_su", k_neighbors=k)
        moran_rows.append({"k_neighbors": k, **dict(r)})
    except Exception as e:
        moran_rows.append({"k_neighbors": k, "status": f"failed: {type(e).__name__}"})
if moran_rows:
    show(pd.DataFrame(moran_rows), title="Global Moran's I for k nearest neighbours")
    pd.DataFrame(moran_rows).to_csv(EXP_OUT / "tables" / "esda_moran_k_sensitivity.csv", index=False)
    mi = next((r for r in moran_rows if r["k_neighbors"] == 8), moran_rows[0])
    pd.DataFrame([mi]).to_csv(EXP_OUT / "tables" / "esda_global_moran_mean_c_su.csv", index=False)
else:
    say("Too few SU points for Moran's I.", "warning")

sub("C", "Getis-Ord Gi*")
if gdf is not None and len(gdf) > 8:
    gi = getis_ord_gi_star(gdf, "mean_c_su", k_neighbors=8)
    if gi.get("status") == "ok":
        _keep = [c for c in ["su_key", "su_no", "mean_c_su", "gi_star", "gi_p"] if c in gi["gdf"].columns]
        gi["gdf"][_keep].to_csv(EXP_OUT / "tables" / "esda_gi_star_mean_c_su.csv", index=False)
        say("Gi* table saved.", "success")
    else:
        say(f"Gi* was not run: {gi.get('reason')}", "note")


def _moran_findings():
    out = [f"Moran's I was computed on <b>{n_(len(_esda))}</b> sampling units, one point per unit at the mean of its plot coordinates."]
    row = mi if isinstance(mi, dict) else {}
    I = next((row[k] for k in ("I", "moran_i", "morans_i") if k in row), None)
    p = next((row[k] for k in ("p_sim", "p_value", "p", "pvalue") if k in row), None)
    if I is not None and p is not None:
        kind = "positive (similar values cluster)" if I > 0 else "negative (neighbours differ)"
        out.append(f"With k = 8 neighbours, I = <b>{float(I):.3f}</b> (p = {float(p):.3f}), which is {kind}"
                   f"{' and statistically distinguishable from random' if float(p) < 0.05 else ' but not distinguishable from random'}.")
    if len(moran_rows) > 1:
        out.append("The table above repeats the test with 4, 8 and 12 neighbours. A pattern is only worth reporting if the sign and size hold across all three.")
    out.append("The value tested is mean carbon per measured tree. Units with no usable trees are left out, so this describes where trees exist, not carbon per hectare. "
               "Distances use degrees, which stretch with latitude, so treat neighbour sets as approximate.")
    return out


report(_moran_findings)
done()


<a id="e11"></a>

## E11 · Height model with SU-blocked validation

Predicts tree height from DBH and compares a mean baseline, a straight line, a log-log allometry and a random forest. Each is scored twice: with random folds, and with folds that keep whole sampling units together.

**About the code below**

| | |
|:--|:--|
| **Reads** | `trees_m` from E6. |
| **Does** | (A) builds the folds, (B) fits and scores four models under both schemes, (C) charts the error, (D) writes findings. |
| **Writes** | `expansion/tables/ml_height_cv_random_vs_group.csv`, `ml_optimism_gap.csv`, `figures/ml_height_cv_rmse.png`. |
| **Check before moving on** | The SU-blocked error is the one to quote. A model that does not beat the simple baselines is not worth its complexity. |

> **Why it matters.** Trees in one SU resemble each other. Random folds put neighbours in both train and test and make every model look better than it will be on a new SU.


In [ ]:
# =============================================================================
#  SECTION E11 · HEIGHT MODEL WITH SU-BLOCKED CROSS-VALIDATION
#  Reads  : trees_m (section E6)
#  Does   : four models on DBH alone, scored with random folds and with SU-blocked folds
#  Writes : expansion/tables/ml_height_cv_random_vs_group.csv, ml_optimism_gap.csv, figures/ml_height_cv_rmse.png
#  Result : cv (fold scores), gap_tbl (per model), ML_GAP (random forest gap, RMSE units)
# =============================================================================
banner("E11", "Height model with SU-blocked validation", "Baselines · linear · log-log · random forest · random versus SU-blocked folds")
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

X = trees_m[[dbh_col]].to_numpy(dtype=float)
y = trees_m[ht_col].to_numpy(dtype=float)
groups = trees_m["su_key"].astype(str).to_numpy()
_m = np.isfinite(X[:, 0]) & np.isfinite(y)
X, y, groups = X[_m], y[_m], groups[_m]
n_groups = len(np.unique(groups))
n_splits = min(5, n_groups)

if n_splits < 2 or len(y) < 50:
    say("Too few trees or sampling units for cross-validation.", "warning")
    cv, gap_tbl = pd.DataFrame(), pd.DataFrame()
else:
    sub("A", "Models and folds")
    models = {"Mean baseline": DummyRegressor(strategy="mean"),
              "Linear (DBH)": LinearRegression()}
    if (X > 0).all() and (y > 0).all():
        models["Log-log allometry"] = TransformedTargetRegressor(
            regressor=Pipeline([("log", FunctionTransformer(np.log, validate=False)), ("lin", LinearRegression())]),
            func=np.log, inverse_func=np.exp)
    models["Random forest"] = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1, min_samples_leaf=5)
    say(f"{n_(len(y))} trees in {n_(n_groups)} sampling units, {n_splits} folds per scheme.", "info")

    def _score(name, model, splitter, scheme, grp=None):
        rows = []
        it = splitter.split(X, y, grp) if grp is not None else splitter.split(X, y)
        for fold, (tr, te) in enumerate(it, 1):
            fit = clone(model).fit(X[tr], y[tr])
            pred = fit.predict(X[te])
            rows.append({"model": name, "scheme": scheme, "fold": fold,
                         "r2": float(r2_score(y[te], pred)),
                         "rmse": float(np.sqrt(mean_squared_error(y[te], pred))),
                         "mae": float(mean_absolute_error(y[te], pred))})
        return rows

    sub("B", "Scores under both schemes")
    rows = []
    for name, model in models.items():
        rows += _score(name, model, KFold(n_splits, shuffle=True, random_state=SEED), "random_kfold")
        rows += _score(name, model, GroupKFold(n_splits), "group_kfold_su", groups)
    cv = pd.DataFrame(rows)
    cv.to_csv(EXP_OUT / "tables" / "ml_height_cv_random_vs_group.csv", index=False)
    means = cv.groupby(["model", "scheme"])[["r2", "rmse", "mae"]].mean().reset_index()
    show(means, title="Mean score per model and scheme", good_low=["rmse", "mae"])
    piv = means.pivot(index="model", columns="scheme", values="rmse")
    gap_tbl = piv.rename(columns={"random_kfold": "rmse_random", "group_kfold_su": "rmse_group"}).reset_index()
    gap_tbl["optimism_gap_rmse"] = gap_tbl["rmse_group"] - gap_tbl["rmse_random"]
    gap_tbl.to_csv(EXP_OUT / "tables" / "ml_optimism_gap.csv", index=False)
    show(gap_tbl, title="Optimism gap (SU-blocked minus random, RMSE)")
    ML_GAP = float(gap_tbl.loc[gap_tbl["model"] == "Random forest", "optimism_gap_rmse"].iloc[0])

    sub("C", "Error by model")
    fig, ax = plt.subplots(figsize=(8.5, 5))
    order = list(models)
    w = 0.38
    xs = np.arange(len(order))
    ax.bar(xs - w / 2, [piv.loc[m_, "random_kfold"] for m_ in order], w, label="Random folds", color=CHART_COLORS[5])
    ax.bar(xs + w / 2, [piv.loc[m_, "group_kfold_su"] for m_ in order], w, label="SU-blocked folds", color=CHART_COLORS[3])
    ax.set_xticks(xs); ax.set_xticklabels(order)
    ax.set_ylabel("RMSE of height"); ax.set_title("Height model error by validation scheme"); ax.legend()
    _p = EXP_OUT / "figures" / "ml_height_cv_rmse.png"
    fig.tight_layout(); fig.savefig(_p); plt.close(fig)
    show_fig(_p, title="Height model error by validation scheme",
             read="Lower is better. The orange bar is the honest error for a new sampling unit. A large drop from the blue bar means random folds flatter the model.")

    def _ml_findings():
        g = means[means["scheme"] == "group_kfold_su"].sort_values("rmse")
        best, base = g.iloc[0], g[g["model"] == "Mean baseline"].iloc[0]
        out = [f"Under SU-blocked validation the best model is <b>{best['model']}</b> with RMSE <b>{best['rmse']:.2f}</b> and R² <b>{best['r2']:.2f}</b>. "
               f"The mean baseline scores RMSE {base['rmse']:.2f}, so the best model cuts the error by <b>{(1 - best['rmse'] / base['rmse']) * 100:.0f}%</b>."]
        if "Linear (DBH)" in set(g["model"]) and best["model"] == "Random forest":
            lin = g[g["model"] == "Linear (DBH)"].iloc[0]
            out.append(f"The random forest improves on the straight line by {(1 - best['rmse'] / lin['rmse']) * 100:.0f}% in RMSE. If that is small, prefer the simpler allometry.")
        rf_rand = float(gap_tbl.loc[gap_tbl["model"] == "Random forest", "rmse_random"].iloc[0])
        if rf_rand > 0 and ML_GAP / rf_rand < 0.01:
            out.append(f"For the random forest, SU-blocked and random-fold RMSE differ by only <b>{ML_GAP:+.3f}</b>, so SU clustering does not inflate the score here. "
                       "Quote the SU-blocked figure anyway, because it is the one that answers the question for a new unit.")
        else:
            out.append(f"For the random forest, SU-blocked RMSE exceeds random-fold RMSE by <b>{ML_GAP:.3f}</b> ({ML_GAP / rf_rand * 100:.0f}%). "
                       "Random folds flatter the model. Quote the SU-blocked figure.")
        out.append("DBH is the only predictor, so these models describe the height to DBH curve. They do not justify imputing heights for official carbon.")
        return out
    report(_ml_findings)
done()


<a id="e12"></a>

## E12 · Additional national maps

Maps the validation outcome and the SU mean provisional carbon on a latitude-corrected plane.

**About the code below**

| | |
|:--|:--|
| **Reads** | `plot_lulc` (E7) and `su_pts` (E10). |
| **Does** | (A) maps accepted and updated office points, (B) maps SU mean carbon, drawing high values last so they stay visible. |
| **Writes** | `expansion/maps/map_lulc_acceptance.png`, `map_su_mean_provisional_carbon.png`. |
| **Check before moving on** | Both maps show the same extent. Gaps are areas without sampling, not zero values. |


In [ ]:
# =============================================================================
#  SECTION E12 · ADDITIONAL NATIONAL MAPS
#  Reads  : plot_lulc (section E7), su_pts (section E10)
#  Does   : maps accepted versus updated points, and SU mean provisional carbon
#  Writes : expansion/maps/map_lulc_acceptance.png, map_su_mean_provisional_carbon.png
#  Result : two PNG maps
# =============================================================================
banner("E12", "Additional national maps", "Validation outcome · SU mean provisional carbon")


def _geo_axes(ax, lat):
    ax.set_aspect(1.0 / np.cos(np.deg2rad(np.nanmean(lat))))      # keeps shapes honest away from the equator
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")


sub("A", "Accepted versus updated office points")
_pl = plot_lulc.dropna(subset=["lon", "lat"])
_acc = _pl["validation_true"].astype(int).eq(1)
fig, ax = plt.subplots(figsize=(9, 8), dpi=130)
ax.scatter(_pl.loc[~_acc, "lon"], _pl.loc[~_acc, "lat"], s=5, alpha=0.6, color=CHART_COLORS[3], label=f"Updated ({int((~_acc).sum()):,})")
ax.scatter(_pl.loc[_acc, "lon"], _pl.loc[_acc, "lat"], s=4, alpha=0.35, color=CHART_COLORS[0], label=f"Accepted ({int(_acc.sum()):,})")
ax.set_title("National LULC validation outcome at office points")
_geo_axes(ax, _pl["lat"]); ax.legend(markerscale=3)
_p = EXP_OUT / "maps" / "map_lulc_acceptance.png"
fig.tight_layout(); fig.savefig(_p); plt.close(fig)
show_fig(_p, title="National LULC validation outcome",
         read="Blue dots were accepted as first read, orange dots were changed in validation. Clusters of orange show where the first reading struggled.")

sub("B", "SU mean provisional carbon")
_c = su_pts.dropna(subset=["mean_c_su"]).sort_values("mean_c_su")
fig, ax = plt.subplots(figsize=(9, 8), dpi=130)
sc = ax.scatter(_c["lon"], _c["lat"], c=_c["mean_c_su"], s=14, cmap="viridis", alpha=0.85)
fig.colorbar(sc, ax=ax, shrink=0.8, label="SU mean provisional tree carbon (Mg per tree)")
ax.set_title("SU mean provisional tree carbon (sample, not design-based)")
_geo_axes(ax, _c["lat"])
_p = EXP_OUT / "maps" / "map_su_mean_provisional_carbon.png"
fig.tight_layout(); fig.savefig(_p); plt.close(fig)
show_fig(_p, title="SU mean provisional tree carbon",
         read="Each dot is one sampling unit, coloured by the mean carbon of its measured trees. Brighter means larger trees on average. This is not carbon per hectare.")
done()


<a id="e13"></a>

## E13 · Expansion run summary and limitations

Collects the headline numbers, states what the expansion cannot claim, and writes a summary file for hand-over.

**About the code below**

| | |
|:--|:--|
| **Reads** | Everything computed in E2 to E12. |
| **Does** | (A) headline cards, (B) limitations, (C) writes the run summary and findings, (D) lists the tables written. |
| **Writes** | `expansion/logs/expansion_run_summary.json`, `expansion_findings.csv`. |
| **Check before moving on** | The limitations table matches what you intend to say in the report. |

> **Why it matters.** A reader who sees only this summary should still know what the numbers are and what they are not.


In [ ]:
# =============================================================================
#  SECTION E13 · EXPANSION RUN SUMMARY AND LIMITATIONS
#  Reads  : values from sections E2 to E12 and the findings recorded so far
#  Does   : headline cards, limitations, run summary file, table inventory
#  Writes : expansion/logs/expansion_run_summary.json, expansion_findings.csv
#  Result : summary (dict)
# =============================================================================
banner("E13", "Expansion run summary", "Headline numbers · limitations · hand-over files")

sub("A", "Headline numbers")
_acc_share = pct(lulc["validation_true"].astype(int).sum(), len(lulc)) if len(lulc) else "n/a"
kpis([kpi("Trees with metrics", len(trees_m)),
      kpi("Shrubs processed", len(sh_ok)),
      kpi("LULC points accepted", _acc_share),
      kpi("SU-blocked gap (RF, RMSE)", "n/a" if ML_GAP is None else f"{ML_GAP:.3f}", kind="info")])

sub("B", "What this block cannot claim")
limits = [
    "Design-based national totals are blocked without weights and SU area.",
    "Carbon is provisional. It uses calculation-document constants that have not been checked against local allometry.",
    "The shrub equation assumes a natural logarithm. The calculation document leaves the base open.",
    "The HTML report removed its multi-class charts. They were restored here from CSV and PNG assets.",
    "Spatial regression, Bayesian and survival modules run only when assumptions and data support them. None is run here.",
    "Moran's I describes SU mean carbon per measured tree. It says nothing about carbon per hectare or about cause.",
]
show(pd.DataFrame({"limitation": limits}), title="Limitations")

sub("C", "Hand-over files")
summary = {
    "project": CFG["project"]["name"],
    "preserved_foundation": "Part A of this notebook (all V5 cells retained)",
    "lulc_key_numbers": kn,
    "n_trees_metrics": int(len(trees_m)),
    "shrub_note": shrub_note,
    "esda_moran": mi,
    "optimism_gap_rmse": ML_GAP,
    "policy": CFG.get("policy"),
    "limitations": limits,
}
(EXP_OUT / "logs" / "expansion_run_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
_f = pd.DataFrame([r for r in FINDINGS_LOG if _part(r["section"]) == 1])
_f.to_csv(EXP_OUT / "logs" / "expansion_findings.csv", index=False)
say(f"Run summary and {len(_f)} findings written to", "save", code=EXP_OUT / "logs")
file_inventory(EXP_OUT / "tables", "*.csv", title="Tables written by Part B", limit=40)
done()


# Part C · National-first geopipeline (entire country)

This block runs the **national tabular universe** (all regions in the cleansed SU master), not an Asir-only clip.

| Item | Detail |
|:--|:--|
| LEFT master | `nfi_of_ksa_program_su.xlsx` (all SUs) joined to a 5-plot lattice |
| Biomass | Word calculation document, trees **and** shrubs |
| GIS `Area_H` | Asir and Makkah grids only. Other regions are marked `AREA_BLOCKED_NO_GRID` |
| Outputs | `outputs_national_v5/`. It does not overwrite Part A `outputs/` or the raw data |
| Atlas cartography | Fixed class colours (land-cover style), points or Voronoi, satellite basemap |
| Command-line twin | `python run_national_pipeline.py` |
| Dedicated guide | `NFI_National_GeoPipeline_Guide_V5.ipynb` |

| Section | Purpose |
|:--|:--|
| N0 | Environment and output folder |
| N1 | Run, or reload, the national pipeline |
| N2 | Gates, integrity check and sample figures |
| N3 | Thematic atlas refresh |
| N4 | Open the HTML deliverables |
| N5 | Scope reminder |

**How to run.** Part C uses its own names (`NAT_CFG`, `NAT_OUT`, `NAT_SUMMARY`) so it cannot overwrite Part B values such as `CFG` and the random seed `SEED`. It needs the presentation kit cell, but not Part A or Part B.


<a id="n0"></a>

## N0 · National environment

Sets the national root and output folder and imports the national modules. Safe to re-run.

**About the code below**

| | |
|:--|:--|
| **Reads** | `config.yaml`; modules under `src/`. |
| **Does** | (1) finds the package root, (2) loads the national config, (3) creates `outputs_national_v5/`, (4) reports which products already exist. |
| **Writes** | `outputs_national_v5/`. Defines `NAT_ROOT`, `NAT_CFG`, `NAT_OUT`, `SUMMARY_JSON`, `ATLAS_INDEX`, `REPORT_PATH`. |
| **Check before moving on** | The *Existing products* checklist tells you whether N1 can reuse a previous run. |

> **Why it matters.** Prefer the National Guide notebook if you want step-by-step national sections with inline EDA tables.


In [ ]:
# =============================================================================
#  SECTION N0 · NATIONAL ENVIRONMENT
#  Reads  : config.yaml and the project modules under src/
#  Does   : finds the package root, loads the national config, creates the output folder, lists existing products
#  Writes : outputs_national_v5/ (folder only)
#  Result : NAT_ROOT, NAT_CFG, NAT_OUT, SUMMARY_JSON, ATLAS_INDEX, REPORT_PATH
# =============================================================================
import os, sys, json, warnings
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML, Image, IFrame

warnings.filterwarnings("ignore")
banner("N0", "National environment", "Package root · national config · output folder · existing products")


def _find_national_root():
    candidates = [Path(globals()[n]) for n in ("EXP_ROOT", "ROOT") if n in globals()]
    candidates += [Path.cwd(), Path(r"F:/1.KSA/29.NFI/12.NFI_Comprehensive_Expanded_V5")]
    for p in candidates:
        if (p / "config.yaml").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("config.yaml and src/ were not found. Run the bootstrap cell first, "
                            "or change into the package folder.")


NAT_ROOT = _find_national_root()
os.chdir(NAT_ROOT)
if str(NAT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(NAT_ROOT / "src"))

from nfi_audit import load_config, resolve_portable_paths
from nfi_national_pipeline import run_national_pipeline

NAT_CFG = resolve_portable_paths(load_config(NAT_ROOT / "config.yaml"), NAT_ROOT)
NAT_OUT = Path(NAT_CFG["paths"].get("output_national_root", NAT_ROOT / "outputs_national_v5"))
if not NAT_OUT.is_absolute():
    NAT_OUT = NAT_ROOT / NAT_OUT
NAT_OUT.mkdir(parents=True, exist_ok=True)

SUMMARY_JSON = NAT_OUT / "logs" / "national_run_summary.json"
ATLAS_INDEX = NAT_OUT / "maps" / "html_atlas" / "index.html"
REPORT_PATH = NAT_OUT / "documents" / "NFI_national_geopipeline_report.html"

existing = pd.DataFrame({"item": ["Run summary", "HTML atlas", "HTML report"],
                         "pass": [SUMMARY_JSON.exists(), ATLAS_INDEX.exists(), REPORT_PATH.exists()]})
checklist(existing, item_col="item", pass_col="pass", title="Existing products")
say("Package root", "info", code=NAT_ROOT)
say("National output folder", "save", code=NAT_OUT)
done()


<a id="n1"></a>

## N1 · Run the national pipeline

Runs ingest, lattice, trees, shrubs, EDA, maps and HTML for the whole country, or reloads the previous run summary.

**About the code below**

| | |
|:--|:--|
| **Reads** | The national cleansed tables and the calculation-document constants. |
| **Does** | (A) runs or reloads the pipeline, (B) shows the run summary, the assumptions and the SU universe by region. |
| **Writes** | Everything under `outputs_national_v5/` (a full run only). |
| **Check before moving on** | The assumptions callout reads as you expect. Set `SKIP_IF_PRODUCTS_EXIST = False` to force a full recompute. |

> **Why it matters.** A reused summary can be out of date if the inputs or the config changed since it was written.


In [ ]:
# =============================================================================
#  SECTION N1 · RUN THE NATIONAL PIPELINE
#  Reads  : national cleansed tables (full run) or logs/national_run_summary.json (reuse)
#  Does   : runs or reloads the pipeline, shows the summary, the assumptions and the SU universe
#  Writes : outputs_national_v5/ (full run only)
#  Result : NAT (pipeline result), NAT_SUMMARY (run summary dict)
# =============================================================================
SKIP_IF_PRODUCTS_EXIST = True        # set False to force a full recompute

banner("N1", "National pipeline", "Ingest · lattice · trees · shrubs · EDA · maps · HTML")

have_core = SUMMARY_JSON.exists() and ATLAS_INDEX.exists() and REPORT_PATH.exists()
if SKIP_IF_PRODUCTS_EXIST and have_core:
    NAT_SUMMARY = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
    NAT = {"run_summary": NAT_SUMMARY, "skipped_rerun": True}
    callout("info", "Previous run reused",
            f"Core products already exist, so the full pipeline was skipped. Set <code>SKIP_IF_PRODUCTS_EXIST = False</code> to recompute. "
            f"The summary was last written {pd.Timestamp(SUMMARY_JSON.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M}.")
else:
    NAT = run_national_pipeline(NAT_CFG, out_root=NAT_OUT)
    NAT_SUMMARY = NAT["run_summary"]

sub("A", "Run summary")
_skip = ("figures", "maps", "assumptions", "interactive_maps")
_scalars = {k: v for k, v in NAT_SUMMARY.items() if k not in _skip and isinstance(v, (int, float, str, bool))}
_nested = {k: v for k, v in NAT_SUMMARY.items() if k not in _skip and k not in _scalars}
show(pd.DataFrame({"item": [_pretty(k) for k in _scalars], "value": [str(v)[:140] for v in _scalars.values()]}),
     title="Run summary (single values)", max_rows=40)
if _nested:
    show(pd.DataFrame({"item": [_pretty(k) for k in _nested],
                       "type": [type(v).__name__ for v in _nested.values()],
                       "size": [len(v) if hasattr(v, "__len__") else 1 for v in _nested.values()]}),
         title="Run summary (grouped values)", caption="Gates and counts are opened in N2.", max_rows=40)

sub("B", "Assumptions")
_assume = NAT_SUMMARY.get("assumptions", [])
if _assume:
    callout("note", "Assumptions recorded by the pipeline", "<ol style='margin:4px 0 0 18px'>" + "".join(f"<li>{a}</li>" for a in _assume) + "</ol>")

sub("C", "SU universe and ancillary pools")
by_region_csv = NAT_OUT / "audit" / "national_su_by_region.csv"
anc_csv = NAT_OUT / "audit" / "national_ancillary_pools.csv"
if by_region_csv.exists():
    _br = pd.read_csv(by_region_csv)
    _num = list(_br.select_dtypes("number").columns)[:2]
    show(_br.head(20), title="SU universe by region", bars=_num or None)
if anc_csv.exists():
    show(pd.read_csv(anc_csv), title="Ancillary pools (inventory only)")
done()


<a id="n2"></a>

## N2 · National gates, integrity check and sample figures

Reads the tree and shrub presence gates from the run summary, checks the lattice size against the design, and shows a short figure gallery.

**About the code below**

| | |
|:--|:--|
| **Reads** | `NAT_SUMMARY` and files under `outputs_national_v5/`. |
| **Does** | (A) gate cards, (B) lattice integrity check, (C) SU biomass sample, (D) selected figures. |
| **Writes** | Nothing. Numbers come from the run summary and the CSV files, and nothing is invented here. |
| **Check before moving on** | The lattice row count equals the number of SUs times the designed plots per SU. Any other count means a unit was lost or duplicated. |


In [ ]:
# =============================================================================
#  SECTION N2 · NATIONAL GATES, INTEGRITY CHECK AND SAMPLE FIGURES
#  Reads  : NAT_SUMMARY and files under outputs_national_v5/
#  Does   : gate cards, lattice integrity check, SU biomass sample, selected figures
#  Writes : nothing
#  Result : gate, gate_s (tree and shrub presence gates)
# =============================================================================
banner("N2", "National gates and sample figures", "Presence gates · lattice check · figure gallery")

NAT_SUMMARY = NAT.get("run_summary") or json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
gate = NAT_SUMMARY.get("gate_tree_presence") or {}
gate_s = NAT_SUMMARY.get("gate_shrub_presence") or {}

sub("A", "Presence gates")
kpis([kpi("SUs in the universe", gate.get("total_sus", "n/a")),
      kpi("SUs with trees", gate.get("sus_with_trees", "n/a")),
      kpi("SUs without trees", gate.get("sus_no_trees", "n/a"), kind="warn"),
      kpi("SUs with shrubs", gate_s.get("sus_with_shrubs", "n/a")),
      kpi("SUs without shrubs", gate_s.get("sus_no_shrubs", "n/a"), kind="warn")])

sub("B", "Lattice integrity")
_plots = NAT_CFG["project"]["plots_per_su_design"]
_expect = gate.get("total_sus", 0) * _plots if isinstance(gate.get("total_sus"), (int, float)) else None
_lat = NAT_SUMMARY.get("lattice_rows")
show(pd.DataFrame({"measure": ["Lattice rows", f"Expected rows (SUs × {_plots})", "Tree biomass rows", "Shrub biomass rows"],
                   "value": [_lat, _expect, NAT_SUMMARY.get("n_tree_biomass_rows"), NAT_SUMMARY.get("n_shrub_biomass_rows")]}),
     title="Lattice and biomass row counts")


def _gate_findings():
    out = []
    if isinstance(gate.get("total_sus"), (int, float)) and gate.get("total_sus"):
        out.append(f"<b>{pct(gate.get('sus_no_trees', 0), gate['total_sus'])}</b> of {n_(gate['total_sus'])} sampling units have no tree records, "
                   f"and <b>{pct(gate_s.get('sus_no_shrubs', 0), gate_s.get('total_sus', gate['total_sus']))}</b> have no shrub records. "
                   "Zero-tree units are kept in the frame so they count as zero, not as missing.")
    if _expect is not None and _lat is not None:
        if int(_lat) == int(_expect):
            out.append(f"The lattice has <b>{n_(_lat)}</b> rows, exactly the SU count times {_plots} designed plots. No unit was lost or duplicated.")
        else:
            out.append(f"The lattice has <b>{n_(_lat)}</b> rows but {n_(_expect)} were expected ({_plots} plots for each SU). Find the difference before using any national figure.")
    return out


report(_gate_findings)

sub("C", "SU biomass sample")
su_metrics = NAT_OUT / "validated" / "national_su_biomass_sample.csv"
if su_metrics.exists():
    show(pd.read_csv(su_metrics).head(10), title="SU biomass sample (first 10 rows)")

sub("D", "Selected national figures")
fig_dir = NAT_OUT / "figures" / "national_eda"
map_dir = NAT_OUT / "maps" / "national"
them_dir = map_dir / "thematic"
show_figs([
    (fig_dir / "univariate_trees_fv_national.png", "Tree variables, national distributions",
     "Each panel is one tree variable. Long right tails are normal for size. Spikes at round numbers suggest rounding in the field."),
    (fig_dir / "bivariate_corr_pair_trees_national.png", "Tree variables, pairwise relationships",
     "Each panel pairs two variables. Tight bands are formula-linked variables, not new evidence."),
    (fig_dir / "univariate_shrubs_national.png", "Shrub variables, national distributions",
     "Read as for the tree panel. Check the height scale against the unit assumption in E8."),
    (map_dir / "map_national_plots.png", "National plot locations",
     "Each dot is one plot. Gaps are unsampled areas, not zero values."),
    (them_dir / "thematic_land_use.png", "Land use, thematic map",
     "Colours are fixed class colours. They show sample points, not area shares."),
])
done()


<a id="n3"></a>

## N3 · Thematic atlas with land-cover style cartography

Rebuilds the dashboard atlas from the thematic tables when they exist. Categorical themes use fixed class colours, so a map point, its legend swatch and the donut panel share one colour per class.

**About the code below**

| | |
|:--|:--|
| **Reads** | `validated/national_su_thematic_attributes.csv`, `maps/national_thematic_catalog.csv`, optional Voronoi GeoJSON. |
| **Does** | (A) refreshes the atlas from existing tables, or regenerates the thematic maps when they are missing, (B) shows the catalogue and a link to the atlas. |
| **Writes** | `maps/html_atlas/` and, when regenerated, the thematic maps. |
| **Check before moving on** | The catalogue lists the themes you expect. If the tables were missing, the heavier route ran and a warning says so. |

> **Why it matters.** Class colours are display choices. They do not carry design-based area shares.


In [ ]:
# =============================================================================
#  SECTION N3 · THEMATIC ATLAS WITH LAND-COVER STYLE CARTOGRAPHY
#  Reads  : validated/national_su_thematic_attributes.csv, maps/national_thematic_catalog.csv, optional Voronoi GeoJSON
#  Does   : refreshes the atlas from existing tables, or regenerates the thematic maps first
#  Writes : maps/html_atlas/
#  Result : HTML_PRODUCTS (paths used by N4), ATLAS_INDEX
# =============================================================================
EMBED_ATLAS_PREVIEW = not globals().get("IN_COLAB", False)     # file links do not render inside Colab

banner("N3", "Thematic atlas", "Fixed class colours · points or Voronoi · satellite basemap")
from nfi_national_atlas_dashboard import write_national_dashboard_atlas
from nfi_national_thematic_maps import produce_national_thematic_maps
from nfi_national_boundary import load_country, country_geojson

su_them = NAT_OUT / "validated" / "national_su_thematic_attributes.csv"
cat_path = NAT_OUT / "maps" / "national_thematic_catalog.csv"
vor_path = NAT_OUT / "maps" / "cartography" / "su_voronoi_thiessen.geojson"

HTML_PRODUCTS = {}
if su_them.exists() and cat_path.exists():
    themes_df = pd.read_csv(su_them)
    catalog = pd.read_csv(cat_path)
    voronoi_fc = json.loads(vor_path.read_text(encoding="utf-8")) if vor_path.exists() else None
    try:
        country_fc = country_geojson(load_country())
    except Exception as e:
        say(f"Country boundary skipped: {e}", "note")
        country_fc = None
    ATLAS_INDEX = Path(write_national_dashboard_atlas(
        themes_df, NAT_OUT / "maps" / "html_atlas", catalog=catalog, voronoi_fc=voronoi_fc, country_fc=country_fc))
    say("Atlas refreshed from the existing thematic tables.", "success")
else:
    callout("warning", "Thematic tables missing",
            "Running <code>produce_national_thematic_maps</code>, which is the heavier route. "
            + ("The full pipeline runs first because no tables are in memory." if NAT.get("tables") is None else ""))
    if NAT.get("tables") is None:
        NAT = run_national_pipeline(NAT_CFG, out_root=NAT_OUT)
    THEMATIC = produce_national_thematic_maps(
        tables=NAT["tables"], tree_m=NAT["tree_m"], shrub_m=NAT["shrub_m"], lattice=NAT["lattice"],
        plot=NAT["tables"]["plot"], cfg=NAT_CFG, out_root=NAT_OUT)
    ATLAS_INDEX = Path(write_national_dashboard_atlas(
        THEMATIC["su_thematic"], NAT_OUT / "maps" / "html_atlas", catalog=THEMATIC["catalog"]))
    catalog = THEMATIC["catalog"]
    HTML_PRODUCTS["interactive_maps"] = THEMATIC.get("interactive") or {}

REPORT_PATH = NAT_OUT / "documents" / "NFI_national_geopipeline_report.html"
HTML_PRODUCTS["atlas_index"] = str(ATLAS_INDEX)
HTML_PRODUCTS["html_report"] = str(REPORT_PATH)

sub("B", "Catalogue and atlas link")
show(pd.DataFrame(catalog).head(20), title="Thematic catalogue (first 20)")
uri = ATLAS_INDEX.resolve().as_uri()
callout("info", "Land-cover style atlas",
        "Fixed class colours for land use, canopy, herbaceous, fire, environmental problems, causative agents, tree and crown condition, and region. "
        f"<a href=\"{uri}\" target=\"_blank\">Open the interactive atlas</a>.")
if EMBED_ATLAS_PREVIEW:
    try:
        display(IFrame(src=uri, width="100%", height=560))
    except Exception as e:
        say(f"Atlas preview skipped: {e}", "note")
done()


<a id="n4"></a>

## N4 · Open the HTML deliverables

Lists the shareable products of the national run: the methodology and results report, the atlas portal and the first interactive maps.

**About the code below**

| | |
|:--|:--|
| **Reads** | `HTML_PRODUCTS` from N3, or the files on disk. |
| **Does** | (A) builds a link list with file sizes, (B) optionally opens the report and the atlas in the browser. |
| **Writes** | Nothing. |
| **Check before moving on** | Every row shows the file as found. Browser opening is **off** by default; set `OPEN_IN_BROWSER = True` to enable it. |


In [ ]:
# =============================================================================
#  SECTION N4 · OPEN THE HTML DELIVERABLES
#  Reads  : HTML_PRODUCTS (section N3) or the files on disk
#  Does   : lists the shareable products with sizes and links, optionally opens them in a browser
#  Writes : nothing
#  Result : deliverables (table of products)
# =============================================================================
OPEN_IN_BROWSER = False              # set True to open the report and the atlas automatically

banner("N4", "HTML deliverables", "Report · atlas portal · interactive maps")
import html as _h
import webbrowser

REPORT_PATH = Path(HTML_PRODUCTS.get("html_report") or NAT_OUT / "documents" / "NFI_national_geopipeline_report.html")
ATLAS_INDEX = Path(HTML_PRODUCTS.get("atlas_index") or NAT_OUT / "maps" / "html_atlas" / "index.html")
imap = NAT_OUT / "maps" / "html_interactive"
INTERACTIVE = HTML_PRODUCTS.get("interactive_maps") or {}
if not INTERACTIVE and imap.exists():
    INTERACTIVE = {p.stem.replace("interactive_national_", ""): str(p) for p in imap.glob("interactive_national_*.html")}

links = [("Methodology and results report", REPORT_PATH), ("Interactive map atlas (portal)", ATLAS_INDEX)]
links += [(f"Interactive map: {_pretty(name)}", Path(path)) for name, path in list(INTERACTIVE.items())[:6]]

rows, bits = [], []
for label, path in links:
    ok = path.exists()
    rows.append({"deliverable": label, "found": ok, "size_mb": path.stat().st_size / 1e6 if ok else 0.0, "path": str(path)})
    if ok:
        bits.append(f'<div style="margin:8px 0"><a style="display:inline-block;padding:9px 14px;background:{BRAND_DK};color:#fff;'
                    f'text-decoration:none;border-radius:8px;font-weight:600" href="{path.resolve().as_uri()}" target="_blank">{_h.escape(label)}</a>'
                    f'<div style="font-size:12px;color:{MUTED};margin-top:3px"><code>{_h.escape(str(path))}</code></div></div>')
    else:
        bits.append(f'<div style="margin:8px 0;color:{MUTED}">Missing: {_h.escape(label)} ({_h.escape(str(path))})</div>')
display(HTML(f'<div style="font-family:{FONT};line-height:1.5">' + "".join(bits) + "</div>"))
deliverables = pd.DataFrame(rows)
show(deliverables, title="Deliverables", bars=["size_mb"])

if OPEN_IN_BROWSER:
    for path in (REPORT_PATH, ATLAS_INDEX):
        if path.exists():
            try:
                webbrowser.open(path.resolve().as_uri())
            except Exception as e:
                say(f"Browser open skipped for {path.name}: {e}", "note")
done()


<a id="n5"></a>

## N5 · Scope reminder

| Layer | Coverage in this run |
|:--|:--|
| SU and plot tables | Entire national cleansed master |
| Tree biomass (calculation document) | All SUs with eligible tree records |
| Shrub biomass (calculation document) | All SUs with eligible shrub records |
| Deadwood, litter, regeneration, vegetation | Count inventory only (no calculation-document equations) |
| Design-based `Area_H` totals | Only SUs with Asir or Makkah grid `Area_H`. Others are blocked |
| Atlas categorical colours | Fixed land-cover style palettes, not design-based area shares |

See `outputs_national_v5/logs/national_run_summary.json`, `docs/04_national_pipeline.md` and `docs/07_landcover_style_cartography.md`.

For the step-by-step national tutorial with inline univariate and bivariate tables, use **`NFI_National_GeoPipeline_Guide_V5.ipynb`**.
